# 34 — Repeated cluster-disjoint CV: five arms on one partition

**Cold-start summary.** Notebook 33 built a single 5-fold cluster-disjoint (Butina) partition
using the organisers' own confirmed recipe and found that this project's AID auxiliary-head
conclusion — confirmed at 25-fold random CV in notebook 29, deployed in 30 and 32 — **flips
sign on three of four isoforms** under that partition. That is a consequential result sitting at
the weakest evidentiary tier this project uses: one partition, five folds, exactly where notebook
28's AID screen sat before notebook 29 overturned it.

This notebook raises that comparison to a **repeated** cluster-disjoint design, and at the same
time trains **every model that could be an ensemble member on the SAME partition**, because blend
weights and error correlations are only comparable when computed on one partition.

**Five arms** (Part 2): PLAIN, AID, SINGLE (single-task, one model per isoform — an axis never
tested for decorrelation on this project), TREES (RF/XGBoost/LightGBM on raw descriptor and
fingerprint features, *not* on CheMeleon embeddings), DEADZONE (notebook 31's clipped target under
absolute-error loss, with its target rebuilt from **this** partition's own out-of-fold predictions).

**HONEST PRIOR, stated before any result.** Eleven modelling screens on this project have produced
**one** effect that survived a full confirmation (notebook 29's AID heads), and that effect then
**failed to transfer to the board** (NB30, macro 0.8160 against `10c`'s raw 0.7138). A null result
here, or a finding that the two partitions simply agree, is a legitimate and likely outcome — not a
failure of this notebook.

**Scope.** Evidence only. No submission is built, validated or sent; no full-data retrain; no
adoption recommended. `data/folds/cv_folds.csv` and `data/folds/cv_folds_butina.csv` are never
modified — the new partition is a new file.

**Autonomy.** Run with no human available. Every decision was pre-specified; anything that was not
is recorded in the **"Decisions taken without approval"** section near the end, which exists even
if empty.

In [1]:
import os

# THREAD ENVIRONMENT -- deliberately NOT set, and asserted unset. Notebook 33's own setup cell
# sets OMP_NUM_THREADS=1; this notebook's first execution copied that, and its Part 3 sanity
# check FAILED as a direct result (max abs ST-RAE difference 0.0378 against notebook 05's
# stored chemprop_chemeleoninit repeat0_fold0 scores, against a 0.03 threshold). Diagnosed
# rather than worked around: the training input CSV was byte-identical to notebook 05's own
# (sha256 66a1b380...), the chemprop CLI argv was identical to notebook 31's own baseline, and
# val_loss was bit-identical for epochs 0 and 1 before diverging at epoch 2 -- i.e. the data
# and the initialisation were right and the arithmetic was not. Re-running with these two
# variables unset and nothing else changed reproduced notebook 05 EXACTLY (0.00000000 on all
# four isoforms, and a bit-identical val_loss trajectory). Notebooks 29 and 31, the two that
# reproduce notebook 05 to 0.0000, do not set them either. Restricting BLAS/OMP to one thread
# changes floating-point reduction order and so changes the training trajectory.
for _v in ["OMP_NUM_THREADS", "KMP_DUPLICATE_LIB_OK"]:
    assert _v not in os.environ, (
        f"{_v} is set in this kernel's environment. It changes Chemprop's training trajectory "
        f"and breaks bit-reproducibility against notebook 05's stored scores -- see the comment "
        f"above. Unset it and restart the kernel before running this notebook.")

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import hashlib
import importlib.metadata
import itertools
import json
import subprocess
import time
import warnings
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from scipy import stats
from scipy.optimize import linear_sum_assignment
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import lightgbm as lgb

from src.features import butina_clusters, assign_screen_split
from src.chemprop_screen import (
    build_predict_csv,
    build_training_csv,
    run_chemprop_predict,
    run_chemprop_train,
    setup_logging,
    verify_predictions,
)
from src.cv_bootstrap import per_fold_bootstrap_seed
from src.vendor.openadmet_eval.config import ACTIVITY_METRICS, REGRESSION_ENDPOINTS
from src.vendor.openadmet_eval.custom_scoring_functions import rae_soft_threshold_absolute_error
from src.vendor.openadmet_eval.evaluate_predictions import add_macro_endpoint, score_activity_predictions

print(f"python: {sys.version.split()[0]}")
for pkg in ["numpy", "pandas", "scipy", "matplotlib", "rdkit", "chemprop", "torch",
            "scikit-learn", "lightgbm", "xgboost"]:
    try:
        print(f"{pkg}: {importlib.metadata.version(pkg)}")
    except importlib.metadata.PackageNotFoundError:
        pass

METRIC_NAMES = [name for name, _ in ACTIVITY_METRICS]
ISOFORMS = [e.split("_")[0] for e in REGRESSION_ENDPOINTS]
PIC50_COL = {iso: f"{iso}_pIC50_direct_inhibition" for iso in ISOFORMS}
EP_TO_ISO = {f"{iso}_pIC50_direct_inhibition": iso for iso in ISOFORMS}
AUX_COLS = [f"{iso}_AID1851_max_inhibition" for iso in ISOFORMS]
PANEL_TO_ISOFORM = {"p450-cyp1a2": "CYP1A2", "p450-cyp2c9": "CYP2C9",
                    "p450-cyp2d6": "CYP2D6", "p450-cyp3a4": "CYP3A4"}

OUT = REPO_ROOT / "outputs" / "34_butina_5x5"
FIG_OUT = OUT / "figures"
CHEMPROP_RUNS_DIR = OUT / "chemprop_runs"
PRED_DIR = OUT / "predictions"
SCORE_DIR = OUT / "scores"
OOF_DIR = OUT / "oof_long"
LOG_DIR = REPO_ROOT / "logs" / "34_butina_5x5"
for d in [OUT, FIG_OUT, CHEMPROP_RUNS_DIR, PRED_DIR, SCORE_DIR, OOF_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# READ-ONLY paths
FOLDS_PATH = REPO_ROOT / "data" / "folds" / "cv_folds.csv"
BUTINA1_PATH = REPO_ROOT / "data" / "folds" / "cv_folds_butina.csv"
CURATED_PATH = REPO_ROOT / "data" / "processed" / "train_inhibition_curated.csv"
BLIND_PATH = REPO_ROOT / "data" / "processed" / "test_blinded_curated.csv"
AID_CURATED_PATH = REPO_ROOT / "data" / "processed" / "aid1851_curated.csv"
AID_RAW_DIR = REPO_ROOT / "data" / "raw" / "aid1851"
ECFP4_NARROW_PATH = REPO_ROOT / "data" / "processed" / "tabular_baseline_features.csv"
MORDRED_PCA_PATH = REPO_ROOT / "data" / "processed" / "tabular_mordred_pca.csv"
NB05_OUT = REPO_ROOT / "outputs" / "05_cv_comparison"
NB28_OUT = REPO_ROOT / "outputs" / "28_external_data"
NB29_OUT = REPO_ROOT / "outputs" / "29_cv_confirmation"
NB31_OUT = REPO_ROOT / "outputs" / "31_deadzone"
NB33_OUT = REPO_ROOT / "outputs" / "33_butina_split"
NB24_OUT = REPO_ROOT / "outputs" / "24_nonnegative_stacking"
MANIFEST_PATH = NB05_OUT / "manifest.csv"

# THE new partition file this notebook writes -- a NEW name, never an existing one.
BUTINA5X5_PATH = REPO_ROOT / "data" / "folds" / "cv_folds_butina_5x5.csv"

VAL_FRACTION = 0.15
EPOCHS = 50
PATIENCE = 5
CHEMELEON_ARCH_ARGS = ["--from-foundation", "CHEMELEON", "--multi-hot-atom-featurizer-mode", "V2"]
WINSOR_LOW_PCT, WINSOR_HIGH_PCT = 1.0, 99.0
XGB_LGBM_EARLY_STOPPING_ROUNDS = 10   # matches scripts/run_5x5_cv_comparison.py / 04a precedent
N_FOLDS = 5
N_REPEATS_MAX = 5
TOL = 1e-8
SANITY_THRESHOLD = 0.03

# Conda env bin prepended so subprocess `chemprop` resolves to cyp-admet-v2, matching
# scripts/10_final_retrain_predict.py's own convention.
_env_bin = str(Path(sys.executable).parent)
_parts = os.environ.get("PATH", "").split(os.pathsep)
if _env_bin not in _parts:
    os.environ["PATH"] = os.pathsep.join([_env_bin, *_parts])
print(f"\nchemprop: {subprocess.run(['which','chemprop'],capture_output=True,text=True).stdout.strip()}")
print(f"OUT: {OUT}")

# RUN_STARTED_AT is persisted on the FIRST execution and reused on every resume, so the
# final scope check's mtime test spans the whole campaign rather than only the last
# nbconvert invocation.
CAMPAIGN_PATH = OUT / "campaign.json"
if CAMPAIGN_PATH.exists():
    CAMPAIGN = json.loads(CAMPAIGN_PATH.read_text())
    print(f"[resume] campaign started {CAMPAIGN['started_at_utc']}")
else:
    CAMPAIGN = {"started_at": time.time(),
                "started_at_utc": datetime.now(timezone.utc).isoformat()}
    CAMPAIGN_PATH.write_text(json.dumps(CAMPAIGN, indent=2))
    print(f"[new campaign] started {CAMPAIGN['started_at_utc']}")
RUN_STARTED_AT = float(CAMPAIGN["started_at"])

AUTONOMOUS_DECISIONS = []
ABORT = {"aborted": False, "reason": None, "where": None}

def record_decision(decision, reason, alternatives, authority):
    AUTONOMOUS_DECISIONS.append({"decision": decision, "reason": reason,
                                 "alternatives": alternatives, "authority": authority,
                                 "at_utc": datetime.now(timezone.utc).isoformat()})
    with open(OUT / "autonomous_decisions.json", "w") as fh:
        json.dump(AUTONOMOUS_DECISIONS, fh, indent=2)

def hard_fail(reason, where):
    # Records the abort and switches every later run cell into a no-op. Partial results
    # are already on disk (scores are appended after every single run), so a clean abort
    # keeps everything completed up to this point.
    ABORT.update({"aborted": True, "reason": reason, "where": where})
    with open(OUT / "abort.json", "w") as fh:
        json.dump(ABORT, fh, indent=2)
    print(f"*** RUN ABORTED *** at {where}: {reason}")

python: 3.11.13
numpy: 1.26.4
pandas: 2.3.3
scipy: 1.17.1
matplotlib: 3.11.1
rdkit: 2026.3.3
chemprop: 2.3.1
torch: 2.13.0
scikit-learn: 1.9.0
lightgbm: 4.7.0
xgboost: 3.2.0

chemprop: /Users/codiefreeman/miniconda3-arm64/envs/cyp-admet-v2/bin/chemprop
OUT: /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/34_butina_5x5
[resume] campaign started 2026-09-26T01:41:36.662118+00:00


## Part 1 — The repeated cluster-disjoint partition

**Butina clustering is deterministic given the fingerprint and the cutoff. It cannot be
re-seeded the way random CV can.** What *is* re-seedable is the assignment of whole clusters to
folds. So in this notebook a **"repeat" re-randomises cluster-to-fold assignment, not the
clustering itself** — the cluster membership is identical in all five repeats, and only which
fold each cluster lands in changes. This is a weaker form of repetition than notebook 03's random
5x5 (which re-randomises the compound-level split itself), and it is stated as such rather than
presented as equivalent: it samples the *assignment* noise, not the *clustering* noise, because
there is no clustering noise to sample.

Notebook 33 did not save its cluster assignments to disk (`outputs/33_butina_split/` holds a
`cluster_size_distribution.csv` but no per-compound cluster id), so the clustering is recomputed
here with identical parameters and asserted against notebook 33's own saved distribution.

In [2]:
def ecfp_radius3_fingerprints(smiles_list, n_bits=2048):
    # Reproduced verbatim from notebook 33 cell 3. A local, one-off helper -- NOT added to
    # src/features.py, whose established default is ECFP4 (radius=2). The organisers'
    # confirmed CYP3A4 clustering recipe specifies radius=3.
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=n_bits)
    fps = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        fps.append(generator.GetFingerprint(mol) if mol is not None else None)
    return fps


curated = pd.read_csv(CURATED_PATH)
blind = pd.read_csv(BLIND_PATH)
print(f"curated: {curated.shape}   blind: {blind.shape}")
assert len(curated) == 4905, "curated row count drifted from this project's own baseline"
assert len(blind) == 750, "blind row count drifted from this project's own baseline"

smiles_list = curated["canonical_smiles"].tolist()
fps_r3 = ecfp_radius3_fingerprints(smiles_list, n_bits=2048)
assert sum(fp is None for fp in fps_r3) == 0, "a curated SMILES failed to parse under radius=3"

DIST_CUTOFF = 0.65
SIM_CUTOFF = 1.0 - DIST_CUTOFF   # butina_clusters' own `cutoff` arg is a SIMILARITY cutoff
clusters = butina_clusters(fps_r3, cutoff=SIM_CUTOFF)
cluster_sizes = np.array([len(c) for c in clusters])
n_clusters = len(clusters)
n_singletons = int((cluster_sizes == 1).sum())

print(f"\nclusters: {n_clusters}   compounds: {int(cluster_sizes.sum())}   "
      f"largest cluster: {int(cluster_sizes.max())}")
print(f"singleton CLUSTERS: {n_singletons} ({n_singletons/n_clusters:.1%} of clusters)")
frac_compounds_nonsingleton = float((cluster_sizes[cluster_sizes > 1].sum()) / cluster_sizes.sum())
print(f"compounds in a NON-singleton cluster: {int(cluster_sizes[cluster_sizes>1].sum())} "
      f"({frac_compounds_nonsingleton:.1%} of compounds)")

curated: (4905, 20)   blind: (750, 4)



clusters: 2617   compounds: 4905   largest cluster: 19
singleton CLUSTERS: 1465 (56.0% of clusters)
compounds in a NON-singleton cluster: 3440 (70.1% of compounds)


In [3]:
# ---- Verification against notebook 33's own saved output, per the brief. Quoted values have
# been wrong several times in this project, so the check is against the saved CSV, not the prose.
nb33_dist = pd.read_csv(NB33_OUT / "cluster_size_distribution.csv")
nb33_n_clusters = int(nb33_dist["n_clusters"].sum())
nb33_n_compounds = int((nb33_dist["cluster_size"] * nb33_dist["n_clusters"]).sum())
nb33_singletons = int(nb33_dist.loc[nb33_dist["cluster_size"] == 1, "n_clusters"].iloc[0])
nb33_largest = int(nb33_dist["cluster_size"].max())

mine_dist = (pd.Series(cluster_sizes).value_counts().sort_index()
             .rename_axis("cluster_size").reset_index(name="n_clusters"))

print("notebook 33's saved distribution vs. this notebook's recomputation:")
for name, got, want in [("n_clusters", n_clusters, nb33_n_clusters),
                        ("n_compounds", int(cluster_sizes.sum()), nb33_n_compounds),
                        ("n_singleton_clusters", n_singletons, nb33_singletons),
                        ("largest_cluster", int(cluster_sizes.max()), nb33_largest)]:
    print(f"  {name:22s} got={got:6d}  nb33={want:6d}  {'OK' if got == want else 'MISMATCH'}")

assert n_clusters == nb33_n_clusters, "cluster count does not reproduce notebook 33's -- HARD FAILURE"
assert n_singletons == nb33_singletons, "singleton count does not reproduce notebook 33's -- HARD FAILURE"
assert int(cluster_sizes.max()) == nb33_largest, "largest cluster does not reproduce notebook 33's"
merged_dist = mine_dist.merge(nb33_dist, on="cluster_size", how="outer", suffixes=("_mine", "_nb33"))
assert (merged_dist["n_clusters_mine"] == merged_dist["n_clusters_nb33"]).all(), \
    "the full size distribution differs from notebook 33's -- HARD FAILURE"
print("\nCONFIRMED: the clustering reproduces notebook 33's exactly, size-for-size.")

# ---- A real quoted-figure ERROR found in notebook 33's own markdown, and repeated in this
# notebook's own task brief. Corrected here rather than carried forward.
NB33_QUOTED_SINGLETON_FRAC = 0.560
NB33_QUOTED_COMPOUND_SHARE = 0.44
print(f"\nnotebook 33's quoted singleton fraction {NB33_QUOTED_SINGLETON_FRAC:.1%} vs. recomputed "
      f"{n_singletons/n_clusters:.1%} -- CONFIRMED (this figure is correct).")
print(f"notebook 33's markdown, and this notebook's brief, both state '44% of COMPOUNDS sit in a")
print(f"non-singleton cluster'. That is the fraction of CLUSTERS that are non-singleton")
print(f"({1 - n_singletons/n_clusters:.1%}), not of compounds. The real compound share is")
print(f"{frac_compounds_nonsingleton:.1%} -- a unit error, not a numerical one. It makes the")
print(f"merging MORE substantial than either source claimed, not less.")

notebook 33's saved distribution vs. this notebook's recomputation:
  n_clusters             got=  2617  nb33=  2617  OK
  n_compounds            got=  4905  nb33=  4905  OK
  n_singleton_clusters   got=  1465  nb33=  1465  OK
  largest_cluster        got=    19  nb33=    19  OK

CONFIRMED: the clustering reproduces notebook 33's exactly, size-for-size.

notebook 33's quoted singleton fraction 56.0% vs. recomputed 56.0% -- CONFIRMED (this figure is correct).
notebook 33's markdown, and this notebook's brief, both state '44% of COMPOUNDS sit in a
non-singleton cluster'. That is the fraction of CLUSTERS that are non-singleton
(44.0%), not of compounds. The real compound share is
70.1% -- a unit error, not a numerical one. It makes the
merging MORE substantial than either source claimed, not less.


**Part 1 verification result, including one correction.** The clustering reproduces notebook 33's
exactly — 2,617 clusters, 1,465 singletons, largest cluster 19 compounds, and the *whole* size
distribution matches size-for-size. Notebook 33's headline singleton figure (56.0%) is confirmed.

**But a quoted figure in notebook 33's own markdown, repeated verbatim in this notebook's task
brief, is wrong in its units**: "44% of compounds sit in a non-singleton cluster" is actually the
fraction of *clusters* that are non-singleton (1,152/2,617 = 44.0%). The real share of
**compounds** sitting in a non-singleton cluster is **70.1%** (3,440/4,905). The correction cuts
in the direction of *more* merging than either source claimed, so notebook 33's conclusion that
this recipe merges far more than the external 93.6%-singleton prior suggested stands, and stands
more strongly. Corrected here; notebook 33's own files are not edited.

The external 93.6%-Butina-singleton figure remains unverifiable against anything in this
repository and is not used as a comparison basis.

In [4]:
# ---- Part 1.3: FIVE repeats of cluster-to-fold assignment, seeds 0-4.
# Rule, identical in every repeat: whole clusters largest-first to the currently smallest
# fold. The seed enters only as the TIE-BREAK, in the two places ties arise:
#   (a) between clusters of EQUAL size -- broken by that repeat's random permutation,
#       applied before a stable sort by descending size;
#   (b) between folds with EQUAL current counts -- broken by that repeat's own RNG.
# Notebook 33's own rule broke both ties deterministically by lowest index (a stable
# argsort, then np.argmin), so it is one particular draw from this family rather than a
# member of the seeded set; the reproduction check below reports exactly how far apart
# they are.
def assign_clusters_to_folds(clusters, cluster_sizes, seed, n_folds=N_FOLDS):
    rng = np.random.default_rng(seed)
    perm = rng.permutation(len(clusters))
    order = perm[np.argsort(-cluster_sizes[perm], kind="stable")]
    fold_counts = np.zeros(n_folds, dtype=int)
    assignment = {}
    for ci in order:
        smallest = np.flatnonzero(fold_counts == fold_counts.min())
        f = int(smallest[0]) if len(smallest) == 1 else int(rng.choice(smallest))
        for idx in clusters[ci]:
            assignment[idx] = f
        fold_counts[f] += len(clusters[ci])
    return assignment, fold_counts


REPEAT_COLS = [f"butina_repeat_{r}" for r in range(N_REPEATS_MAX)]
fold_sizes_by_repeat = {}
for r in range(N_REPEATS_MAX):
    assignment, fold_counts = assign_clusters_to_folds(clusters, cluster_sizes, seed=r)
    assert set(assignment.keys()) == set(range(len(curated))), \
        f"repeat {r}: not every compound landed in exactly one fold"
    assert fold_counts.sum() == len(curated)
    curated[REPEAT_COLS[r]] = [assignment[i] for i in range(len(curated))]
    fold_sizes_by_repeat[r] = fold_counts.tolist()
    print(f"repeat {r} (seed {r}): fold sizes {fold_counts.tolist()}  "
          f"(range {fold_counts.max() - fold_counts.min()})")

# Cluster-disjointness, checked directly rather than assumed, on every repeat.
cluster_of = np.empty(len(curated), dtype=int)
for ci, c in enumerate(clusters):
    for idx in c:
        cluster_of[idx] = ci
for r in range(N_REPEATS_MAX):
    fv = curated[REPEAT_COLS[r]].to_numpy()
    n_split = sum(len(set(fv[list(c)])) > 1 for c in clusters)
    assert n_split == 0, f"repeat {r}: {n_split} clusters are split across folds -- HARD FAILURE"
print("\nCONFIRMED: 0 clusters split across folds, in all 5 repeats (cluster-disjointness holds).")

repeat 0 (seed 0): fold sizes [981, 981, 981, 981, 981]  (range 0)
repeat 1 (seed 1): fold sizes [981, 981, 981, 981, 981]  (range 0)
repeat 2 (seed 2): fold sizes [981, 981, 981, 981, 981]  (range 0)
repeat 3 (seed 3): fold sizes [981, 981, 981, 981, 981]  (range 0)
repeat 4 (seed 4): fold sizes [981, 981, 981, 981, 981]  (range 0)

CONFIRMED: 0 clusters split across folds, in all 5 repeats (cluster-disjointness holds).


In [5]:
# ---- Part 1.3 (cont.): does repeat 0 reproduce notebook 33's own partition?
nb33_folds = pd.read_csv(BUTINA1_PATH)
assert len(nb33_folds) == 4905
nb33_merged = curated[["inchikey", "butina_repeat_0"]].merge(
    nb33_folds[["inchikey", "butina_fold_0"]], on="inchikey", how="left")
assert nb33_merged["butina_fold_0"].notna().all(), "a curated compound is missing from cv_folds_butina.csv"

exact = bool((nb33_merged["butina_repeat_0"] == nb33_merged["butina_fold_0"]).all())
cont = pd.crosstab(nb33_merged["butina_fold_0"], nb33_merged["butina_repeat_0"])
row_ind, col_ind = linear_sum_assignment(-cont.to_numpy())
best_overlap = float(cont.to_numpy()[row_ind, col_ind].sum() / len(nb33_merged))
print(f"repeat 0 reproduces cv_folds_butina.csv exactly (label-for-label): {exact}")
print(f"best-case (Hungarian-matched) agreement with notebook 33's partition: {best_overlap:.1%}")

REPEAT0_REPRODUCES_NB33 = exact
if not exact:
    msg = (f"Repeat 0 does not reproduce notebook 33's own partition (best-case agreement "
           f"{best_overlap:.1%}). Notebook 33 broke both ties deterministically by lowest index "
           f"(stable argsort, then np.argmin over fold counts); the brief specifies the tie be "
           f"broken by that repeat's seed, which makes nb33's draw one particular member of the "
           f"family rather than seed 0's member. Carrying on with the five seeded assignments, "
           f"per the brief's own instruction -- notebook 33's results therefore sit on a "
           f"partition that is NOT in this notebook's set of five.")
    print(f"\n*** REPORTED DIFFERENCE (informative, not fatal) ***\n{msg}")
    record_decision(
        decision="Carried on with five seeded cluster-to-fold assignments although repeat 0 does not reproduce notebook 33's partition label-for-label",
        reason=msg,
        alternatives=("Force repeat 0 to be notebook 33's own deterministic assignment and seed only "
                      "repeats 1-4 (would make the five repeats non-exchangeable, so the across-repeat "
                      "spread would no longer estimate one thing); or abandon the repeated design."),
        authority="the brief's own instruction for this exact outcome: 'report the difference and carry on using the new assignments for all five repeats'")
else:
    print("\nrepeat 0 reproduces notebook 33's partition exactly -- notebook 33's own five-fold "
          "results therefore sit inside this notebook's repeat 0.")

repeat 0 reproduces cv_folds_butina.csv exactly (label-for-label): False
best-case (Hungarian-matched) agreement with notebook 33's partition: 22.4%

*** REPORTED DIFFERENCE (informative, not fatal) ***
Repeat 0 does not reproduce notebook 33's own partition (best-case agreement 22.4%). Notebook 33 broke both ties deterministically by lowest index (stable argsort, then np.argmin over fold counts); the brief specifies the tie be broken by that repeat's seed, which makes nb33's draw one particular member of the family rather than seed 0's member. Carrying on with the five seeded assignments, per the brief's own instruction -- notebook 33's results therefore sit on a partition that is NOT in this notebook's set of five.


In [6]:
# ---- Part 1.4: per repeat, per fold, per isoform LABELLED compound counts, and the
# 15%-of-an-isoform's-labels flag. Reported, never reshuffled to look balanced.
rows = []
for r in range(N_REPEATS_MAX):
    for f in range(N_FOLDS):
        sel = curated[REPEAT_COLS[r]] == f
        row = {"repeat": r, "fold": f, "n_compounds": int(sel.sum())}
        for iso in ISOFORMS:
            col = PIC50_COL[iso]
            n = int((sel & curated[col].notna()).sum())
            row[iso] = n
            row[f"{iso}_frac"] = n / int(curated[col].notna().sum())
        rows.append(row)
label_counts = pd.DataFrame(rows)
label_counts.to_csv(OUT / "fold_isoform_label_counts.csv", index=False)

frac_cols = [f"{iso}_frac" for iso in ISOFORMS]
print("per-repeat, per-fold labelled-compound counts (fraction of that isoform's total):")
for r in range(N_REPEATS_MAX):
    sub = label_counts[label_counts["repeat"] == r]
    print(f"  repeat {r}: " + "  ".join(
        f"{iso} " + "/".join(f"{v:.3f}" for v in sub[f'{iso}_frac']) for iso in ISOFORMS))

FLAG = 0.15
flagged = label_counts.melt(id_vars=["repeat", "fold"], value_vars=frac_cols,
                            var_name="isoform", value_name="frac")
flagged = flagged[flagged["frac"] < FLAG]
print(f"\nmin fraction anywhere: {label_counts[frac_cols].to_numpy().min():.4f}   "
      f"max: {label_counts[frac_cols].to_numpy().max():.4f}")
if len(flagged):
    print(f"\n*** DESIGN LIMITATION, REPORTED PROMINENTLY: {len(flagged)} (repeat, fold, isoform) "
          f"cells hold fewer than {FLAG:.0%} of that isoform's labels: ***")
    print(flagged.to_string(index=False))
    print("Not reshuffled -- the partition is cluster-disjoint by construction and re-drawing it "
          "until it looked balanced would make the balance a selected property.")
else:
    print(f"\nNo (repeat, fold, isoform) cell falls below the {FLAG:.0%} flag threshold -- "
          "no balance limitation to report.")
LABEL_BALANCE_FLAGGED = flagged.copy()

per-repeat, per-fold labelled-compound counts (fraction of that isoform's total):
  repeat 0: CYP1A2 0.210/0.183/0.217/0.198/0.192  CYP2C9 0.201/0.218/0.212/0.186/0.184  CYP2D6 0.207/0.177/0.192/0.212/0.212  CYP3A4 0.200/0.209/0.197/0.194/0.200
  repeat 1: CYP1A2 0.196/0.188/0.219/0.195/0.202  CYP2C9 0.215/0.206/0.197/0.183/0.199  CYP2D6 0.198/0.202/0.230/0.191/0.180  CYP3A4 0.200/0.206/0.179/0.207/0.209
  repeat 2: CYP1A2 0.199/0.215/0.199/0.198/0.190  CYP2C9 0.202/0.187/0.204/0.207/0.201  CYP2D6 0.196/0.209/0.203/0.208/0.184  CYP3A4 0.202/0.197/0.192/0.197/0.212
  repeat 3: CYP1A2 0.203/0.202/0.185/0.208/0.203  CYP2C9 0.202/0.201/0.202/0.198/0.196  CYP2D6 0.218/0.196/0.196/0.204/0.186  CYP3A4 0.197/0.204/0.205/0.189/0.204
  repeat 4: CYP1A2 0.201/0.186/0.209/0.205/0.199  CYP2C9 0.201/0.197/0.198/0.199/0.205  CYP2D6 0.194/0.209/0.184/0.202/0.212  CYP3A4 0.194/0.201/0.204/0.200/0.201

min fraction anywhere: 0.1768   max: 0.2297

No (repeat, fold, isoform) cell falls below the 15% flag 

In [7]:
# ---- Part 1.5: write the NEW partition file. cv_folds.csv and cv_folds_butina.csv are
# never opened for writing anywhere in this notebook and are asserted untouched at the end.
out_folds = curated[["Molecule_Name", "inchikey"] + REPEAT_COLS].copy()
out_folds.to_csv(BUTINA5X5_PATH, index=False)
sha = hashlib.sha256(BUTINA5X5_PATH.read_bytes()).hexdigest()
print(f"wrote {BUTINA5X5_PATH}")
print(f"rows: {len(out_folds)}   columns: {out_folds.columns.tolist()}")
print(f"sha256: {sha}")
(OUT / "cv_folds_butina_5x5_sha256.txt").write_text(sha + "\n")

reread = pd.read_csv(BUTINA5X5_PATH)
assert len(reread) == 4905
assert (reread["inchikey"].values == curated["inchikey"].values).all()
for rc in REPEAT_COLS:
    assert (reread[rc].values == curated[rc].values).all()
print("round-trip confirmed: re-read file matches the in-memory assignment exactly.")
cv_folds_butina5 = reread

# Both protected fold files are read here (never written) purely to confirm they are intact.
_r = pd.read_csv(FOLDS_PATH); _b = pd.read_csv(BUTINA1_PATH)
print(f"\ndata/folds/cv_folds.csv        {_r.shape} -- read-only")
print(f"data/folds/cv_folds_butina.csv {_b.shape} -- read-only")

wrote /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/data/folds/cv_folds_butina_5x5.csv
rows: 4905   columns: ['Molecule_Name', 'inchikey', 'butina_repeat_0', 'butina_repeat_1', 'butina_repeat_2', 'butina_repeat_3', 'butina_repeat_4']
sha256: 52f39b1a1cea9200b58dce0a061d4a21ce94d9d58ec77067cf6508b0f4b3e87e
round-trip confirmed: re-read file matches the in-memory assignment exactly.

data/folds/cv_folds.csv        (4905, 7) -- read-only
data/folds/cv_folds_butina.csv (4905, 3) -- read-only


## Part 2 — The five arms

All five arms run on the **same** folds, with **one seed per (repeat, fold) shared across arms**, so
every difference between arms is attributable to the arm and not to the split or the seed.

| arm | what it is | rows | heads |
|---|---|---|---|
| `plain` | `chemprop_chemeleoninit`, multitask, point-estimate target, MSE loss | 4,905 | 4 |
| `aid` | the same plus notebook 28/29's four AID 1851 auxiliary heads | 21,364 | 8 |
| `single` | `chemprop_chemeleoninit` trained **single-task**, one model per isoform | per-isoform | 1 x 4 runs |
| `trees` | RF / XGBoost / LightGBM on `ecfp4_narrow` and `mordred_pca` — raw fingerprint and descriptor features, **never** CheMeleon embeddings | 4,905 | 4 (one fit per endpoint) |
| `deadzone` | the plain architecture against `clip(oof_pred, conf_low, conf_high)` under absolute-error loss | 4,905 | 4 |

**Why SINGLE and TREES are here at all.** Notebook 33 found a ridge head on frozen CheMeleon
embeddings never decorrelates from Chemprop, which is unsurprising — it is downstream of the same
representation. TREES on raw features is the real test of whether the error correlation this
project keeps measuring (0.79–0.91, notebook 24; 0.850–0.933 for AID, notebook 29) is a property
of the **models** or of the **compounds**. SINGLE varies task scope while holding architecture
fixed, an axis this project has never tested for decorrelation.

**DEADZONE's dependency is handled structurally, not by convention.** Its target must come from
out-of-fold predictions on *this* partition. `outputs/11_caruana_prep/` holds OOF predictions on
the **random** partition and is never used for it. DEADZONE for repeat *r* therefore only runs
after PLAIN has completed all five folds of repeat *r*; if PLAIN's repeat *r* is incomplete the
DEADZONE arm for that repeat is **skipped and recorded**, never backed off to the random-partition
OOF.

In [8]:
# ---- AID table construction, reproduced verbatim from notebook 29 cells 6-7 (via notebook 33
# cell 16, which reproduced the same code). Not reimplemented.
aid_curated = pd.read_csv(AID_CURATED_PATH)
NB28_CACHE = NB28_OUT / "aid1851_max_response.csv"
NB33_CACHE = NB33_OUT / "aid1851_max_response.csv"
LOCAL_CACHE = OUT / "aid1851_max_response.csv"

if NB28_CACHE.exists():
    aid_eff = pd.read_csv(NB28_CACHE)
    print(f"[reusing notebook 28's cache, READ-ONLY] {NB28_CACHE} ({len(aid_eff)} rows)")
elif NB33_CACHE.exists():
    aid_eff = pd.read_csv(NB33_CACHE)
    print(f"[reusing notebook 33's cache, READ-ONLY] {NB33_CACHE} ({len(aid_eff)} rows)")
elif LOCAL_CACHE.exists():
    aid_eff = pd.read_csv(LOCAL_CACHE)
    print(f"[cached locally] {LOCAL_CACHE} ({len(aid_eff)} rows)")
else:
    from src.features import canonicalize_smiles, smiles_to_inchikey
    print("no cache found -- re-deriving Max_Response from the raw batches")
    frames = []
    for f in sorted(AID_RAW_DIR.glob("batch_*.csv")):
        df = pd.read_csv(f, low_memory=False)
        frames.append(df[df["PUBCHEM_SID"].notna()])
    raw = pd.concat(frames, ignore_index=True)
    for col in ["PUBCHEM_SID", "PUBCHEM_CID", "Fit_LogAC50", "Max_Response"]:
        raw[col] = pd.to_numeric(raw[col], errors="coerce")
    raw["isoform"] = raw["Panel Name"].map(PANEL_TO_ISOFORM)
    raw = raw[raw["isoform"].notna()].copy()
    raw = raw.dropna(subset=["PUBCHEM_CID", "PUBCHEM_EXT_DATASOURCE_SMILES"]).copy()
    uniq = raw[["PUBCHEM_CID", "PUBCHEM_EXT_DATASOURCE_SMILES"]].drop_duplicates("PUBCHEM_CID").copy()
    uniq["canonical_smiles"] = uniq["PUBCHEM_EXT_DATASOURCE_SMILES"].apply(canonicalize_smiles)
    uniq["inchikey"] = uniq["PUBCHEM_EXT_DATASOURCE_SMILES"].apply(smiles_to_inchikey)
    raw = raw.merge(uniq[["PUBCHEM_CID", "canonical_smiles", "inchikey"]], on="PUBCHEM_CID", how="left")
    raw = raw.dropna(subset=["inchikey"]).copy()
    aid_eff = raw.groupby(["inchikey", "isoform"], as_index=False).agg(
        max_response=("Max_Response", "mean"), n_replicates_eff=("Max_Response", "size"))
    aid_eff.to_csv(LOCAL_CACHE, index=False)
    print(f"wrote {LOCAL_CACHE}")

aid_eff["max_inhibition"] = -aid_eff["max_response"]

cur_counts = aid_curated["isoform"].value_counts().sort_index()
eff_counts = aid_eff["isoform"].value_counts().sort_index()
assert (cur_counts == eff_counts).all(), "re-derived population does not match aid1851_curated.csv"
assert set(aid_curated["inchikey"]) == set(aid_eff["inchikey"]), "compound sets differ"

aid = aid_curated.merge(aid_eff[["inchikey", "isoform", "max_response", "max_inhibition"]],
                        on=["inchikey", "isoform"], how="inner")
assert len(aid) == len(aid_curated), "join changed row count"

blind_ik = set(blind["inchikey"].dropna())
BLIND_OVERLAP = sorted(set(aid["inchikey"]) & blind_ik)
print(f"\nAID compounds also in the blind 750 -- EXCLUDED ({len(BLIND_OVERLAP)}): {BLIND_OVERLAP}")
aid = aid[~aid["inchikey"].isin(blind_ik)].copy()
assert len(set(aid["inchikey"]) & blind_ik) == 0

aid_wide = aid.pivot(index="inchikey", columns="isoform", values="max_inhibition")
aid_wide.columns = [f"{c}_AID1851_max_inhibition" for c in aid_wide.columns]
aid_wide = aid_wide.reset_index().merge(
    aid[["inchikey", "canonical_smiles"]].drop_duplicates("inchikey"), on="inchikey", how="left")
aid_wide = aid_wide[["inchikey", "canonical_smiles"] + AUX_COLS]
assert aid_wide["inchikey"].is_unique

curated_ik = set(curated["inchikey"].dropna())
BASE_COLS = ["Molecule_Name", "inchikey", "canonical_smiles"]

curated_part = curated[BASE_COLS + list(REGRESSION_ENDPOINTS)].copy()
n_before = len(curated_part)
curated_part = curated_part.merge(aid_wide[["inchikey"] + AUX_COLS], on="inchikey", how="left")
assert len(curated_part) == n_before
n_attached = int(curated_part[AUX_COLS].notna().any(axis=1).sum())

extra_part = aid_wide[~aid_wide["inchikey"].isin(curated_ik)].copy()
extra_part["Molecule_Name"] = "AID1851_" + extra_part["inchikey"]
for c in REGRESSION_ENDPOINTS:
    extra_part[c] = np.nan
extra_part = extra_part[BASE_COLS + list(REGRESSION_ENDPOINTS) + AUX_COLS]

FULL_POP_RAW = pd.concat([curated_part, extra_part], ignore_index=True)
EXTRA_IK = set(extra_part["inchikey"])
assert FULL_POP_RAW["inchikey"].is_unique and FULL_POP_RAW["Molecule_Name"].is_unique
assert FULL_POP_RAW["canonical_smiles"].notna().all()

NB_ROWS, NB_ATTACHED, NB_EXTRA, NB_AUX_NONNULL = 21364, 98, 16459, 16557
checks = [("FULL_POP row count", len(FULL_POP_RAW), NB_ROWS),
          ("attached overlapping compounds", n_attached, NB_ATTACHED),
          ("AID-only extra compounds", len(extra_part), NB_EXTRA)]
for c in AUX_COLS:
    checks.append((f"non-null {c}", int(FULL_POP_RAW[c].notna().sum()), NB_AUX_NONNULL))
ok = True
for name, got, want in checks:
    good = got == want
    ok &= good
    print(f"  {name:45s} got={got:8d}  want={want:8d}  {'OK' if good else 'MISMATCH'}")
assert ok, "augmented table does not match notebooks 28/29's own row counts -- HARD FAILURE"
print("\nCONFIRMED: augmented table matches notebooks 28/29/33's own construction exactly.")

[reusing notebook 28's cache, READ-ONLY] /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/28_external_data/aid1851_max_response.csv (66232 rows)



AID compounds also in the blind 750 -- EXCLUDED (1): ['RWZAIHHFBKOVIK-UHFFFAOYSA-N']
  FULL_POP row count                            got=   21364  want=   21364  OK
  attached overlapping compounds                got=      98  want=      98  OK
  AID-only extra compounds                      got=   16459  want=   16459  OK
  non-null CYP1A2_AID1851_max_inhibition        got=   16557  want=   16557  OK
  non-null CYP2C9_AID1851_max_inhibition        got=   16557  want=   16557  OK
  non-null CYP2D6_AID1851_max_inhibition        got=   16557  want=   16557  OK
  non-null CYP3A4_AID1851_max_inhibition        got=   16557  want=   16557  OK

CONFIRMED: augmented table matches notebooks 28/29/33's own construction exactly.


### Seeds — one per (repeat, fold), shared across arms

The two partitions' fold *indices* carry no shared meaning, so mapping notebook 05's
`manifest.csv` seeds onto Butina folds by index is a **convention, not a correspondence**. It is
used anyway, for two reasons: it reuses this project's own established 25-seed table rather than
inventing a new one (the most conservative option), and the seed is held constant across arms
within a fold, so whatever the mapping is, it cancels out of every arm-vs-arm comparison.

In [9]:
manifest = pd.read_csv(MANIFEST_PATH)
mf = manifest[manifest["config"] == "chemprop_chemeleoninit"].sort_values(["repeat", "fold"])
assert len(mf) == 25, f"expected 25 chemprop_chemeleoninit manifest rows, got {len(mf)}"
FOLD_SEED = {(int(r.repeat), int(r.fold)): int(r.seed) for r in mf.itertuples()}
KNOWN_SEEDS_3 = [2684470948, 4091952314, 233227757]
assert [FOLD_SEED[(0, f)] for f in range(3)] == KNOWN_SEEDS_3, \
    "repeat_0's first three manifest seeds do not match this project's established SEEDS list"
print("confirmed: this project's established SEEDS = "
      f"{KNOWN_SEEDS_3} is exactly repeat_0's first three manifest fold seeds.")
print("\n(repeat, fold) -> seed:")
for r in range(N_REPEATS_MAX):
    print(f"  repeat {r}: " + "  ".join(f"f{f}={FOLD_SEED[(r,f)]}" for f in range(N_FOLDS)))

confirmed: this project's established SEEDS = [2684470948, 4091952314, 233227757] is exactly repeat_0's first three manifest fold seeds.

(repeat, fold) -> seed:
  repeat 0: f0=2684470948  f1=4091952314  f2=233227757  f3=3276785861  f4=3644269654
  repeat 1: f0=1206282609  f1=3543069911  f2=3479688010  f3=877132087  f4=1244265337
  repeat 2: f0=2740186692  f1=3884972972  f2=829139639  f3=2545567356  f4=2718398101
  repeat 3: f0=176092387  f1=3528048386  f2=569531098  f3=1337295874  f4=612016888
  repeat 4: f0=232150720  f1=2324106676  f2=1658700540  f3=609942288  f4=3405943863


### Per-fold winsorisation of the AID auxiliary columns

Reproduced from notebook 29's own correction: bounds are recomputed **per (repeat, fold)** on that
fold's own training portion, never on the pooled population — carrying one fixed set of bounds
across folds would leak each fold's held-out compounds into its own clip thresholds.

In [10]:
def fold_test_inchikeys(repeat, fold):
    return set(cv_folds_butina5.loc[cv_folds_butina5[REPEAT_COLS[repeat]] == fold, "inchikey"])


def winsorise_for_fold(full_pop_raw, repeat, fold):
    test_ik = fold_test_inchikeys(repeat, fold)
    train_mask = ~full_pop_raw["inchikey"].isin(test_ik)
    pop = full_pop_raw.copy()
    bounds = {}
    for c in AUX_COLS:
        vals = pop.loc[train_mask, c].dropna()
        lo = float(np.percentile(vals, WINSOR_LOW_PCT))
        hi = float(np.percentile(vals, WINSOR_HIGH_PCT))
        bounds[c] = (lo, hi)
        pop[c] = pop[c].clip(lower=lo, upper=hi)
    return pop, bounds, int(train_mask.sum())


bound_rows = []
for r in range(N_REPEATS_MAX):
    for f in range(N_FOLDS):
        _, bnds, n_train = winsorise_for_fold(FULL_POP_RAW, r, f)
        row = {"repeat": r, "fold": f, "n_train_portion": n_train}
        for c in AUX_COLS:
            row[f"{c.split('_')[0]}_low"] = bnds[c][0]
            row[f"{c.split('_')[0]}_high"] = bnds[c][1]
        bound_rows.append(row)
bounds_df = pd.DataFrame(bound_rows)
bounds_df.to_csv(OUT / "winsor_bounds_by_fold.csv", index=False)
print(f"winsorisation bounds across all {len(bounds_df)} (repeat, fold) cells:")
for iso in ISOFORMS:
    lo, hi = bounds_df[f"{iso}_low"], bounds_df[f"{iso}_high"]
    print(f"  {iso:9s} low [{lo.min():.3f}, {lo.max():.3f}] range={lo.max()-lo.min():.3f}   "
          f"high [{hi.min():.3f}, {hi.max():.3f}] range={hi.max()-hi.min():.3f}")

winsorisation bounds across all 25 (repeat, fold) cells:


  CYP1A2    low [-13.206, -13.190] range=0.016   high [116.239, 116.297] range=0.057
  CYP2C9    low [-184.353, -184.295] range=0.058   high [121.180, 121.184] range=0.004
  CYP2D6    low [-29.332, -29.328] range=0.003   high [100.335, 100.357] range=0.022
  CYP3A4    low [-75.076, -74.493] range=0.582   high [108.115, 108.135] range=0.020


### Building the population per (repeat, fold, arm)

`assign_screen_split` (`src/features.py`, unmodified) is reused directly: passing the new partition
with `repeat_col="butina_repeat_{r}"` works unchanged, since the function only ever compares
`cv_folds[repeat_col] == test_fold`. No new split logic is written anywhere in this notebook.

In [11]:
CHEMPROP_ARMS = ["plain", "aid", "single", "deadzone"]
TREE_ARMS = [f"{feat}__{algo}" for feat in ["ecfp4_narrow", "mordred_pca"]
             for algo in ["rf", "xgboost", "lightgbm"]]
ALL_ARMS = ["plain"] + TREE_ARMS + ["single", "aid", "deadzone"]
ARM_ORDER_IN_REPEAT = ["plain", "trees", "single", "aid", "deadzone"]
print(f"tree arms ({len(TREE_ARMS)}): {TREE_ARMS}")
print(f"all arms ({len(ALL_ARMS)}): {ALL_ARMS}")

LOSS_ARGS = {"plain": ["--loss-function", "mse"],
             "aid": ["--loss-function", "mse"],
             "single": ["--loss-function", "mse"],
             "deadzone": ["--loss-function", "mae"]}
ARCH_ARGS = {a: CHEMELEON_ARCH_ARGS + LOSS_ARGS[a] for a in LOSS_ARGS}

# Confirm chemprop's mae loss really is plain absolute error, against the installed API --
# notebook 31's own check, repeated rather than assumed from the flag name.
import inspect
import chemprop.nn as cp_nn
_mae_src = inspect.getsource(cp_nn.LossFunctionRegistry["mae"]).replace(" ", "")
assert "(preds-targets).abs()" in _mae_src, "unexpected mae loss implementation -- stopping"
assert cp_nn.PredictorRegistry["regression"]._T_default_criterion is cp_nn.LossFunctionRegistry["mse"]
print("\nCONFIRMED against the installed chemprop: 'mae' computes (preds - targets).abs(); the")
print("regression default is mse. Both are passed explicitly rather than relied on implicitly.")


def build_population(full_pop, repeat, fold, seed, arm, curated_source=None):
    # `curated_source` supplies the 4 scored target columns; None means the real curated
    # labels. DEADZONE passes its own clipped-target table. The SPLIT never depends on it.
    split_df = assign_screen_split(cv_folds_butina5, repeat_col=REPEAT_COLS[repeat],
                                   test_fold=fold, val_fraction=VAL_FRACTION, seed=seed)
    use_aux = arm == "aid"
    cols = BASE_COLS + list(REGRESSION_ENDPOINTS) + (AUX_COLS if use_aux else [])
    src_pop = full_pop if use_aux else full_pop[full_pop["inchikey"].isin(curated_ik)]
    pop = src_pop[cols].copy()
    if curated_source is not None:
        pop = pop.drop(columns=list(REGRESSION_ENDPOINTS)).merge(
            curated_source[["inchikey"] + list(REGRESSION_ENDPOINTS)], on="inchikey", how="left")
        assert len(pop) == len(src_pop), "target-substitution merge changed row count"
    pop = pop.merge(split_df[["inchikey", "screen_split"]], on="inchikey", how="left")
    assert len(pop) == len(src_pop), "split merge changed row count"
    pop.loc[pop["screen_split"].isna(), "screen_split"] = "screen_inner_train"
    assert pop["screen_split"].isin(["screen_test", "screen_inner_train", "screen_inner_val"]).all()
    return pop


ARM_TARGET_COLS = {"plain": list(REGRESSION_ENDPOINTS),
                   "aid": list(REGRESSION_ENDPOINTS) + AUX_COLS,
                   "deadzone": list(REGRESSION_ENDPOINTS)}

print("\nstructural checks on the split construction (repeat 0, folds 0/2/4):")
for f in [0, 2, 4]:
    seed = FOLD_SEED[(0, f)]
    wp, _, _ = winsorise_for_fold(FULL_POP_RAW, 0, f)
    pa = build_population(wp, 0, f, seed, "aid")
    pb = build_population(wp, 0, f, seed, "plain")
    ta = set(pa.loc[pa["screen_split"] == "screen_test", "Molecule_Name"])
    tb = set(pb.loc[pb["screen_split"] == "screen_test", "Molecule_Name"])
    assert ta == tb, f"held-out sets differ between arms at fold {f}"
    held_ik = set(pa.loc[pa["screen_split"] == "screen_test", "inchikey"])
    va = set(pa.loc[pa["screen_split"] == "screen_inner_val", "inchikey"])
    assert held_ik.isdisjoint(EXTRA_IK), f"an AID-only compound was held out at fold {f}"
    assert va.isdisjoint(EXTRA_IK), f"an AID-only compound is in the validation set at fold {f}"
    assert held_ik == fold_test_inchikeys(0, f), f"held-out set is not this fold's own cluster block"
    print(f"  fold {f} (seed {seed}): held-out identical across arms ({len(ta)} compounds); "
          f"aid split {pa['screen_split'].value_counts().to_dict()}")
print("\nCONFIRMED: held-out sets identical across arms; AID-only extras never held out or validated on.")

tree arms (6): ['ecfp4_narrow__rf', 'ecfp4_narrow__xgboost', 'ecfp4_narrow__lightgbm', 'mordred_pca__rf', 'mordred_pca__xgboost', 'mordred_pca__lightgbm']
all arms (10): ['plain', 'ecfp4_narrow__rf', 'ecfp4_narrow__xgboost', 'ecfp4_narrow__lightgbm', 'mordred_pca__rf', 'mordred_pca__xgboost', 'mordred_pca__lightgbm', 'single', 'aid', 'deadzone']



CONFIRMED against the installed chemprop: 'mae' computes (preds - targets).abs(); the
regression default is mse. Both are passed explicitly rather than relied on implicitly.

structural checks on the split construction (repeat 0, folds 0/2/4):
  fold 0 (seed 2684470948): held-out identical across arms (981 compounds); aid split {'screen_inner_train': 19794, 'screen_test': 981, 'screen_inner_val': 589}
  fold 2 (seed 233227757): held-out identical across arms (981 compounds); aid split {'screen_inner_train': 19794, 'screen_test': 981, 'screen_inner_val': 589}
  fold 4 (seed 3644269654): held-out identical across arms (981 compounds); aid split {'screen_inner_train': 19794, 'screen_test': 981, 'screen_inner_val': 589}

CONFIRMED: held-out sets identical across arms; AID-only extras never held out or validated on.


### TREES — feature matrices

`load_feature_matrix` is reproduced verbatim from `scripts/run_5x5_cv_comparison.py` (the same
reuse notebook 05c made), so the features, the `split == "train"` filter and the index alignment
are this project's own, unchanged. Only `ecfp4_narrow` (2,048-bit ECFP4 plus this project's narrow
descriptor block) and `mordred_pca` are used — **the CheMeleon embedding matrix is deliberately
not loaded**, because notebook 33 already established that a head downstream of the same
representation cannot decorrelate from Chemprop, which is the question TREES exists to test.

In [12]:
FEATURE_FILES = {"ecfp4_narrow": ECFP4_NARROW_PATH, "mordred_pca": MORDRED_PCA_PATH}


def load_feature_matrix(name):
    # Reproduced verbatim from scripts/run_5x5_cv_comparison.py's own load_feature_matrix
    # (CSV branch only -- the .npy/CheMeleon branch is deliberately unused here).
    path = FEATURE_FILES[name]
    df = pd.read_csv(path)
    if "split" in df.columns:
        df = df[df["split"] == "train"].reset_index(drop=True)
    index_df = df[["Molecule_Name", "inchikey"]].reset_index(drop=True)
    drop_cols = [c for c in ["Molecule_Name", "inchikey", "split"] if c in df.columns]
    X = df.drop(columns=drop_cols).to_numpy(dtype=np.float64)
    return index_df, X


FEATURES = {}
for name in FEATURE_FILES:
    idx, X = load_feature_matrix(name)
    FEATURES[name] = (idx, X)
    assert len(idx) == 4905, f"{name}: expected 4905 training rows, got {len(idx)}"
    assert set(idx["inchikey"]) == curated_ik, f"{name}: compound set does not match curated"
    print(f"{name:14s} X={X.shape}  finite={np.isfinite(X).all()}")


def run_trees_one(config, repeat, fold, seed):
    # Reproduced from scripts/run_5x5_cv_comparison.py's run_tabular_or_naive: RF fits on the
    # pooled inner train+val (no early stopping concept); XGBoost/LightGBM fit on inner-train
    # with inner-val as the early-stopping eval set. One fit per endpoint, as there.
    feature_name, algo = config.split("__")
    feat_index, X = FEATURES[feature_name]
    population = build_population(FULL_POP_RAW, repeat, fold, seed, "plain")
    pop_pos = feat_index.reset_index().merge(population, on=["Molecule_Name", "inchikey"], how="left")
    assert pop_pos["screen_split"].notna().all(), f"{config}: feature index did not align 1:1"

    pool_mask = pop_pos["screen_split"].isin(["screen_inner_train", "screen_inner_val"])
    inner_train_mask = pop_pos["screen_split"] == "screen_inner_train"
    inner_val_mask = pop_pos["screen_split"] == "screen_inner_val"
    test_sel = pop_pos["screen_split"] == "screen_test"
    test_positions = pop_pos.loc[test_sel, "index"].to_numpy()

    pred_df = pop_pos.loc[test_sel, ["Molecule_Name", "inchikey"]].reset_index(drop=True)
    for endpoint in REGRESSION_ENDPOINTS:
        has_label = pop_pos[endpoint].notna()
        if algo == "rf":
            idx = pop_pos.loc[pool_mask & has_label, "index"].to_numpy()
            y = pop_pos.loc[pool_mask & has_label, endpoint].to_numpy()
            model = RandomForestRegressor(random_state=seed)
            model.fit(X[idx], y)
        else:
            tr_idx = pop_pos.loc[inner_train_mask & has_label, "index"].to_numpy()
            va_idx = pop_pos.loc[inner_val_mask & has_label, "index"].to_numpy()
            y_tr = pop_pos.loc[inner_train_mask & has_label, endpoint].to_numpy()
            y_va = pop_pos.loc[inner_val_mask & has_label, endpoint].to_numpy()
            if algo == "xgboost":
                model = XGBRegressor(early_stopping_rounds=XGB_LGBM_EARLY_STOPPING_ROUNDS,
                                     random_state=seed)
                model.fit(X[tr_idx], y_tr, eval_set=[(X[va_idx], y_va)], verbose=False)
            else:
                model = LGBMRegressor(random_state=seed, verbosity=-1)
                model.fit(X[tr_idx], y_tr, eval_set=[(X[va_idx], y_va)],
                          callbacks=[lgb.early_stopping(stopping_rounds=XGB_LGBM_EARLY_STOPPING_ROUNDS,
                                                        verbose=False)])
        pred_df[endpoint] = np.asarray(model.predict(X[test_positions])).reshape(-1)
    return pred_df

ecfp4_narrow   X=(4905, 2057)  finite=True
mordred_pca    X=(4905, 132)  finite=True


## Part 3 — Execution

**The run is ordered by repeat, not by arm.** All arms of repeat 0 complete, then repeat 1, and so
on. This is a deliberate design choice: if the budget runs out after three repeats it leaves a
**complete 3x5 design** rather than one finished arm and four half-finished ones. Within a repeat
the order is PLAIN (5 folds), TREES (5 folds), SINGLE (5 folds), AID (5 folds), DEADZONE (5 folds)
— DEADZONE last because it depends on PLAIN's completed repeat.

**One run per cell**, each protected by an on-disk cache-skip with a content check, so
`nbconvert`'s per-cell timeout can lose at most one run and re-executing resumes rather than
restarts. Scores are appended to `run_scores.csv` after **every single run**.

`epochs=50`, `patience=5` set explicitly; `src/chemprop_screen.py` reused unmodified (its
`run_subprocess_streamed` already passes `stdin=subprocess.DEVNULL`).

In [13]:
RUN_SCORES_PATH = OUT / "run_scores.csv"
TIMINGS_PATH = OUT / "run_timings.csv"
DESIGN_PATH = OUT / "design.json"

# How many repeats are active. Set mechanically by the timing gate after repeat 0's PLAIN and
# AID first folds complete; persisted so a resume honours the same decision.
if DESIGN_PATH.exists():
    DESIGN = json.loads(DESIGN_PATH.read_text())
    N_REPEATS_ACTIVE = int(DESIGN["n_repeats_active"])
    print(f"[resume] timing gate already fired: N_REPEATS_ACTIVE = {N_REPEATS_ACTIVE}")
else:
    DESIGN = {}
    N_REPEATS_ACTIVE = N_REPEATS_MAX
    print(f"timing gate has not fired yet -- provisionally N_REPEATS_ACTIVE = {N_REPEATS_ACTIVE}")

# A wall-clock guard, NOT in the brief -- recorded as a decision taken without approval below.
# The brief caps the run at 24 hours with no human available; the timing gate reduces the
# DESIGN but cannot react to notebook 28's documented 10.6x mid-run per-epoch degradation, so
# without a deadline an unlucky slowdown could leave the analysis sections never reached.
# Skipped runs are recorded, never silently dropped, and because the order is by repeat the
# result is still a complete design at whatever repeat count finished.
WALL_DEADLINE_H = 22.0
DEADLINE_SKIPS = []


def _deadline_exceeded():
    return (time.time() - RUN_STARTED_AT) / 3600.0 > WALL_DEADLINE_H


def _last_epoch(tag):
    m = CHEMPROP_RUNS_DIR / tag / "model_0" / "trainer_logs" / "version_0" / "metrics.csv"
    if not m.exists():
        return None
    eps = [int(r.split(",")[0]) for r in m.read_text().splitlines()[1:]
           if r and r.split(",")[0].isdigit()]
    return max(eps) if eps else None


def _append_rows(path, new_rows, key_cols):
    new_df = pd.DataFrame(new_rows)
    if path.exists():
        old = pd.read_csv(path)
        keys = set(map(tuple, new_df[key_cols].values.tolist()))
        mask = ~old[key_cols].apply(lambda r: tuple(r) in keys, axis=1)
        combined = pd.concat([old[mask], new_df], ignore_index=True)
    else:
        combined = new_df
    combined = combined.sort_values(key_cols).reset_index(drop=True)
    combined.to_csv(path, index=False)
    return combined


def score_and_append(repeat, fold, arm, seed, pred_df):
    ground_truth = curated[curated["inchikey"].isin(pred_df["inchikey"])].copy()
    with per_fold_bootstrap_seed(seed):
        scored = score_activity_predictions(pred_df, ground_truth, list(REGRESSION_ENDPOINTS))
        scored = add_macro_endpoint(scored, list(REGRESSION_ENDPOINTS), ACTIVITY_METRICS)
    scored.to_csv(SCORE_DIR / f"{arm}__r{repeat}f{fold}.csv", index=False)
    pe = scored.groupby("Endpoint")[METRIC_NAMES].mean()
    rows = []
    for endpoint in pe.index:
        row = {"repeat": repeat, "fold": fold, "arm": arm, "seed": seed, "Endpoint": endpoint}
        row.update(pe.loc[endpoint].to_dict())
        rows.append(row)
    _append_rows(RUN_SCORES_PATH, rows, ["repeat", "fold", "arm", "Endpoint"])
    return pe


def save_oof_long(repeat, fold, arm, pred_df):
    # One file per run, overwritten idempotently (never appended to), so a resume can never
    # duplicate rows. The per-isoform WIDE tables matching
    # outputs/11_caruana_prep/oof_long_{iso}.csv's own column shape are assembled from these
    # in Part 4 -- see the assembly cell there.
    frames = []
    for iso in ISOFORMS:
        ep = PIC50_COL[iso]
        m = pred_df[["inchikey", ep]].rename(columns={ep: "y_pred"}).copy()
        m["isoform"] = iso
        frames.append(m[["inchikey", "isoform", "y_pred"]])
    out = pd.concat(frames, ignore_index=True)
    out.insert(0, "fold", fold)
    out.insert(0, "repeat", repeat)
    out.insert(0, "arm", arm)
    out.to_csv(OOF_DIR / f"{arm}__r{repeat}f{fold}.csv", index=False)


def _cached(arm, repeat, fold, expected_names):
    p = PRED_DIR / f"{arm}__r{repeat}f{fold}.csv"
    if not p.exists():
        return None
    try:
        df = pd.read_csv(p)
    except Exception:
        return None
    if len(df) != len(expected_names) or set(df["Molecule_Name"]) != expected_names:
        print(f"  [cache MISMATCH] {arm}__r{repeat}f{fold}: content check failed -- will retrain")
        return None
    if df[list(REGRESSION_ENDPOINTS)].isna().to_numpy().any():
        print(f"  [cache MISMATCH] {arm}__r{repeat}f{fold}: NaN in a predicted column -- will retrain")
        return None
    return df

[resume] timing gate already fired: N_REPEATS_ACTIVE = 2


In [14]:
def run_chemprop_arm(repeat, fold, arm, curated_source=None):
    # PLAIN / AID / DEADZONE: one multitask chemprop run.
    seed = FOLD_SEED[(repeat, fold)]
    tag = f"{arm}__r{repeat}f{fold}"
    wp, _, _ = winsorise_for_fold(FULL_POP_RAW, repeat, fold) if arm == "aid" else (FULL_POP_RAW, None, None)
    population = build_population(wp, repeat, fold, seed, arm, curated_source=curated_source)
    target_cols = ARM_TARGET_COLS[arm]
    expected_names = set(population.loc[population["screen_split"] == "screen_test", "Molecule_Name"])

    hit = _cached(arm, repeat, fold, expected_names)
    if hit is not None:
        print(f"  [cache hit] {tag}")
        score_and_append(repeat, fold, arm, seed, hit)
        save_oof_long(repeat, fold, arm, hit)
        return hit, 0.0, _last_epoch(tag)

    run_dir = CHEMPROP_RUNS_DIR / tag
    run_dir.mkdir(parents=True, exist_ok=True)
    logger = setup_logging(LOG_DIR / f"{tag}.log", f"nb34_{tag}")
    logger.info(f"=== {tag}: repeat={repeat} fold={fold} arm={arm} seed={seed} ===")

    t0 = time.time()
    train_csv = run_dir / "train_input.csv"
    build_training_csv(population, target_cols, train_csv, logger, require_all_targets=False)
    predict_csv = build_predict_csv(population, run_dir, logger)
    run_chemprop_train(train_csv, target_cols, run_dir, logger, ARCH_ARGS[arm], EPOCHS, PATIENCE, seed)
    raw_pred_csv = run_dir / "raw_predictions.csv"
    run_chemprop_predict(run_dir / "model_0", predict_csv, raw_pred_csv, logger)
    verify_predictions(raw_pred_csv, expected_names, list(REGRESSION_ENDPOINTS), logger)
    elapsed = (time.time() - t0) / 60.0

    pred_df = pd.read_csv(raw_pred_csv)
    assert "inchikey" in pred_df.columns, "raw_predictions.csv lacks inchikey -- stopping"
    pred_df.to_csv(PRED_DIR / f"{tag}.csv", index=False)
    score_and_append(repeat, fold, arm, seed, pred_df)
    save_oof_long(repeat, fold, arm, pred_df)
    return pred_df, elapsed, _last_epoch(tag)


def run_single_arm(repeat, fold):
    # SINGLE: four independent single-task chemprop runs, one per isoform, assembled into one
    # prediction frame before scoring. require_all_targets=True drops rows lacking that one
    # isoform's label -- src/chemprop_screen.py's own contract for single-task models.
    seed = FOLD_SEED[(repeat, fold)]
    tag = f"single__r{repeat}f{fold}"
    population = build_population(FULL_POP_RAW, repeat, fold, seed, "plain")
    expected_names = set(population.loc[population["screen_split"] == "screen_test", "Molecule_Name"])

    hit = _cached("single", repeat, fold, expected_names)
    if hit is not None:
        print(f"  [cache hit] {tag}")
        score_and_append(repeat, fold, "single", seed, hit)
        save_oof_long(repeat, fold, "single", hit)
        return hit, 0.0, None

    logger = setup_logging(LOG_DIR / f"{tag}.log", f"nb34_{tag}")
    t0 = time.time()
    merged = None
    last_eps = []
    for endpoint in REGRESSION_ENDPOINTS:
        sub_tag = f"{tag}__{endpoint.split('_')[0]}"
        run_dir = CHEMPROP_RUNS_DIR / sub_tag
        run_dir.mkdir(parents=True, exist_ok=True)
        sub_pred = run_dir / "raw_predictions.csv"
        sub_ok = False
        if sub_pred.exists():
            try:
                _c = pd.read_csv(sub_pred)
                sub_ok = (len(_c) == len(expected_names)
                          and set(_c["Molecule_Name"]) == expected_names
                          and not _c[endpoint].isna().any())
            except Exception:
                sub_ok = False
            if not sub_ok:
                print(f"  [cache MISMATCH] {sub_tag}: content check failed -- retraining")
        if not sub_ok:
            train_csv = run_dir / "train_input.csv"
            build_training_csv(population, [endpoint], train_csv, logger, require_all_targets=True)
            predict_csv = build_predict_csv(population, run_dir, logger)
            run_chemprop_train(train_csv, [endpoint], run_dir, logger,
                               ARCH_ARGS["single"], EPOCHS, PATIENCE, seed)
            run_chemprop_predict(run_dir / "model_0", predict_csv, sub_pred, logger)
            verify_predictions(sub_pred, expected_names, [endpoint], logger)
        else:
            logger.info(f"[cache hit] {sub_tag}")
        last_eps.append(_last_epoch(sub_tag))
        part = pd.read_csv(sub_pred)
        keep = ["Molecule_Name", "inchikey", "canonical_smiles", endpoint]
        keep = [c for c in keep if c in part.columns]
        part = part[keep]
        merged = part if merged is None else merged.merge(
            part[["inchikey", endpoint]], on="inchikey", how="inner")
    elapsed = (time.time() - t0) / 60.0
    assert len(merged) == len(expected_names), "single-task assembly changed row count"
    assert set(merged["Molecule_Name"]) == expected_names
    assert not merged[list(REGRESSION_ENDPOINTS)].isna().to_numpy().any()
    merged.to_csv(PRED_DIR / f"{tag}.csv", index=False)
    score_and_append(repeat, fold, "single", seed, merged)
    save_oof_long(repeat, fold, "single", merged)
    return merged, elapsed, last_eps


def run_trees_arm(repeat, fold):
    seed = FOLD_SEED[(repeat, fold)]
    population = build_population(FULL_POP_RAW, repeat, fold, seed, "plain")
    expected_names = set(population.loc[population["screen_split"] == "screen_test", "Molecule_Name"])
    total = 0.0
    for config in TREE_ARMS:
        hit = _cached(config, repeat, fold, expected_names)
        if hit is not None:
            print(f"  [cache hit] {config}__r{repeat}f{fold}")
            score_and_append(repeat, fold, config, seed, hit)
            save_oof_long(repeat, fold, config, hit)
            continue
        t0 = time.time()
        pred_df = run_trees_one(config, repeat, fold, seed)
        el = (time.time() - t0) / 60.0
        total += el
        assert set(pred_df["Molecule_Name"]) == expected_names
        pred_df.to_csv(PRED_DIR / f"{config}__r{repeat}f{fold}.csv", index=False)
        score_and_append(repeat, fold, config, seed, pred_df)
        save_oof_long(repeat, fold, config, pred_df)
        print(f"  {config}__r{repeat}f{fold}: {el:.2f} min")
    return None, total, None


def run_unit(repeat, fold, arm_kind):
    # The single entry point every run cell calls. Handles abort state, the reduced design,
    # the wall-clock guard, timing, and the per-epoch degradation flag.
    label = f"{arm_kind}__r{repeat}f{fold}"
    if ABORT["aborted"]:
        print(f"[skipped -- RUN ABORTED earlier: {ABORT['reason']} at {ABORT['where']}] {label}")
        return None
    if repeat >= N_REPEATS_ACTIVE:
        print(f"[outside the reduced design: {N_REPEATS_ACTIVE} repeats active] {label}")
        return None
    if arm_kind == "deadzone" and not DEADZONE_READY.get(repeat, False):
        print(f"[skipped -- DEADZONE target unavailable for repeat {repeat}] {label}")
        return None
    if _deadline_exceeded():
        DEADLINE_SKIPS.append(label)
        print(f"[skipped -- {WALL_DEADLINE_H}h wall-clock guard exceeded] {label}")
        return None

    t0 = time.time()
    try:
        if arm_kind == "trees":
            _, el, ep = run_trees_arm(repeat, fold)
        elif arm_kind == "single":
            _, el, ep = run_single_arm(repeat, fold)
        elif arm_kind == "deadzone":
            _, el, ep = run_chemprop_arm(repeat, fold, "deadzone",
                                         curated_source=DEADZONE_TARGETS[repeat])
        else:
            _, el, ep = run_chemprop_arm(repeat, fold, arm_kind)
    except Exception as exc:
        # One retry, then a clean abort -- never improvise around a hard failure.
        print(f"  *** {label} FAILED: {type(exc).__name__}: {exc}")
        print(f"  *** retrying once ***")
        try:
            if arm_kind == "trees":
                _, el, ep = run_trees_arm(repeat, fold)
            elif arm_kind == "single":
                _, el, ep = run_single_arm(repeat, fold)
            elif arm_kind == "deadzone":
                _, el, ep = run_chemprop_arm(repeat, fold, "deadzone",
                                             curated_source=DEADZONE_TARGETS[repeat])
            else:
                _, el, ep = run_chemprop_arm(repeat, fold, arm_kind)
        except Exception as exc2:
            hard_fail(f"{type(exc2).__name__}: {exc2} (failed twice)", label)
            return None

    wall = (time.time() - t0) / 60.0
    row = {"repeat": repeat, "fold": fold, "arm_kind": arm_kind, "wall_min": wall,
           "last_epoch": (max(e for e in ep if e is not None) if isinstance(ep, list) and any(e is not None for e in ep) else ep)}
    row["min_per_epoch"] = (wall / row["last_epoch"]) if row.get("last_epoch") else np.nan

    # A CACHE HIT MUST NOT OVERWRITE A COMPLETED RUN'S TIMING RECORD. Re-executing this
    # notebook after training finishes (which this notebook did, to apply corrections to the
    # Part 4 analysis) replays every run cell as a cache hit taking a few hundredths of a
    # minute. Appending those would silently destroy the only record of what the training
    # actually cost -- the same class of loss the brief's "never delete or overwrite a
    # completed run's cached predictions" rule exists to prevent, applied to the timing file.
    # A unit whose wall time is implausibly short for real training is therefore recorded only
    # if nothing is on file for it yet.
    CACHE_HIT_MAX_MIN = 0.5
    is_cache_hit = wall < CACHE_HIT_MAX_MIN
    already = False
    if TIMINGS_PATH.exists():
        _t = pd.read_csv(TIMINGS_PATH)
        already = bool(((_t["repeat"] == repeat) & (_t["fold"] == fold)
                        & (_t["arm_kind"] == arm_kind)).any())
    if is_cache_hit and already:
        print(f"[cache hit] {label}: keeping the existing timing record, not overwriting it")
    else:
        _append_rows(TIMINGS_PATH, [row], ["repeat", "fold", "arm_kind"])
    print(f"{label}: {wall:.2f} min  (last_epoch={row['last_epoch']}, "
          f"min/epoch={row['min_per_epoch'] if not pd.isna(row['min_per_epoch']) else float('nan'):.3f})")

    # Per-epoch degradation flag: notebook 28 saw 10.6x on an unchanged workload; notebook 29
    # did not. Flagged beyond 3x against this arm's own fastest observed epoch -- never aborted.
    tm = pd.read_csv(TIMINGS_PATH)
    same = tm[(tm["arm_kind"] == arm_kind) & tm["min_per_epoch"].notna()]
    if len(same) > 1 and not pd.isna(row["min_per_epoch"]):
        floor = float(same["min_per_epoch"].min())
        if floor > 0 and row["min_per_epoch"] / floor > 3.0:
            print(f"  *** PER-EPOCH DEGRADATION FLAG: {row['min_per_epoch']:.3f} min/epoch is "
                  f"{row['min_per_epoch']/floor:.1f}x this arm's own fastest ({floor:.3f}) -- "
                  f"logged, NOT aborted (notebook 28's documented failure mode).")
    return True


DEADZONE_READY = {}
DEADZONE_TARGETS = {}
print("run harness ready.")

run harness ready.


### SANITY CHECK — before the main run

**This check earned its place on the first execution: it failed, and it was right to.** The first
run of this notebook copied notebook 33's setup cell verbatim, including
`os.environ.setdefault("OMP_NUM_THREADS", "1")`, and the sanity check came back at **0.0378** max
abs ST-RAE difference against a 0.03 threshold — so the run aborted cleanly at this cell, before
any of the 125 training units started. The diagnosis, and why it was not worked around: the
training input CSV was **byte-identical** to notebook 05's own (sha256 `66a1b380…`), the chemprop
CLI argv was **identical** to notebook 31's own baseline invocation, and val_loss was
**bit-identical for epochs 0 and 1** before diverging at epoch 2. Data and initialisation were
right; the arithmetic was not. Re-running with `OMP_NUM_THREADS`/`KMP_DUPLICATE_LIB_OK` unset and
nothing else changed reproduced notebook 05 **exactly** (0.00000000 on all four isoforms, with a
bit-identical val_loss trajectory). Notebooks 29 and 31 — the two that reproduce notebook 05 to
0.0000 — do not set those variables. The setup cell now asserts they are unset.

**A consequence for notebook 33, stated rather than left implicit.** Notebook 33's Chemprop runs
were produced under `OMP_NUM_THREADS=1`, and it never ran this check (it noted, correctly, that no
stored external comparison exists for a brand-new partition). Its own arm-vs-arm comparisons remain
internally consistent, since both its arms ran under the same setting. But its *absolute* Butina
numbers are not on the same numerical footing as notebook 05/29/31's random-partition numbers, which
is exactly the comparison it drew — a second reason, independent of fold count, to treat its
cross-partition conclusion as provisional.

PLAIN is retrained on the **RANDOM** partition's `repeat_0`/`fold_0` and compared against
`outputs/05_cv_comparison/scores/chemprop_chemeleoninit__repeat0_fold0.csv` (READ-ONLY).
Notebooks 18, 20, 22, 28, 29 and 31 all matched to 0.0000 max abs ST-RAE difference. The threshold
here is **0.03**; a mismatch beyond that is a **HARD FAILURE** and aborts the run, because it would
mean the pipeline has drifted and nothing downstream is interpretable.

In [15]:
from src.chemprop_screen import load_screen_population

SANITY_TAG = "sanity_plain__random_r0f0"
sanity_seed = FOLD_SEED[(0, 0)]
logger = setup_logging(LOG_DIR / f"{SANITY_TAG}.log", f"nb34_{SANITY_TAG}")
sanity_pred_path = PRED_DIR / f"{SANITY_TAG}.csv"

# Exactly scripts/run_5x5_cv_comparison.py's own run_chemprop path, on the frozen random
# partition, read-only.
sanity_pop = load_screen_population(FOLDS_PATH, CURATED_PATH, "repeat_0", 0,
                                    VAL_FRACTION, sanity_seed, logger)
sanity_expected = set(sanity_pop.loc[sanity_pop["screen_split"] == "screen_test", "Molecule_Name"])

if sanity_pred_path.exists() and set(pd.read_csv(sanity_pred_path)["Molecule_Name"]) == sanity_expected:
    print(f"[cache hit] {SANITY_TAG}")
    sanity_pred = pd.read_csv(sanity_pred_path)
else:
    run_dir = CHEMPROP_RUNS_DIR / SANITY_TAG
    run_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    train_csv = run_dir / "train_input.csv"
    build_training_csv(sanity_pop, list(REGRESSION_ENDPOINTS), train_csv, logger,
                       require_all_targets=False)
    predict_csv = build_predict_csv(sanity_pop, run_dir, logger)
    run_chemprop_train(train_csv, list(REGRESSION_ENDPOINTS), run_dir, logger,
                       ARCH_ARGS["plain"], EPOCHS, PATIENCE, sanity_seed)
    raw_pred_csv = run_dir / "raw_predictions.csv"
    run_chemprop_predict(run_dir / "model_0", predict_csv, raw_pred_csv, logger)
    verify_predictions(raw_pred_csv, sanity_expected, list(REGRESSION_ENDPOINTS), logger)
    sanity_pred = pd.read_csv(raw_pred_csv)
    sanity_pred.to_csv(sanity_pred_path, index=False)
    print(f"\nsanity run wall time: {(time.time()-t0)/60:.2f} min")

sanity_gt = curated[curated["inchikey"].isin(sanity_pred["inchikey"])].copy()
with per_fold_bootstrap_seed(sanity_seed):
    sanity_scored = score_activity_predictions(sanity_pred, sanity_gt, list(REGRESSION_ENDPOINTS))
mine = sanity_scored.groupby("Endpoint")["ST-RAE"].mean()

nb05 = pd.read_csv(NB05_OUT / "scores" / "chemprop_chemeleoninit__repeat0_fold0.csv")
theirs = nb05.groupby("Endpoint")["ST-RAE"].mean()

print(f"\nSANITY CHECK -- PLAIN retrained on random repeat_0/fold_0 vs. notebook 05's stored scores")
print(f"{'endpoint':42s} {'this run':>10s} {'nb05':>10s} {'abs diff':>10s}")
max_diff = 0.0
for ep in sorted(set(mine.index) & set(theirs.index)):
    d = abs(float(mine[ep]) - float(theirs[ep]))
    max_diff = max(max_diff, d)
    print(f"{ep:42s} {mine[ep]:10.4f} {theirs[ep]:10.4f} {d:10.4f}")
print(f"\nmax abs ST-RAE difference: {max_diff:.6f}   threshold: {SANITY_THRESHOLD}")

pd.DataFrame({"Endpoint": list(mine.index), "this_run": mine.to_numpy(),
              "nb05": [float(theirs.get(e, np.nan)) for e in mine.index]}
             ).to_csv(OUT / "sanity_check.csv", index=False)

if max_diff > SANITY_THRESHOLD:
    hard_fail(f"sanity check FAILED: max abs ST-RAE difference {max_diff:.6f} exceeds "
              f"{SANITY_THRESHOLD} against notebook 05's stored chemprop_chemeleoninit "
              f"repeat0_fold0 scores -- the pipeline has drifted and nothing downstream is "
              f"interpretable.", "Part 3 sanity check")
else:
    print("PASS -- the pipeline reproduces notebook 05's own stored numbers within threshold.")

2026-09-26 14:01:47,552 [INFO] loaded cv_folds.csv: (4905, 7)


2026-09-26 14:01:47,552 [INFO] loaded train_inhibition_curated.csv: (4905, 20)


2026-09-26 14:01:47,555 [INFO] assign_screen_split(repeat_col='repeat_0', test_fold=0, val_fraction=0.15, seed=2684470948) split counts: {'screen_inner_train': 3335, 'screen_test': 981, 'screen_inner_val': 589}


2026-09-26 14:01:47,557 [INFO] rows after SMILES/target join: 4905 (missing: 0)


2026-09-26 14:01:47,558 [INFO]   CYP1A2_pIC50_direct_inhibition: labeled-compound counts per split = {'screen_inner_train': 975, 'screen_inner_val': 160, 'screen_test': 277}


...

2026-09-26 14:01:48.961 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 686 compound(s) with no ground truth for endpoint CYP2D6_pIC50_direct_inhibition


2026-09-26 14:01:49.655 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP3A4_pIC50_direct_inhibition


2026-09-26 14:01:49.655 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 531 compound(s) with no ground truth for endpoint CYP3A4_pIC50_direct_inhibition


2026-09-26 14:01:50.388 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:169 - Completed scoring activity predictions



SANITY CHECK -- PLAIN retrained on random repeat_0/fold_0 vs. notebook 05's stored scores
endpoint                                     this run       nb05   abs diff
CYP1A2_pIC50_direct_inhibition                 0.7884     0.7884     0.0000
CYP2C9_pIC50_direct_inhibition                 0.7113     0.7113     0.0000
CYP2D6_pIC50_direct_inhibition                 0.9580     0.9580     0.0000
CYP3A4_pIC50_direct_inhibition                 0.4910     0.4910     0.0000

max abs ST-RAE difference: 0.000000   threshold: 0.03
PASS -- the pipeline reproduces notebook 05's own stored numbers within threshold.


### Run order

Repeat by repeat. Cells for repeats beyond the timing gate's decision remain in place and report
themselves as outside the reduced design rather than being deleted, so the notebook records the
full intended plan alongside what was actually run.

#### Repeat 0

In [16]:
# PLAIN -- repeat 0, fold 0
run_unit(0, 0, 'plain')

2026-09-26 14:01:50.406 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:01:50.408 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:01:50.408 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:01:50.408 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 684 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] plain__r0f0


...

2026-09-26 14:01:53.233 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:01:53.234 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:01:53.234 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] plain__r0f0: keeping the existing timing record, not overwriting it
plain__r0f0: 0.05 min  (last_epoch=12, min/epoch=0.004)


True

In [17]:
# PLAIN -- repeat 0, fold 1
run_unit(0, 1, 'plain')

2026-09-26 14:01:53.281 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:01:53.282 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:01:53.283 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:01:53.283 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 723 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] plain__r0f1


...

2026-09-26 14:01:56.100 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:01:56.101 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:01:56.101 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] plain__r0f1: keeping the existing timing record, not overwriting it
plain__r0f1: 0.05 min  (last_epoch=9, min/epoch=0.005)


True

In [18]:
# PLAIN -- repeat 0, fold 2
run_unit(0, 2, 'plain')

2026-09-26 14:01:56.148 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:01:56.149 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:01:56.150 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:01:56.150 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 675 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] plain__r0f2


...

2026-09-26 14:01:58.972 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:01:58.972 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:01:58.973 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] plain__r0f2: keeping the existing timing record, not overwriting it
plain__r0f2: 0.05 min  (last_epoch=22, min/epoch=0.002)


True

In [19]:
# PLAIN -- repeat 0, fold 3
run_unit(0, 3, 'plain')

2026-09-26 14:01:59.022 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:01:59.023 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:01:59.023 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:01:59.023 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 701 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] plain__r0f3


...

2026-09-26 14:02:01.815 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:02:01.815 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:02:01.816 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] plain__r0f3: keeping the existing timing record, not overwriting it
plain__r0f3: 0.05 min  (last_epoch=9, min/epoch=0.005)


True

In [20]:
# PLAIN -- repeat 0, fold 4
run_unit(0, 4, 'plain')

2026-09-26 14:02:01.862 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:02:01.863 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:02:01.864 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:02:01.864 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 710 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] plain__r0f4


...

2026-09-26 14:02:04.676 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:02:04.676 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:02:04.677 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] plain__r0f4: keeping the existing timing record, not overwriting it
plain__r0f4: 0.05 min  (last_epoch=7, min/epoch=0.007)


True

In [21]:
# TREES (6 configs) -- repeat 0, fold 0
run_unit(0, 0, 'trees')

2026-09-26 14:02:04.722 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:02:04.724 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 30 columns.


2026-09-26 14:02:04.724 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:02:04.724 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 684 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] ecfp4_narrow__rf__r0f0


...

2026-09-26 14:02:21.939 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:02:21.939 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:02:21.940 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] trees__r0f0: keeping the existing timing record, not overwriting it
trees__r0f0: 0.29 min  (last_epoch=None, min/epoch=nan)


True

In [22]:
# TREES (6 configs) -- repeat 0, fold 1
run_unit(0, 1, 'trees')

2026-09-26 14:02:21.988 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:02:21.989 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 30 columns.


2026-09-26 14:02:21.989 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:02:21.990 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 723 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] ecfp4_narrow__rf__r0f1


...

2026-09-26 14:02:39.110 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:02:39.111 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:02:39.111 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] trees__r0f1: keeping the existing timing record, not overwriting it
trees__r0f1: 0.29 min  (last_epoch=None, min/epoch=nan)


True

In [23]:
# TREES (6 configs) -- repeat 0, fold 2
run_unit(0, 2, 'trees')

2026-09-26 14:02:39.159 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:02:39.160 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 30 columns.


2026-09-26 14:02:39.161 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:02:39.161 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 675 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] ecfp4_narrow__rf__r0f2


...

2026-09-26 14:02:56.380 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:02:56.380 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:02:56.381 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] trees__r0f2: keeping the existing timing record, not overwriting it
trees__r0f2: 0.29 min  (last_epoch=None, min/epoch=nan)


True

In [24]:
# TREES (6 configs) -- repeat 0, fold 3
run_unit(0, 3, 'trees')

2026-09-26 14:02:56.428 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:02:56.429 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 30 columns.


2026-09-26 14:02:56.429 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:02:56.429 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 701 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] ecfp4_narrow__rf__r0f3


...

2026-09-26 14:03:13.535 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:03:13.535 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:03:13.536 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] trees__r0f3: keeping the existing timing record, not overwriting it
trees__r0f3: 0.29 min  (last_epoch=None, min/epoch=nan)


True

In [25]:
# TREES (6 configs) -- repeat 0, fold 4
run_unit(0, 4, 'trees')

2026-09-26 14:03:13.583 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:03:13.584 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 30 columns.


2026-09-26 14:03:13.584 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:03:13.585 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 710 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] ecfp4_narrow__rf__r0f4


...

2026-09-26 14:03:30.684 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:03:30.684 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:03:30.685 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] trees__r0f4: keeping the existing timing record, not overwriting it
trees__r0f4: 0.29 min  (last_epoch=None, min/epoch=nan)


True

In [26]:
# SINGLE (4 single-task models) -- repeat 0, fold 0
run_unit(0, 0, 'single')

2026-09-26 14:03:30.733 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:03:30.734 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:03:30.735 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:03:30.735 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 684 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] single__r0f0


...

2026-09-26 14:03:33.553 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:03:33.553 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:03:33.553 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] single__r0f0: keeping the existing timing record, not overwriting it
single__r0f0: 0.05 min  (last_epoch=None, min/epoch=nan)


True

In [27]:
# SINGLE (4 single-task models) -- repeat 0, fold 1
run_unit(0, 1, 'single')

2026-09-26 14:03:33.599 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:03:33.601 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:03:33.601 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:03:33.601 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 723 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] single__r0f1


...

2026-09-26 14:03:36.410 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:03:36.411 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:03:36.411 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] single__r0f1: keeping the existing timing record, not overwriting it
single__r0f1: 0.05 min  (last_epoch=None, min/epoch=nan)


True

In [28]:
# SINGLE (4 single-task models) -- repeat 0, fold 2
run_unit(0, 2, 'single')

2026-09-26 14:03:36.458 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:03:36.459 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:03:36.460 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:03:36.460 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 675 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] single__r0f2


...

2026-09-26 14:03:39.273 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:03:39.273 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:03:39.274 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] single__r0f2: keeping the existing timing record, not overwriting it
single__r0f2: 0.05 min  (last_epoch=None, min/epoch=nan)


True

In [29]:
# SINGLE (4 single-task models) -- repeat 0, fold 3
run_unit(0, 3, 'single')

2026-09-26 14:03:39.320 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:03:39.321 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:03:39.321 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:03:39.321 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 701 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] single__r0f3


...

2026-09-26 14:03:42.126 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:03:42.126 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:03:42.127 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] single__r0f3: keeping the existing timing record, not overwriting it
single__r0f3: 0.05 min  (last_epoch=None, min/epoch=nan)


True

In [30]:
# SINGLE (4 single-task models) -- repeat 0, fold 4
run_unit(0, 4, 'single')

2026-09-26 14:03:42.173 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:03:42.174 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:03:42.174 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:03:42.175 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 710 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] single__r0f4


...

2026-09-26 14:03:45.024 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:03:45.024 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:03:45.024 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] single__r0f4: keeping the existing timing record, not overwriting it
single__r0f4: 0.05 min  (last_epoch=None, min/epoch=nan)


True

In [31]:
# AID -- repeat 0, fold 0
run_unit(0, 0, 'aid')

2026-09-26 14:03:45.078 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:03:45.079 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 35 columns.


2026-09-26 14:03:45.080 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:03:45.080 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 684 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] aid__r0f0


...

2026-09-26 14:03:47.933 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:03:47.934 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:03:47.934 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] aid__r0f0: keeping the existing timing record, not overwriting it
aid__r0f0: 0.05 min  (last_epoch=12, min/epoch=0.004)


True

**Timing gate.** Fired mechanically now that repeat 0's PLAIN and AID first folds are complete,
projecting the five-repeat total from measured per-fold times. Thresholds, frozen in the brief
before any run: <=20 h run five repeats; 20–30 h run three; over 30 h run two; over 60 h even at
two repeats run one. The run is never abandoned over budget — only reduced.

In [32]:
# ---- TIMING GATE, applied mechanically now that repeat 0's PLAIN and AID first folds are
# complete. Projects the full five-repeat total from MEASURED per-fold times (cache hits,
# which record ~0 min, are excluded). DEADZONE is estimated at PLAIN's own cost -- same
# architecture, same row count, only the target values and the loss differ.
if DESIGN_PATH.exists() and "n_repeats_active" in DESIGN:
    print(f"[already decided on an earlier execution] N_REPEATS_ACTIVE = {N_REPEATS_ACTIVE}")
    print(json.dumps(DESIGN, indent=2))
elif ABORT["aborted"]:
    print("[run aborted before the gate could fire -- no reduction decision made]")
else:
    tm = pd.read_csv(TIMINGS_PATH)
    real = tm[tm["wall_min"] > 0.5]
    per_kind = real.groupby("arm_kind")["wall_min"].mean().to_dict()
    n_kind = real.groupby("arm_kind")["wall_min"].size().to_dict()
    print("measured mean wall time per (repeat, fold) unit, by arm:")
    for k in ["plain", "trees", "single", "aid"]:
        if k in per_kind:
            print(f"  {k:9s} {per_kind[k]:7.2f} min   (n={n_kind[k]} measured units)")
        else:
            print(f"  {k:9s}  no measured unit -- falling back to an estimate")

    # Notebook 29's own verified figures, used only where nothing was measured here.
    nb29_tm = pd.read_csv(NB29_OUT / "run_timings.csv")
    nb29_plain = float(nb29_tm[nb29_tm["arm"] == "baseline"]["wall_min"].mean())
    nb29_aid = float(nb29_tm[nb29_tm["arm"] == "aid"]["wall_min"].mean())
    print(f"\nnotebook 29's own measured means (verified from its run_timings.csv, not quoted):")
    print(f"  PLAIN {nb29_plain:.2f} min/fold   AID {nb29_aid:.2f} min/fold")
    print(f"  (the brief quotes ~5.5 and ~34; 5.5 is nb29's fold-0 PLAIN run, not its mean of "
          f"{nb29_plain:.2f} -- corrected here rather than repeated)")

    est = {"plain": per_kind.get("plain", nb29_plain),
           "trees": per_kind.get("trees", 3.0),
           "single": per_kind.get("single", per_kind.get("plain", nb29_plain)),
           "aid": per_kind.get("aid", nb29_aid)}
    est["deadzone"] = est["plain"]
    per_fold = sum(est.values())
    per_repeat_h = N_FOLDS * per_fold / 60.0
    total5_h = N_REPEATS_MAX * per_repeat_h
    total2_h = 2 * per_repeat_h
    print(f"\nper-fold total across all five arms: {per_fold:.2f} min")
    print(f"per-repeat: {per_repeat_h:.2f} h   projected five-repeat total: {total5_h:.2f} h")

    if total5_h <= 20.0:
        n_active, band = 5, "projected <= 20 h"
    elif total5_h <= 30.0:
        n_active, band = 3, "projected 20-30 h"
    elif total2_h > 60.0:
        n_active, band = 1, "projected over 60 h even at two repeats"
    else:
        n_active, band = 2, "projected over 30 h"

    N_REPEATS_ACTIVE = n_active
    DESIGN = {"n_repeats_active": n_active, "band": band,
              "measured_min_per_fold_by_arm": est,
              "per_fold_total_min": per_fold, "per_repeat_h": per_repeat_h,
              "projected_5repeat_h": total5_h, "projected_2repeat_h": total2_h,
              "nb29_plain_mean_min": nb29_plain, "nb29_aid_mean_min": nb29_aid,
              "wall_deadline_h": WALL_DEADLINE_H,
              "decided_at_utc": datetime.now(timezone.utc).isoformat()}
    DESIGN_PATH.write_text(json.dumps(DESIGN, indent=2))
    print(f"\nGATE: {band} -> running {n_active} repeat(s).")

    if n_active < N_REPEATS_MAX:
        tier = ("This is a SCREENING-TIER design, not a confirmation."
                if n_active <= 2 else
                "Three complete repeats (a 3x5 design), below the intended 5x5.")
        msg = (f"Projected five-repeat total {total5_h:.1f} h ({band}); reduced to {n_active} "
               f"repeat(s) per the brief's own mechanical rule. {tier}")
        print(f"*** DEVIATION *** {msg}")
        record_decision(
            decision=f"Reduced the design from 5 repeats to {n_active}",
            reason=msg,
            alternatives="Run all five repeats regardless (would exceed the brief's stated budget bands); abandon the run (explicitly ruled out by the brief).",
            authority="pre-specified mechanical timing gate in the task brief")

[already decided on an earlier execution] N_REPEATS_ACTIVE = 2
{
  "n_repeats_active": 2,
  "band": "projected over 30 h",
  "measured_min_per_fold_by_arm": {
    "plain": 8.54400487502416,
    "trees": 1.1505896298090617,
    "single": 11.162591482798259,
    "aid": 46.65563990275065,
    "deadzone": 8.54400487502416
  },
  "per_fold_total_min": 76.05683076540629,
  "per_repeat_h": 6.338069230450524,
  "projected_5repeat_h": 31.690346152252623,
  "projected_2repeat_h": 12.676138460901049,
  "nb29_plain_mean_min": 7.29788,
  "nb29_aid_mean_min": 34.02459999999999,
  "wall_deadline_h": 22.0,
  "decided_at_utc": "2026-09-26T04:17:58.045849+00:00"
}


In [33]:
# AID -- repeat 0, fold 1
run_unit(0, 1, 'aid')

2026-09-26 14:03:47.994 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:03:47.995 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 35 columns.


2026-09-26 14:03:47.996 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:03:47.996 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 723 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] aid__r0f1


...

2026-09-26 14:03:50.827 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:03:50.828 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:03:50.828 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] aid__r0f1: keeping the existing timing record, not overwriting it
aid__r0f1: 0.05 min  (last_epoch=7, min/epoch=0.007)


True

In [34]:
# AID -- repeat 0, fold 2
run_unit(0, 2, 'aid')

2026-09-26 14:03:50.882 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:03:50.883 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 35 columns.


2026-09-26 14:03:50.883 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:03:50.883 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 675 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] aid__r0f2


...

2026-09-26 14:03:53.717 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:03:53.718 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:03:53.718 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] aid__r0f2: keeping the existing timing record, not overwriting it
aid__r0f2: 0.05 min  (last_epoch=12, min/epoch=0.004)


True

In [35]:
# AID -- repeat 0, fold 3
run_unit(0, 3, 'aid')

2026-09-26 14:03:53.771 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:03:53.772 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 35 columns.


2026-09-26 14:03:53.772 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:03:53.773 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 701 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] aid__r0f3


...

2026-09-26 14:03:56.593 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:03:56.593 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:03:56.594 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] aid__r0f3: keeping the existing timing record, not overwriting it
aid__r0f3: 0.05 min  (last_epoch=7, min/epoch=0.007)


True

In [36]:
# AID -- repeat 0, fold 4
run_unit(0, 4, 'aid')

2026-09-26 14:03:56.647 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:03:56.648 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 35 columns.


2026-09-26 14:03:56.649 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:03:56.649 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 710 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] aid__r0f4


...

2026-09-26 14:03:59.458 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:03:59.459 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:03:59.459 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] aid__r0f4: keeping the existing timing record, not overwriting it
aid__r0f4: 0.05 min  (last_epoch=10, min/epoch=0.005)


True

**DEADZONE target for repeat 0** — built from PLAIN's completed repeat-0 out-of-fold predictions on this partition.

In [37]:
# ---- DEADZONE target for repeat 0. Built from PLAIN's own out-of-fold predictions ON THIS
# PARTITION (each compound is held out exactly once per repeat), never from
# outputs/11_caruana_prep/, which lives on the random partition.
R = 0
DEADZONE_READY[R] = False
if ABORT["aborted"] or R >= N_REPEATS_ACTIVE:
    print(f"[skipped] repeat {R} is outside the active design or the run has aborted")
else:
    plain_files = [PRED_DIR / f"plain__r{R}f{f}.csv" for f in range(N_FOLDS)]
    missing = [p.name for p in plain_files if not p.exists()]
    if missing:
        msg = (f"PLAIN's repeat {R} is incomplete ({len(missing)} of {N_FOLDS} fold "
               f"predictions missing: {missing}). DEADZONE for repeat {R} is SKIPPED -- "
               f"falling back to the random-partition OOF is explicitly ruled out.")
        print(f"*** {msg}")
        record_decision(decision=f"Skipped the DEADZONE arm for repeat {R}", reason=msg,
                        alternatives="Use outputs/11_caruana_prep/'s random-partition OOF as the target (explicitly forbidden by the brief).",
                        authority="the brief's own DEADZONE dependency rule")
    else:
        oof = pd.concat([pd.read_csv(p)[["inchikey"] + list(REGRESSION_ENDPOINTS)]
                         for p in plain_files], ignore_index=True)
        assert len(oof) == len(curated), f"PLAIN repeat {R} OOF has {len(oof)} rows, expected {len(curated)}"
        assert oof["inchikey"].is_unique, "a compound appears in more than one fold's held-out set"
        assert set(oof["inchikey"]) == curated_ik

        curated_dz = curated.copy()
        report = []
        for iso in ISOFORMS:
            ep = PIC50_COL[iso]
            m = curated[["inchikey", ep, f"{ep}_conf_low", f"{ep}_conf_high"]].merge(
                oof[["inchikey", ep]].rename(columns={ep: "_oof"}), on="inchikey", how="left")
            assert m["inchikey"].tolist() == curated["inchikey"].tolist(), \
                "left merge did not preserve curated row order -- positional assignment would misalign"
            assert m["_oof"].notna().all(), f"{iso}: a compound has no PLAIN OOF prediction"
            has_label = m[ep].notna()
            clipped = m["_oof"].clip(lower=m[f"{ep}_conf_low"], upper=m[f"{ep}_conf_high"])
            target = np.where(has_label, clipped, np.nan)
            curated_dz[ep] = target

            differs = (np.abs(target - m[ep].to_numpy()) > 1e-6) & has_label.to_numpy()
            frac_differs = float(differs.sum() / has_label.sum())
            inside = ((m["_oof"] >= m[f"{ep}_conf_low"]) & (m["_oof"] <= m[f"{ep}_conf_high"]))
            hit_rate = float(inside[has_label].mean())
            report.append({"repeat": R, "isoform": iso, "n_labelled": int(has_label.sum()),
                            "frac_target_differs_from_label": frac_differs,
                            "band_hit_rate": hit_rate})
        rep = pd.DataFrame(report)
        print(rep.to_string(index=False))
        _append_rows(OUT / "deadzone_target_checks.csv", report, ["repeat", "isoform"])

        # Check A (materiality, deliberately loose) and Check B (the real guard: an in-fold /
        # overfit donor would push the band hit rate toward 1.0). Notebook 31 reported 100% and
        # 18.0-41.7% respectively.
        bad = rep[(rep["frac_target_differs_from_label"] < 0.50) | (rep["band_hit_rate"] >= 0.90)]
        if len(bad):
            msg = (f"DEADZONE target check failed for repeat {R}:\n{bad.to_string(index=False)}\n"
                   f"The clipped target has silently become the identity, or the OOF join leaked "
                   f"in-fold predictions. Arm SKIPPED for this repeat.")
            print(f"*** {msg}")
            record_decision(decision=f"Skipped the DEADZONE arm for repeat {R}", reason=msg,
                            alternatives="Train the arm anyway on a target that is not the intended intervention.",
                            authority="the brief's own pre-training assertion on the DEADZONE target")
        else:
            dz_path = OUT / f"curated_deadzone_target_r{R}.csv"
            curated_dz.to_csv(dz_path, index=False)
            DEADZONE_TARGETS[R] = curated_dz
            DEADZONE_READY[R] = True
            print(f"\nDEADZONE target for repeat {R} PASSED both checks -> {dz_path.name}")

 repeat isoform  n_labelled  frac_target_differs_from_label  band_hit_rate
      0  CYP1A2        1412                             1.0       0.211048
      0  CYP2C9        1285                             1.0       0.403891
      0  CYP2D6        1493                             1.0       0.183523
      0  CYP3A4        2335                             1.0       0.363597

DEADZONE target for repeat 0 PASSED both checks -> curated_deadzone_target_r0.csv


In [38]:
# DEADZONE -- repeat 0, fold 0
run_unit(0, 0, 'deadzone')

2026-09-26 14:03:59.568 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:03:59.569 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:03:59.569 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:03:59.569 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 684 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] deadzone__r0f0


...

2026-09-26 14:04:02.376 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:04:02.376 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:04:02.377 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] deadzone__r0f0: keeping the existing timing record, not overwriting it
deadzone__r0f0: 0.05 min  (last_epoch=10, min/epoch=0.005)


True

In [39]:
# DEADZONE -- repeat 0, fold 1
run_unit(0, 1, 'deadzone')

2026-09-26 14:04:02.424 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:04:02.425 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:04:02.426 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:04:02.426 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 723 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] deadzone__r0f1


...

2026-09-26 14:04:05.249 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:04:05.249 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:04:05.250 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] deadzone__r0f1: keeping the existing timing record, not overwriting it
deadzone__r0f1: 0.05 min  (last_epoch=15, min/epoch=0.003)


True

In [40]:
# DEADZONE -- repeat 0, fold 2
run_unit(0, 2, 'deadzone')

2026-09-26 14:04:05.296 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:04:05.298 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:04:05.298 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:04:05.298 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 675 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] deadzone__r0f2


...

2026-09-26 14:04:08.132 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:04:08.132 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:04:08.133 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] deadzone__r0f2: keeping the existing timing record, not overwriting it
deadzone__r0f2: 0.05 min  (last_epoch=17, min/epoch=0.003)


True

In [41]:
# DEADZONE -- repeat 0, fold 3
run_unit(0, 3, 'deadzone')

2026-09-26 14:04:08.180 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:04:08.182 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:04:08.182 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:04:08.182 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 701 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] deadzone__r0f3


...

2026-09-26 14:04:11.019 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:04:11.020 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:04:11.020 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] deadzone__r0f3: keeping the existing timing record, not overwriting it
deadzone__r0f3: 0.05 min  (last_epoch=18, min/epoch=0.003)


True

In [42]:
# DEADZONE -- repeat 0, fold 4
run_unit(0, 4, 'deadzone')

2026-09-26 14:04:11.068 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:04:11.069 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:04:11.069 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:04:11.069 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 710 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] deadzone__r0f4


...

2026-09-26 14:04:13.904 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:04:13.905 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:04:13.905 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] deadzone__r0f4: keeping the existing timing record, not overwriting it
deadzone__r0f4: 0.05 min  (last_epoch=12, min/epoch=0.004)


True

In [43]:
# ---- Progress snapshot after repeat 0
if RUN_SCORES_PATH.exists():
    _rs = pd.read_csv(RUN_SCORES_PATH)
    _done = _rs[["repeat", "fold", "arm"]].drop_duplicates()
    print(f"completed (repeat, fold, arm) units so far: {len(_done)}")
    print(_done.groupby(["repeat", "arm"]).size().unstack(fill_value=0).to_string())
    _ma = _rs[(_rs["Endpoint"] == "MA") & (_rs["repeat"] == 0)]
    if len(_ma):
        print(f"\nrepeat 0 macro ST-RAE by arm (mean over completed folds):")
        print(_ma.groupby("arm")["ST-RAE"].agg(["mean", "std", "size"]).round(4).to_string())
_el = (time.time() - RUN_STARTED_AT) / 3600.0
print(f"\nelapsed since campaign start: {_el:.2f} h  (wall guard {WALL_DEADLINE_H} h)")
if DEADLINE_SKIPS:
    print(f"units skipped by the wall guard so far: {len(DEADLINE_SKIPS)}")

completed (repeat, fold, arm) units so far: 100
arm     aid  deadzone  ecfp4_narrow__lightgbm  ecfp4_narrow__rf  ecfp4_narrow__xgboost  mordred_pca__lightgbm  mordred_pca__rf  mordred_pca__xgboost  plain  single
repeat                                                                                                                                                             
0         5         5                       5                 5                      5                      5                5                     5      5       5
1         5         5                       5                 5                      5                      5                5                     5      5       5

repeat 0 macro ST-RAE by arm (mean over completed folds):
                          mean     std  size
arm                                         
aid                     0.7435  0.0361     5
deadzone                0.6713  0.0218     5
ecfp4_narrow__lightgbm  0.7995  0.0338     5
ecfp4_narrow

#### Repeat 1

In [44]:
# PLAIN -- repeat 1, fold 0
run_unit(1, 0, 'plain')

2026-09-26 14:04:13.959 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:04:13.960 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:04:13.960 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:04:13.961 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 704 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] plain__r1f0


...

2026-09-26 14:04:16.807 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:04:16.807 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:04:16.807 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] plain__r1f0: keeping the existing timing record, not overwriting it
plain__r1f0: 0.05 min  (last_epoch=9, min/epoch=0.005)


True

In [45]:
# PLAIN -- repeat 1, fold 1
run_unit(1, 1, 'plain')

2026-09-26 14:04:16.856 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:04:16.857 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:04:16.857 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:04:16.857 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 715 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] plain__r1f1


...

2026-09-26 14:04:19.702 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:04:19.702 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:04:19.703 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] plain__r1f1: keeping the existing timing record, not overwriting it
plain__r1f1: 0.05 min  (last_epoch=9, min/epoch=0.005)


True

In [46]:
# PLAIN -- repeat 1, fold 2
run_unit(1, 2, 'plain')

2026-09-26 14:04:19.750 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:04:19.751 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:04:19.751 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:04:19.751 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 672 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] plain__r1f2


...

2026-09-26 14:04:22.565 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:04:22.566 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:04:22.566 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] plain__r1f2: keeping the existing timing record, not overwriting it
plain__r1f2: 0.05 min  (last_epoch=8, min/epoch=0.006)


True

In [47]:
# PLAIN -- repeat 1, fold 3
run_unit(1, 3, 'plain')

2026-09-26 14:04:22.612 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:04:22.613 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:04:22.614 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:04:22.614 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 706 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] plain__r1f3


...

2026-09-26 14:04:25.428 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:04:25.428 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:04:25.428 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] plain__r1f3: keeping the existing timing record, not overwriting it
plain__r1f3: 0.05 min  (last_epoch=7, min/epoch=0.007)


True

In [48]:
# PLAIN -- repeat 1, fold 4
run_unit(1, 4, 'plain')

2026-09-26 14:04:25.474 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:04:25.475 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:04:25.475 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:04:25.475 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 696 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] plain__r1f4


...

2026-09-26 14:04:28.294 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:04:28.294 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:04:28.295 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] plain__r1f4: keeping the existing timing record, not overwriting it
plain__r1f4: 0.05 min  (last_epoch=8, min/epoch=0.006)


True

In [49]:
# TREES (6 configs) -- repeat 1, fold 0
run_unit(1, 0, 'trees')

2026-09-26 14:04:28.341 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:04:28.342 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 30 columns.


2026-09-26 14:04:28.342 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:04:28.342 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 704 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] ecfp4_narrow__rf__r1f0


...

2026-09-26 14:04:45.572 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:04:45.572 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:04:45.573 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] trees__r1f0: keeping the existing timing record, not overwriting it
trees__r1f0: 0.29 min  (last_epoch=None, min/epoch=nan)


True

In [50]:
# TREES (6 configs) -- repeat 1, fold 1
run_unit(1, 1, 'trees')

2026-09-26 14:04:45.620 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:04:45.622 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 30 columns.


2026-09-26 14:04:45.622 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:04:45.622 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 715 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] ecfp4_narrow__rf__r1f1


...

2026-09-26 14:05:02.787 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:05:02.788 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:05:02.788 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] trees__r1f1: keeping the existing timing record, not overwriting it
trees__r1f1: 0.29 min  (last_epoch=None, min/epoch=nan)


True

In [51]:
# TREES (6 configs) -- repeat 1, fold 2
run_unit(1, 2, 'trees')

2026-09-26 14:05:02.835 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:05:02.837 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 30 columns.


2026-09-26 14:05:02.837 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:05:02.837 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 672 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] ecfp4_narrow__rf__r1f2


...

2026-09-26 14:05:20.042 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:05:20.042 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:05:20.043 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] trees__r1f2: keeping the existing timing record, not overwriting it
trees__r1f2: 0.29 min  (last_epoch=None, min/epoch=nan)


True

In [52]:
# TREES (6 configs) -- repeat 1, fold 3
run_unit(1, 3, 'trees')

2026-09-26 14:05:20.091 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:05:20.092 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 30 columns.


2026-09-26 14:05:20.092 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:05:20.092 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 706 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] ecfp4_narrow__rf__r1f3


...

2026-09-26 14:05:37.185 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:05:37.186 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:05:37.186 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] trees__r1f3: keeping the existing timing record, not overwriting it
trees__r1f3: 0.29 min  (last_epoch=None, min/epoch=nan)


True

In [53]:
# TREES (6 configs) -- repeat 1, fold 4
run_unit(1, 4, 'trees')

2026-09-26 14:05:37.234 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:05:37.235 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 30 columns.


2026-09-26 14:05:37.235 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:05:37.235 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 696 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] ecfp4_narrow__rf__r1f4


...

2026-09-26 14:05:54.450 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:05:54.450 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:05:54.450 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] trees__r1f4: keeping the existing timing record, not overwriting it
trees__r1f4: 0.29 min  (last_epoch=None, min/epoch=nan)


True

In [54]:
# SINGLE (4 single-task models) -- repeat 1, fold 0
run_unit(1, 0, 'single')

2026-09-26 14:05:54.498 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:05:54.500 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:05:54.500 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:05:54.500 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 704 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] single__r1f0


...

2026-09-26 14:05:57.330 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:05:57.330 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:05:57.331 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] single__r1f0: keeping the existing timing record, not overwriting it
single__r1f0: 0.05 min  (last_epoch=None, min/epoch=nan)


True

In [55]:
# SINGLE (4 single-task models) -- repeat 1, fold 1
run_unit(1, 1, 'single')

2026-09-26 14:05:57.377 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:05:57.378 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:05:57.378 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:05:57.379 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 715 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] single__r1f1


...

2026-09-26 14:06:00.213 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:06:00.214 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:06:00.214 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] single__r1f1: keeping the existing timing record, not overwriting it
single__r1f1: 0.05 min  (last_epoch=None, min/epoch=nan)


True

In [56]:
# SINGLE (4 single-task models) -- repeat 1, fold 2
run_unit(1, 2, 'single')

2026-09-26 14:06:00.261 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:06:00.263 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:06:00.263 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:06:00.263 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 672 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] single__r1f2


...

2026-09-26 14:06:03.076 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:06:03.076 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:06:03.077 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] single__r1f2: keeping the existing timing record, not overwriting it
single__r1f2: 0.05 min  (last_epoch=None, min/epoch=nan)


True

In [57]:
# SINGLE (4 single-task models) -- repeat 1, fold 3
run_unit(1, 3, 'single')

2026-09-26 14:06:03.124 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:06:03.125 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:06:03.125 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:06:03.125 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 706 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] single__r1f3


...

2026-09-26 14:06:05.949 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:06:05.949 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:06:05.950 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] single__r1f3: keeping the existing timing record, not overwriting it
single__r1f3: 0.05 min  (last_epoch=None, min/epoch=nan)


True

In [58]:
# SINGLE (4 single-task models) -- repeat 1, fold 4
run_unit(1, 4, 'single')

2026-09-26 14:06:05.997 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:06:05.998 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:06:05.998 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:06:05.998 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 696 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] single__r1f4


...

2026-09-26 14:06:08.823 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:06:08.824 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:06:08.824 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] single__r1f4: keeping the existing timing record, not overwriting it
single__r1f4: 0.05 min  (last_epoch=None, min/epoch=nan)


True

In [59]:
# AID -- repeat 1, fold 0
run_unit(1, 0, 'aid')

2026-09-26 14:06:08.876 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:06:08.877 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 35 columns.


2026-09-26 14:06:08.877 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:06:08.878 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 704 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] aid__r1f0


...

2026-09-26 14:06:11.705 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:06:11.705 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:06:11.706 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] aid__r1f0: keeping the existing timing record, not overwriting it
aid__r1f0: 0.05 min  (last_epoch=9, min/epoch=0.005)


True

In [60]:
# AID -- repeat 1, fold 1
run_unit(1, 1, 'aid')

2026-09-26 14:06:11.759 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:06:11.760 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 35 columns.


2026-09-26 14:06:11.761 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:06:11.761 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 715 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] aid__r1f1


...

2026-09-26 14:06:14.620 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:06:14.620 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:06:14.621 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] aid__r1f1: keeping the existing timing record, not overwriting it
aid__r1f1: 0.05 min  (last_epoch=8, min/epoch=0.006)


True

In [61]:
# AID -- repeat 1, fold 2
run_unit(1, 2, 'aid')

2026-09-26 14:06:14.675 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:06:14.677 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 35 columns.


2026-09-26 14:06:14.677 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:06:14.677 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 672 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] aid__r1f2


...

2026-09-26 14:06:17.533 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:06:17.533 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:06:17.534 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] aid__r1f2: keeping the existing timing record, not overwriting it
aid__r1f2: 0.05 min  (last_epoch=7, min/epoch=0.007)


True

In [62]:
# AID -- repeat 1, fold 3
run_unit(1, 3, 'aid')

2026-09-26 14:06:17.587 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:06:17.588 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 35 columns.


2026-09-26 14:06:17.588 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:06:17.589 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 706 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] aid__r1f3


...

2026-09-26 14:06:20.411 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:06:20.412 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:06:20.412 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] aid__r1f3: keeping the existing timing record, not overwriting it
aid__r1f3: 0.05 min  (last_epoch=7, min/epoch=0.007)


True

In [63]:
# AID -- repeat 1, fold 4
run_unit(1, 4, 'aid')

2026-09-26 14:06:20.466 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:06:20.468 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 35 columns.


2026-09-26 14:06:20.468 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:06:20.468 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 696 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] aid__r1f4


...

2026-09-26 14:06:23.293 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:06:23.294 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:06:23.294 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] aid__r1f4: keeping the existing timing record, not overwriting it
aid__r1f4: 0.05 min  (last_epoch=8, min/epoch=0.006)


True

**DEADZONE target for repeat 1** — built from PLAIN's completed repeat-1 out-of-fold predictions on this partition.

In [64]:
# ---- DEADZONE target for repeat 1. Built from PLAIN's own out-of-fold predictions ON THIS
# PARTITION (each compound is held out exactly once per repeat), never from
# outputs/11_caruana_prep/, which lives on the random partition.
R = 1
DEADZONE_READY[R] = False
if ABORT["aborted"] or R >= N_REPEATS_ACTIVE:
    print(f"[skipped] repeat {R} is outside the active design or the run has aborted")
else:
    plain_files = [PRED_DIR / f"plain__r{R}f{f}.csv" for f in range(N_FOLDS)]
    missing = [p.name for p in plain_files if not p.exists()]
    if missing:
        msg = (f"PLAIN's repeat {R} is incomplete ({len(missing)} of {N_FOLDS} fold "
               f"predictions missing: {missing}). DEADZONE for repeat {R} is SKIPPED -- "
               f"falling back to the random-partition OOF is explicitly ruled out.")
        print(f"*** {msg}")
        record_decision(decision=f"Skipped the DEADZONE arm for repeat {R}", reason=msg,
                        alternatives="Use outputs/11_caruana_prep/'s random-partition OOF as the target (explicitly forbidden by the brief).",
                        authority="the brief's own DEADZONE dependency rule")
    else:
        oof = pd.concat([pd.read_csv(p)[["inchikey"] + list(REGRESSION_ENDPOINTS)]
                         for p in plain_files], ignore_index=True)
        assert len(oof) == len(curated), f"PLAIN repeat {R} OOF has {len(oof)} rows, expected {len(curated)}"
        assert oof["inchikey"].is_unique, "a compound appears in more than one fold's held-out set"
        assert set(oof["inchikey"]) == curated_ik

        curated_dz = curated.copy()
        report = []
        for iso in ISOFORMS:
            ep = PIC50_COL[iso]
            m = curated[["inchikey", ep, f"{ep}_conf_low", f"{ep}_conf_high"]].merge(
                oof[["inchikey", ep]].rename(columns={ep: "_oof"}), on="inchikey", how="left")
            assert m["inchikey"].tolist() == curated["inchikey"].tolist(), \
                "left merge did not preserve curated row order -- positional assignment would misalign"
            assert m["_oof"].notna().all(), f"{iso}: a compound has no PLAIN OOF prediction"
            has_label = m[ep].notna()
            clipped = m["_oof"].clip(lower=m[f"{ep}_conf_low"], upper=m[f"{ep}_conf_high"])
            target = np.where(has_label, clipped, np.nan)
            curated_dz[ep] = target

            differs = (np.abs(target - m[ep].to_numpy()) > 1e-6) & has_label.to_numpy()
            frac_differs = float(differs.sum() / has_label.sum())
            inside = ((m["_oof"] >= m[f"{ep}_conf_low"]) & (m["_oof"] <= m[f"{ep}_conf_high"]))
            hit_rate = float(inside[has_label].mean())
            report.append({"repeat": R, "isoform": iso, "n_labelled": int(has_label.sum()),
                            "frac_target_differs_from_label": frac_differs,
                            "band_hit_rate": hit_rate})
        rep = pd.DataFrame(report)
        print(rep.to_string(index=False))
        _append_rows(OUT / "deadzone_target_checks.csv", report, ["repeat", "isoform"])

        # Check A (materiality, deliberately loose) and Check B (the real guard: an in-fold /
        # overfit donor would push the band hit rate toward 1.0). Notebook 31 reported 100% and
        # 18.0-41.7% respectively.
        bad = rep[(rep["frac_target_differs_from_label"] < 0.50) | (rep["band_hit_rate"] >= 0.90)]
        if len(bad):
            msg = (f"DEADZONE target check failed for repeat {R}:\n{bad.to_string(index=False)}\n"
                   f"The clipped target has silently become the identity, or the OOF join leaked "
                   f"in-fold predictions. Arm SKIPPED for this repeat.")
            print(f"*** {msg}")
            record_decision(decision=f"Skipped the DEADZONE arm for repeat {R}", reason=msg,
                            alternatives="Train the arm anyway on a target that is not the intended intervention.",
                            authority="the brief's own pre-training assertion on the DEADZONE target")
        else:
            dz_path = OUT / f"curated_deadzone_target_r{R}.csv"
            curated_dz.to_csv(dz_path, index=False)
            DEADZONE_TARGETS[R] = curated_dz
            DEADZONE_READY[R] = True
            print(f"\nDEADZONE target for repeat {R} PASSED both checks -> {dz_path.name}")

 repeat isoform  n_labelled  frac_target_differs_from_label  band_hit_rate
      1  CYP1A2        1412                             1.0       0.203258
      1  CYP2C9        1285                             1.0       0.425681
      1  CYP2D6        1493                             1.0       0.178165
      1  CYP3A4        2335                             1.0       0.373019

DEADZONE target for repeat 1 PASSED both checks -> curated_deadzone_target_r1.csv


In [65]:
# DEADZONE -- repeat 1, fold 0
run_unit(1, 0, 'deadzone')

2026-09-26 14:06:23.399 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:06:23.401 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:06:23.401 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:06:23.401 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 704 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] deadzone__r1f0


...

2026-09-26 14:06:26.237 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:06:26.238 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:06:26.238 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] deadzone__r1f0: keeping the existing timing record, not overwriting it
deadzone__r1f0: 0.05 min  (last_epoch=10, min/epoch=0.005)


True

In [66]:
# DEADZONE -- repeat 1, fold 1
run_unit(1, 1, 'deadzone')

2026-09-26 14:06:26.285 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:06:26.287 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:06:26.287 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:06:26.287 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 715 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] deadzone__r1f1


...

2026-09-26 14:06:29.127 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:06:29.128 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:06:29.128 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] deadzone__r1f1: keeping the existing timing record, not overwriting it
deadzone__r1f1: 0.05 min  (last_epoch=13, min/epoch=0.004)


True

In [67]:
# DEADZONE -- repeat 1, fold 2
run_unit(1, 2, 'deadzone')

2026-09-26 14:06:29.176 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:06:29.177 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:06:29.177 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:06:29.178 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 672 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] deadzone__r1f2


...

2026-09-26 14:06:32.019 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:06:32.019 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:06:32.019 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] deadzone__r1f2: keeping the existing timing record, not overwriting it
deadzone__r1f2: 0.05 min  (last_epoch=12, min/epoch=0.004)


True

In [68]:
# DEADZONE -- repeat 1, fold 3
run_unit(1, 3, 'deadzone')

2026-09-26 14:06:32.066 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:06:32.067 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:06:32.068 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:06:32.068 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 706 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] deadzone__r1f3


...

2026-09-26 14:06:34.908 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:06:34.908 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:06:34.909 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] deadzone__r1f3: keeping the existing timing record, not overwriting it
deadzone__r1f3: 0.05 min  (last_epoch=16, min/epoch=0.003)


True

In [69]:
# DEADZONE -- repeat 1, fold 4
run_unit(1, 4, 'deadzone')

2026-09-26 14:06:34.957 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:91 - Scoring activity predictions against ground truth


2026-09-26 14:06:34.959 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:95 - Completed merging predictions with ground truth. Merged dataset contains 981 rows and 31 columns.


2026-09-26 14:06:34.959 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:104 - Scoring endpoint: CYP1A2_pIC50_direct_inhibition


2026-09-26 14:06:34.959 | DEBUG    | src.vendor.openadmet_eval.evaluate_predictions:score_activity_predictions:125 - Excluding 696 compound(s) with no ground truth for endpoint CYP1A2_pIC50_direct_inhibition


  [cache hit] deadzone__r1f4


...

2026-09-26 14:06:37.792 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric R2 across bootstrap iterations


2026-09-26 14:06:37.793 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Spearman_R across bootstrap iterations


2026-09-26 14:06:37.793 | INFO     | src.vendor.openadmet_eval.evaluate_predictions:compute_macro_bootstrap_results:271 - Computing macro-averaged metric Kendall_Tau across bootstrap iterations


[cache hit] deadzone__r1f4: keeping the existing timing record, not overwriting it
deadzone__r1f4: 0.05 min  (last_epoch=12, min/epoch=0.004)


True

In [70]:
# ---- Progress snapshot after repeat 1
if RUN_SCORES_PATH.exists():
    _rs = pd.read_csv(RUN_SCORES_PATH)
    _done = _rs[["repeat", "fold", "arm"]].drop_duplicates()
    print(f"completed (repeat, fold, arm) units so far: {len(_done)}")
    print(_done.groupby(["repeat", "arm"]).size().unstack(fill_value=0).to_string())
    _ma = _rs[(_rs["Endpoint"] == "MA") & (_rs["repeat"] == 1)]
    if len(_ma):
        print(f"\nrepeat 1 macro ST-RAE by arm (mean over completed folds):")
        print(_ma.groupby("arm")["ST-RAE"].agg(["mean", "std", "size"]).round(4).to_string())
_el = (time.time() - RUN_STARTED_AT) / 3600.0
print(f"\nelapsed since campaign start: {_el:.2f} h  (wall guard {WALL_DEADLINE_H} h)")
if DEADLINE_SKIPS:
    print(f"units skipped by the wall guard so far: {len(DEADLINE_SKIPS)}")

completed (repeat, fold, arm) units so far: 100
arm     aid  deadzone  ecfp4_narrow__lightgbm  ecfp4_narrow__rf  ecfp4_narrow__xgboost  mordred_pca__lightgbm  mordred_pca__rf  mordred_pca__xgboost  plain  single
repeat                                                                                                                                                             
0         5         5                       5                 5                      5                      5                5                     5      5       5
1         5         5                       5                 5                      5                      5                5                     5      5       5

repeat 1 macro ST-RAE by arm (mean over completed folds):
                          mean     std  size
arm                                         
aid                     0.7430  0.0421     5
deadzone                0.6752  0.0187     5
ecfp4_narrow__lightgbm  0.8026  0.0166     5
ecfp4_narrow

#### Repeat 2

In [71]:
# PLAIN -- repeat 2, fold 0
run_unit(2, 0, 'plain')

[outside the reduced design: 2 repeats active] plain__r2f0


In [72]:
# PLAIN -- repeat 2, fold 1
run_unit(2, 1, 'plain')

[outside the reduced design: 2 repeats active] plain__r2f1


In [73]:
# PLAIN -- repeat 2, fold 2
run_unit(2, 2, 'plain')

[outside the reduced design: 2 repeats active] plain__r2f2


In [74]:
# PLAIN -- repeat 2, fold 3
run_unit(2, 3, 'plain')

[outside the reduced design: 2 repeats active] plain__r2f3


In [75]:
# PLAIN -- repeat 2, fold 4
run_unit(2, 4, 'plain')

[outside the reduced design: 2 repeats active] plain__r2f4


In [76]:
# TREES (6 configs) -- repeat 2, fold 0
run_unit(2, 0, 'trees')

[outside the reduced design: 2 repeats active] trees__r2f0


In [77]:
# TREES (6 configs) -- repeat 2, fold 1
run_unit(2, 1, 'trees')

[outside the reduced design: 2 repeats active] trees__r2f1


In [78]:
# TREES (6 configs) -- repeat 2, fold 2
run_unit(2, 2, 'trees')

[outside the reduced design: 2 repeats active] trees__r2f2


In [79]:
# TREES (6 configs) -- repeat 2, fold 3
run_unit(2, 3, 'trees')

[outside the reduced design: 2 repeats active] trees__r2f3


In [80]:
# TREES (6 configs) -- repeat 2, fold 4
run_unit(2, 4, 'trees')

[outside the reduced design: 2 repeats active] trees__r2f4


In [81]:
# SINGLE (4 single-task models) -- repeat 2, fold 0
run_unit(2, 0, 'single')

[outside the reduced design: 2 repeats active] single__r2f0


In [82]:
# SINGLE (4 single-task models) -- repeat 2, fold 1
run_unit(2, 1, 'single')

[outside the reduced design: 2 repeats active] single__r2f1


In [83]:
# SINGLE (4 single-task models) -- repeat 2, fold 2
run_unit(2, 2, 'single')

[outside the reduced design: 2 repeats active] single__r2f2


In [84]:
# SINGLE (4 single-task models) -- repeat 2, fold 3
run_unit(2, 3, 'single')

[outside the reduced design: 2 repeats active] single__r2f3


In [85]:
# SINGLE (4 single-task models) -- repeat 2, fold 4
run_unit(2, 4, 'single')

[outside the reduced design: 2 repeats active] single__r2f4


In [86]:
# AID -- repeat 2, fold 0
run_unit(2, 0, 'aid')

[outside the reduced design: 2 repeats active] aid__r2f0


In [87]:
# AID -- repeat 2, fold 1
run_unit(2, 1, 'aid')

[outside the reduced design: 2 repeats active] aid__r2f1


In [88]:
# AID -- repeat 2, fold 2
run_unit(2, 2, 'aid')

[outside the reduced design: 2 repeats active] aid__r2f2


In [89]:
# AID -- repeat 2, fold 3
run_unit(2, 3, 'aid')

[outside the reduced design: 2 repeats active] aid__r2f3


In [90]:
# AID -- repeat 2, fold 4
run_unit(2, 4, 'aid')

[outside the reduced design: 2 repeats active] aid__r2f4


**DEADZONE target for repeat 2** — built from PLAIN's completed repeat-2 out-of-fold predictions on this partition.

In [91]:
# ---- DEADZONE target for repeat 2. Built from PLAIN's own out-of-fold predictions ON THIS
# PARTITION (each compound is held out exactly once per repeat), never from
# outputs/11_caruana_prep/, which lives on the random partition.
R = 2
DEADZONE_READY[R] = False
if ABORT["aborted"] or R >= N_REPEATS_ACTIVE:
    print(f"[skipped] repeat {R} is outside the active design or the run has aborted")
else:
    plain_files = [PRED_DIR / f"plain__r{R}f{f}.csv" for f in range(N_FOLDS)]
    missing = [p.name for p in plain_files if not p.exists()]
    if missing:
        msg = (f"PLAIN's repeat {R} is incomplete ({len(missing)} of {N_FOLDS} fold "
               f"predictions missing: {missing}). DEADZONE for repeat {R} is SKIPPED -- "
               f"falling back to the random-partition OOF is explicitly ruled out.")
        print(f"*** {msg}")
        record_decision(decision=f"Skipped the DEADZONE arm for repeat {R}", reason=msg,
                        alternatives="Use outputs/11_caruana_prep/'s random-partition OOF as the target (explicitly forbidden by the brief).",
                        authority="the brief's own DEADZONE dependency rule")
    else:
        oof = pd.concat([pd.read_csv(p)[["inchikey"] + list(REGRESSION_ENDPOINTS)]
                         for p in plain_files], ignore_index=True)
        assert len(oof) == len(curated), f"PLAIN repeat {R} OOF has {len(oof)} rows, expected {len(curated)}"
        assert oof["inchikey"].is_unique, "a compound appears in more than one fold's held-out set"
        assert set(oof["inchikey"]) == curated_ik

        curated_dz = curated.copy()
        report = []
        for iso in ISOFORMS:
            ep = PIC50_COL[iso]
            m = curated[["inchikey", ep, f"{ep}_conf_low", f"{ep}_conf_high"]].merge(
                oof[["inchikey", ep]].rename(columns={ep: "_oof"}), on="inchikey", how="left")
            assert m["inchikey"].tolist() == curated["inchikey"].tolist(), \
                "left merge did not preserve curated row order -- positional assignment would misalign"
            assert m["_oof"].notna().all(), f"{iso}: a compound has no PLAIN OOF prediction"
            has_label = m[ep].notna()
            clipped = m["_oof"].clip(lower=m[f"{ep}_conf_low"], upper=m[f"{ep}_conf_high"])
            target = np.where(has_label, clipped, np.nan)
            curated_dz[ep] = target

            differs = (np.abs(target - m[ep].to_numpy()) > 1e-6) & has_label.to_numpy()
            frac_differs = float(differs.sum() / has_label.sum())
            inside = ((m["_oof"] >= m[f"{ep}_conf_low"]) & (m["_oof"] <= m[f"{ep}_conf_high"]))
            hit_rate = float(inside[has_label].mean())
            report.append({"repeat": R, "isoform": iso, "n_labelled": int(has_label.sum()),
                            "frac_target_differs_from_label": frac_differs,
                            "band_hit_rate": hit_rate})
        rep = pd.DataFrame(report)
        print(rep.to_string(index=False))
        _append_rows(OUT / "deadzone_target_checks.csv", report, ["repeat", "isoform"])

        # Check A (materiality, deliberately loose) and Check B (the real guard: an in-fold /
        # overfit donor would push the band hit rate toward 1.0). Notebook 31 reported 100% and
        # 18.0-41.7% respectively.
        bad = rep[(rep["frac_target_differs_from_label"] < 0.50) | (rep["band_hit_rate"] >= 0.90)]
        if len(bad):
            msg = (f"DEADZONE target check failed for repeat {R}:\n{bad.to_string(index=False)}\n"
                   f"The clipped target has silently become the identity, or the OOF join leaked "
                   f"in-fold predictions. Arm SKIPPED for this repeat.")
            print(f"*** {msg}")
            record_decision(decision=f"Skipped the DEADZONE arm for repeat {R}", reason=msg,
                            alternatives="Train the arm anyway on a target that is not the intended intervention.",
                            authority="the brief's own pre-training assertion on the DEADZONE target")
        else:
            dz_path = OUT / f"curated_deadzone_target_r{R}.csv"
            curated_dz.to_csv(dz_path, index=False)
            DEADZONE_TARGETS[R] = curated_dz
            DEADZONE_READY[R] = True
            print(f"\nDEADZONE target for repeat {R} PASSED both checks -> {dz_path.name}")

[skipped] repeat 2 is outside the active design or the run has aborted


In [92]:
# DEADZONE -- repeat 2, fold 0
run_unit(2, 0, 'deadzone')

[outside the reduced design: 2 repeats active] deadzone__r2f0


In [93]:
# DEADZONE -- repeat 2, fold 1
run_unit(2, 1, 'deadzone')

[outside the reduced design: 2 repeats active] deadzone__r2f1


In [94]:
# DEADZONE -- repeat 2, fold 2
run_unit(2, 2, 'deadzone')

[outside the reduced design: 2 repeats active] deadzone__r2f2


In [95]:
# DEADZONE -- repeat 2, fold 3
run_unit(2, 3, 'deadzone')

[outside the reduced design: 2 repeats active] deadzone__r2f3


In [96]:
# DEADZONE -- repeat 2, fold 4
run_unit(2, 4, 'deadzone')

[outside the reduced design: 2 repeats active] deadzone__r2f4


In [97]:
# ---- Progress snapshot after repeat 2
if RUN_SCORES_PATH.exists():
    _rs = pd.read_csv(RUN_SCORES_PATH)
    _done = _rs[["repeat", "fold", "arm"]].drop_duplicates()
    print(f"completed (repeat, fold, arm) units so far: {len(_done)}")
    print(_done.groupby(["repeat", "arm"]).size().unstack(fill_value=0).to_string())
    _ma = _rs[(_rs["Endpoint"] == "MA") & (_rs["repeat"] == 2)]
    if len(_ma):
        print(f"\nrepeat 2 macro ST-RAE by arm (mean over completed folds):")
        print(_ma.groupby("arm")["ST-RAE"].agg(["mean", "std", "size"]).round(4).to_string())
_el = (time.time() - RUN_STARTED_AT) / 3600.0
print(f"\nelapsed since campaign start: {_el:.2f} h  (wall guard {WALL_DEADLINE_H} h)")
if DEADLINE_SKIPS:
    print(f"units skipped by the wall guard so far: {len(DEADLINE_SKIPS)}")

completed (repeat, fold, arm) units so far: 100
arm     aid  deadzone  ecfp4_narrow__lightgbm  ecfp4_narrow__rf  ecfp4_narrow__xgboost  mordred_pca__lightgbm  mordred_pca__rf  mordred_pca__xgboost  plain  single
repeat                                                                                                                                                             
0         5         5                       5                 5                      5                      5                5                     5      5       5
1         5         5                       5                 5                      5                      5                5                     5      5       5

elapsed since campaign start: 11.42 h  (wall guard 22.0 h)


#### Repeat 3

In [98]:
# PLAIN -- repeat 3, fold 0
run_unit(3, 0, 'plain')

[outside the reduced design: 2 repeats active] plain__r3f0


In [99]:
# PLAIN -- repeat 3, fold 1
run_unit(3, 1, 'plain')

[outside the reduced design: 2 repeats active] plain__r3f1


In [100]:
# PLAIN -- repeat 3, fold 2
run_unit(3, 2, 'plain')

[outside the reduced design: 2 repeats active] plain__r3f2


In [101]:
# PLAIN -- repeat 3, fold 3
run_unit(3, 3, 'plain')

[outside the reduced design: 2 repeats active] plain__r3f3


In [102]:
# PLAIN -- repeat 3, fold 4
run_unit(3, 4, 'plain')

[outside the reduced design: 2 repeats active] plain__r3f4


In [103]:
# TREES (6 configs) -- repeat 3, fold 0
run_unit(3, 0, 'trees')

[outside the reduced design: 2 repeats active] trees__r3f0


In [104]:
# TREES (6 configs) -- repeat 3, fold 1
run_unit(3, 1, 'trees')

[outside the reduced design: 2 repeats active] trees__r3f1


In [105]:
# TREES (6 configs) -- repeat 3, fold 2
run_unit(3, 2, 'trees')

[outside the reduced design: 2 repeats active] trees__r3f2


In [106]:
# TREES (6 configs) -- repeat 3, fold 3
run_unit(3, 3, 'trees')

[outside the reduced design: 2 repeats active] trees__r3f3


In [107]:
# TREES (6 configs) -- repeat 3, fold 4
run_unit(3, 4, 'trees')

[outside the reduced design: 2 repeats active] trees__r3f4


In [108]:
# SINGLE (4 single-task models) -- repeat 3, fold 0
run_unit(3, 0, 'single')

[outside the reduced design: 2 repeats active] single__r3f0


In [109]:
# SINGLE (4 single-task models) -- repeat 3, fold 1
run_unit(3, 1, 'single')

[outside the reduced design: 2 repeats active] single__r3f1


In [110]:
# SINGLE (4 single-task models) -- repeat 3, fold 2
run_unit(3, 2, 'single')

[outside the reduced design: 2 repeats active] single__r3f2


In [111]:
# SINGLE (4 single-task models) -- repeat 3, fold 3
run_unit(3, 3, 'single')

[outside the reduced design: 2 repeats active] single__r3f3


In [112]:
# SINGLE (4 single-task models) -- repeat 3, fold 4
run_unit(3, 4, 'single')

[outside the reduced design: 2 repeats active] single__r3f4


In [113]:
# AID -- repeat 3, fold 0
run_unit(3, 0, 'aid')

[outside the reduced design: 2 repeats active] aid__r3f0


In [114]:
# AID -- repeat 3, fold 1
run_unit(3, 1, 'aid')

[outside the reduced design: 2 repeats active] aid__r3f1


In [115]:
# AID -- repeat 3, fold 2
run_unit(3, 2, 'aid')

[outside the reduced design: 2 repeats active] aid__r3f2


In [116]:
# AID -- repeat 3, fold 3
run_unit(3, 3, 'aid')

[outside the reduced design: 2 repeats active] aid__r3f3


In [117]:
# AID -- repeat 3, fold 4
run_unit(3, 4, 'aid')

[outside the reduced design: 2 repeats active] aid__r3f4


**DEADZONE target for repeat 3** — built from PLAIN's completed repeat-3 out-of-fold predictions on this partition.

In [118]:
# ---- DEADZONE target for repeat 3. Built from PLAIN's own out-of-fold predictions ON THIS
# PARTITION (each compound is held out exactly once per repeat), never from
# outputs/11_caruana_prep/, which lives on the random partition.
R = 3
DEADZONE_READY[R] = False
if ABORT["aborted"] or R >= N_REPEATS_ACTIVE:
    print(f"[skipped] repeat {R} is outside the active design or the run has aborted")
else:
    plain_files = [PRED_DIR / f"plain__r{R}f{f}.csv" for f in range(N_FOLDS)]
    missing = [p.name for p in plain_files if not p.exists()]
    if missing:
        msg = (f"PLAIN's repeat {R} is incomplete ({len(missing)} of {N_FOLDS} fold "
               f"predictions missing: {missing}). DEADZONE for repeat {R} is SKIPPED -- "
               f"falling back to the random-partition OOF is explicitly ruled out.")
        print(f"*** {msg}")
        record_decision(decision=f"Skipped the DEADZONE arm for repeat {R}", reason=msg,
                        alternatives="Use outputs/11_caruana_prep/'s random-partition OOF as the target (explicitly forbidden by the brief).",
                        authority="the brief's own DEADZONE dependency rule")
    else:
        oof = pd.concat([pd.read_csv(p)[["inchikey"] + list(REGRESSION_ENDPOINTS)]
                         for p in plain_files], ignore_index=True)
        assert len(oof) == len(curated), f"PLAIN repeat {R} OOF has {len(oof)} rows, expected {len(curated)}"
        assert oof["inchikey"].is_unique, "a compound appears in more than one fold's held-out set"
        assert set(oof["inchikey"]) == curated_ik

        curated_dz = curated.copy()
        report = []
        for iso in ISOFORMS:
            ep = PIC50_COL[iso]
            m = curated[["inchikey", ep, f"{ep}_conf_low", f"{ep}_conf_high"]].merge(
                oof[["inchikey", ep]].rename(columns={ep: "_oof"}), on="inchikey", how="left")
            assert m["inchikey"].tolist() == curated["inchikey"].tolist(), \
                "left merge did not preserve curated row order -- positional assignment would misalign"
            assert m["_oof"].notna().all(), f"{iso}: a compound has no PLAIN OOF prediction"
            has_label = m[ep].notna()
            clipped = m["_oof"].clip(lower=m[f"{ep}_conf_low"], upper=m[f"{ep}_conf_high"])
            target = np.where(has_label, clipped, np.nan)
            curated_dz[ep] = target

            differs = (np.abs(target - m[ep].to_numpy()) > 1e-6) & has_label.to_numpy()
            frac_differs = float(differs.sum() / has_label.sum())
            inside = ((m["_oof"] >= m[f"{ep}_conf_low"]) & (m["_oof"] <= m[f"{ep}_conf_high"]))
            hit_rate = float(inside[has_label].mean())
            report.append({"repeat": R, "isoform": iso, "n_labelled": int(has_label.sum()),
                            "frac_target_differs_from_label": frac_differs,
                            "band_hit_rate": hit_rate})
        rep = pd.DataFrame(report)
        print(rep.to_string(index=False))
        _append_rows(OUT / "deadzone_target_checks.csv", report, ["repeat", "isoform"])

        # Check A (materiality, deliberately loose) and Check B (the real guard: an in-fold /
        # overfit donor would push the band hit rate toward 1.0). Notebook 31 reported 100% and
        # 18.0-41.7% respectively.
        bad = rep[(rep["frac_target_differs_from_label"] < 0.50) | (rep["band_hit_rate"] >= 0.90)]
        if len(bad):
            msg = (f"DEADZONE target check failed for repeat {R}:\n{bad.to_string(index=False)}\n"
                   f"The clipped target has silently become the identity, or the OOF join leaked "
                   f"in-fold predictions. Arm SKIPPED for this repeat.")
            print(f"*** {msg}")
            record_decision(decision=f"Skipped the DEADZONE arm for repeat {R}", reason=msg,
                            alternatives="Train the arm anyway on a target that is not the intended intervention.",
                            authority="the brief's own pre-training assertion on the DEADZONE target")
        else:
            dz_path = OUT / f"curated_deadzone_target_r{R}.csv"
            curated_dz.to_csv(dz_path, index=False)
            DEADZONE_TARGETS[R] = curated_dz
            DEADZONE_READY[R] = True
            print(f"\nDEADZONE target for repeat {R} PASSED both checks -> {dz_path.name}")

[skipped] repeat 3 is outside the active design or the run has aborted


In [119]:
# DEADZONE -- repeat 3, fold 0
run_unit(3, 0, 'deadzone')

[outside the reduced design: 2 repeats active] deadzone__r3f0


In [120]:
# DEADZONE -- repeat 3, fold 1
run_unit(3, 1, 'deadzone')

[outside the reduced design: 2 repeats active] deadzone__r3f1


In [121]:
# DEADZONE -- repeat 3, fold 2
run_unit(3, 2, 'deadzone')

[outside the reduced design: 2 repeats active] deadzone__r3f2


In [122]:
# DEADZONE -- repeat 3, fold 3
run_unit(3, 3, 'deadzone')

[outside the reduced design: 2 repeats active] deadzone__r3f3


In [123]:
# DEADZONE -- repeat 3, fold 4
run_unit(3, 4, 'deadzone')

[outside the reduced design: 2 repeats active] deadzone__r3f4


In [124]:
# ---- Progress snapshot after repeat 3
if RUN_SCORES_PATH.exists():
    _rs = pd.read_csv(RUN_SCORES_PATH)
    _done = _rs[["repeat", "fold", "arm"]].drop_duplicates()
    print(f"completed (repeat, fold, arm) units so far: {len(_done)}")
    print(_done.groupby(["repeat", "arm"]).size().unstack(fill_value=0).to_string())
    _ma = _rs[(_rs["Endpoint"] == "MA") & (_rs["repeat"] == 3)]
    if len(_ma):
        print(f"\nrepeat 3 macro ST-RAE by arm (mean over completed folds):")
        print(_ma.groupby("arm")["ST-RAE"].agg(["mean", "std", "size"]).round(4).to_string())
_el = (time.time() - RUN_STARTED_AT) / 3600.0
print(f"\nelapsed since campaign start: {_el:.2f} h  (wall guard {WALL_DEADLINE_H} h)")
if DEADLINE_SKIPS:
    print(f"units skipped by the wall guard so far: {len(DEADLINE_SKIPS)}")

completed (repeat, fold, arm) units so far: 100
arm     aid  deadzone  ecfp4_narrow__lightgbm  ecfp4_narrow__rf  ecfp4_narrow__xgboost  mordred_pca__lightgbm  mordred_pca__rf  mordred_pca__xgboost  plain  single
repeat                                                                                                                                                             
0         5         5                       5                 5                      5                      5                5                     5      5       5
1         5         5                       5                 5                      5                      5                5                     5      5       5

elapsed since campaign start: 11.42 h  (wall guard 22.0 h)


#### Repeat 4

In [125]:
# PLAIN -- repeat 4, fold 0
run_unit(4, 0, 'plain')

[outside the reduced design: 2 repeats active] plain__r4f0


In [126]:
# PLAIN -- repeat 4, fold 1
run_unit(4, 1, 'plain')

[outside the reduced design: 2 repeats active] plain__r4f1


In [127]:
# PLAIN -- repeat 4, fold 2
run_unit(4, 2, 'plain')

[outside the reduced design: 2 repeats active] plain__r4f2


In [128]:
# PLAIN -- repeat 4, fold 3
run_unit(4, 3, 'plain')

[outside the reduced design: 2 repeats active] plain__r4f3


In [129]:
# PLAIN -- repeat 4, fold 4
run_unit(4, 4, 'plain')

[outside the reduced design: 2 repeats active] plain__r4f4


In [130]:
# TREES (6 configs) -- repeat 4, fold 0
run_unit(4, 0, 'trees')

[outside the reduced design: 2 repeats active] trees__r4f0


In [131]:
# TREES (6 configs) -- repeat 4, fold 1
run_unit(4, 1, 'trees')

[outside the reduced design: 2 repeats active] trees__r4f1


In [132]:
# TREES (6 configs) -- repeat 4, fold 2
run_unit(4, 2, 'trees')

[outside the reduced design: 2 repeats active] trees__r4f2


In [133]:
# TREES (6 configs) -- repeat 4, fold 3
run_unit(4, 3, 'trees')

[outside the reduced design: 2 repeats active] trees__r4f3


In [134]:
# TREES (6 configs) -- repeat 4, fold 4
run_unit(4, 4, 'trees')

[outside the reduced design: 2 repeats active] trees__r4f4


In [135]:
# SINGLE (4 single-task models) -- repeat 4, fold 0
run_unit(4, 0, 'single')

[outside the reduced design: 2 repeats active] single__r4f0


In [136]:
# SINGLE (4 single-task models) -- repeat 4, fold 1
run_unit(4, 1, 'single')

[outside the reduced design: 2 repeats active] single__r4f1


In [137]:
# SINGLE (4 single-task models) -- repeat 4, fold 2
run_unit(4, 2, 'single')

[outside the reduced design: 2 repeats active] single__r4f2


In [138]:
# SINGLE (4 single-task models) -- repeat 4, fold 3
run_unit(4, 3, 'single')

[outside the reduced design: 2 repeats active] single__r4f3


In [139]:
# SINGLE (4 single-task models) -- repeat 4, fold 4
run_unit(4, 4, 'single')

[outside the reduced design: 2 repeats active] single__r4f4


In [140]:
# AID -- repeat 4, fold 0
run_unit(4, 0, 'aid')

[outside the reduced design: 2 repeats active] aid__r4f0


In [141]:
# AID -- repeat 4, fold 1
run_unit(4, 1, 'aid')

[outside the reduced design: 2 repeats active] aid__r4f1


In [142]:
# AID -- repeat 4, fold 2
run_unit(4, 2, 'aid')

[outside the reduced design: 2 repeats active] aid__r4f2


In [143]:
# AID -- repeat 4, fold 3
run_unit(4, 3, 'aid')

[outside the reduced design: 2 repeats active] aid__r4f3


In [144]:
# AID -- repeat 4, fold 4
run_unit(4, 4, 'aid')

[outside the reduced design: 2 repeats active] aid__r4f4


**DEADZONE target for repeat 4** — built from PLAIN's completed repeat-4 out-of-fold predictions on this partition.

In [145]:
# ---- DEADZONE target for repeat 4. Built from PLAIN's own out-of-fold predictions ON THIS
# PARTITION (each compound is held out exactly once per repeat), never from
# outputs/11_caruana_prep/, which lives on the random partition.
R = 4
DEADZONE_READY[R] = False
if ABORT["aborted"] or R >= N_REPEATS_ACTIVE:
    print(f"[skipped] repeat {R} is outside the active design or the run has aborted")
else:
    plain_files = [PRED_DIR / f"plain__r{R}f{f}.csv" for f in range(N_FOLDS)]
    missing = [p.name for p in plain_files if not p.exists()]
    if missing:
        msg = (f"PLAIN's repeat {R} is incomplete ({len(missing)} of {N_FOLDS} fold "
               f"predictions missing: {missing}). DEADZONE for repeat {R} is SKIPPED -- "
               f"falling back to the random-partition OOF is explicitly ruled out.")
        print(f"*** {msg}")
        record_decision(decision=f"Skipped the DEADZONE arm for repeat {R}", reason=msg,
                        alternatives="Use outputs/11_caruana_prep/'s random-partition OOF as the target (explicitly forbidden by the brief).",
                        authority="the brief's own DEADZONE dependency rule")
    else:
        oof = pd.concat([pd.read_csv(p)[["inchikey"] + list(REGRESSION_ENDPOINTS)]
                         for p in plain_files], ignore_index=True)
        assert len(oof) == len(curated), f"PLAIN repeat {R} OOF has {len(oof)} rows, expected {len(curated)}"
        assert oof["inchikey"].is_unique, "a compound appears in more than one fold's held-out set"
        assert set(oof["inchikey"]) == curated_ik

        curated_dz = curated.copy()
        report = []
        for iso in ISOFORMS:
            ep = PIC50_COL[iso]
            m = curated[["inchikey", ep, f"{ep}_conf_low", f"{ep}_conf_high"]].merge(
                oof[["inchikey", ep]].rename(columns={ep: "_oof"}), on="inchikey", how="left")
            assert m["inchikey"].tolist() == curated["inchikey"].tolist(), \
                "left merge did not preserve curated row order -- positional assignment would misalign"
            assert m["_oof"].notna().all(), f"{iso}: a compound has no PLAIN OOF prediction"
            has_label = m[ep].notna()
            clipped = m["_oof"].clip(lower=m[f"{ep}_conf_low"], upper=m[f"{ep}_conf_high"])
            target = np.where(has_label, clipped, np.nan)
            curated_dz[ep] = target

            differs = (np.abs(target - m[ep].to_numpy()) > 1e-6) & has_label.to_numpy()
            frac_differs = float(differs.sum() / has_label.sum())
            inside = ((m["_oof"] >= m[f"{ep}_conf_low"]) & (m["_oof"] <= m[f"{ep}_conf_high"]))
            hit_rate = float(inside[has_label].mean())
            report.append({"repeat": R, "isoform": iso, "n_labelled": int(has_label.sum()),
                            "frac_target_differs_from_label": frac_differs,
                            "band_hit_rate": hit_rate})
        rep = pd.DataFrame(report)
        print(rep.to_string(index=False))
        _append_rows(OUT / "deadzone_target_checks.csv", report, ["repeat", "isoform"])

        # Check A (materiality, deliberately loose) and Check B (the real guard: an in-fold /
        # overfit donor would push the band hit rate toward 1.0). Notebook 31 reported 100% and
        # 18.0-41.7% respectively.
        bad = rep[(rep["frac_target_differs_from_label"] < 0.50) | (rep["band_hit_rate"] >= 0.90)]
        if len(bad):
            msg = (f"DEADZONE target check failed for repeat {R}:\n{bad.to_string(index=False)}\n"
                   f"The clipped target has silently become the identity, or the OOF join leaked "
                   f"in-fold predictions. Arm SKIPPED for this repeat.")
            print(f"*** {msg}")
            record_decision(decision=f"Skipped the DEADZONE arm for repeat {R}", reason=msg,
                            alternatives="Train the arm anyway on a target that is not the intended intervention.",
                            authority="the brief's own pre-training assertion on the DEADZONE target")
        else:
            dz_path = OUT / f"curated_deadzone_target_r{R}.csv"
            curated_dz.to_csv(dz_path, index=False)
            DEADZONE_TARGETS[R] = curated_dz
            DEADZONE_READY[R] = True
            print(f"\nDEADZONE target for repeat {R} PASSED both checks -> {dz_path.name}")

[skipped] repeat 4 is outside the active design or the run has aborted


In [146]:
# DEADZONE -- repeat 4, fold 0
run_unit(4, 0, 'deadzone')

[outside the reduced design: 2 repeats active] deadzone__r4f0


In [147]:
# DEADZONE -- repeat 4, fold 1
run_unit(4, 1, 'deadzone')

[outside the reduced design: 2 repeats active] deadzone__r4f1


In [148]:
# DEADZONE -- repeat 4, fold 2
run_unit(4, 2, 'deadzone')

[outside the reduced design: 2 repeats active] deadzone__r4f2


In [149]:
# DEADZONE -- repeat 4, fold 3
run_unit(4, 3, 'deadzone')

[outside the reduced design: 2 repeats active] deadzone__r4f3


In [150]:
# DEADZONE -- repeat 4, fold 4
run_unit(4, 4, 'deadzone')

[outside the reduced design: 2 repeats active] deadzone__r4f4


In [151]:
# ---- Progress snapshot after repeat 4
if RUN_SCORES_PATH.exists():
    _rs = pd.read_csv(RUN_SCORES_PATH)
    _done = _rs[["repeat", "fold", "arm"]].drop_duplicates()
    print(f"completed (repeat, fold, arm) units so far: {len(_done)}")
    print(_done.groupby(["repeat", "arm"]).size().unstack(fill_value=0).to_string())
    _ma = _rs[(_rs["Endpoint"] == "MA") & (_rs["repeat"] == 4)]
    if len(_ma):
        print(f"\nrepeat 4 macro ST-RAE by arm (mean over completed folds):")
        print(_ma.groupby("arm")["ST-RAE"].agg(["mean", "std", "size"]).round(4).to_string())
_el = (time.time() - RUN_STARTED_AT) / 3600.0
print(f"\nelapsed since campaign start: {_el:.2f} h  (wall guard {WALL_DEADLINE_H} h)")
if DEADLINE_SKIPS:
    print(f"units skipped by the wall guard so far: {len(DEADLINE_SKIPS)}")

completed (repeat, fold, arm) units so far: 100
arm     aid  deadzone  ecfp4_narrow__lightgbm  ecfp4_narrow__rf  ecfp4_narrow__xgboost  mordred_pca__lightgbm  mordred_pca__rf  mordred_pca__xgboost  plain  single
repeat                                                                                                                                                             
0         5         5                       5                 5                      5                      5                5                     5      5       5
1         5         5                       5                 5                      5                      5                5                     5      5       5

elapsed since campaign start: 11.42 h  (wall guard 22.0 h)


## Part 4 — Analysis

Read-only from here on: no model is trained below this point. Everything comes from
`run_scores.csv` and the saved per-run out-of-fold predictions.

In [152]:
# ---- What actually completed. Every table below is built from this, so a reduced or
# interrupted run analyses exactly what exists rather than assuming the full design.
assert RUN_SCORES_PATH.exists(), "no run_scores.csv -- nothing was trained"
rs = pd.read_csv(RUN_SCORES_PATH)
units = rs[["repeat", "fold", "arm"]].drop_duplicates()
COMPLETED = {arm: sorted(map(tuple, units.loc[units["arm"] == arm, ["repeat", "fold"]].values.tolist()))
             for arm in units["arm"].unique()}
print("completed (repeat, fold) units per arm:")
for arm in ALL_ARMS:
    u = COMPLETED.get(arm, [])
    print(f"  {arm:24s} {len(u):3d} units  repeats present: {sorted({r for r, _ in u})}")
ARMS_PRESENT = [a for a in ALL_ARMS if len(COMPLETED.get(a, [])) > 0]
# Units where PLAIN and this arm both ran -- the only ones a paired test may use.
PLAIN_UNITS = set(COMPLETED.get("plain", []))
print(f"\nPLAIN completed {len(PLAIN_UNITS)} units -- the pairing base.")
if ABORT["aborted"]:
    print(f"\n*** NOTE: the run aborted at {ABORT['where']}: {ABORT['reason']}")
if DEADLINE_SKIPS:
    print(f"*** NOTE: {len(DEADLINE_SKIPS)} units skipped by the {WALL_DEADLINE_H}h wall guard")

completed (repeat, fold) units per arm:
  plain                     10 units  repeats present: [0, 1]
  ecfp4_narrow__rf          10 units  repeats present: [0, 1]
  ecfp4_narrow__xgboost     10 units  repeats present: [0, 1]
  ecfp4_narrow__lightgbm    10 units  repeats present: [0, 1]
  mordred_pca__rf           10 units  repeats present: [0, 1]
  mordred_pca__xgboost      10 units  repeats present: [0, 1]
  mordred_pca__lightgbm     10 units  repeats present: [0, 1]
  single                    10 units  repeats present: [0, 1]
  aid                       10 units  repeats present: [0, 1]
  deadzone                  10 units  repeats present: [0, 1]

PLAIN completed 10 units -- the pairing base.


In [153]:
# ---- 4.0 Assemble the per-isoform WIDE out-of-fold tables, one column per arm, matching
# outputs/11_caruana_prep/oof_long_{iso}.csv's own column shape
# (repeat, fold, inchikey, Molecule_Name, <one column per model>, y_true, conf_low, conf_high)
# so downstream blending can consume them with no converter.
oof_parts = []
for arm in ARMS_PRESENT:
    for (r, f) in COMPLETED[arm]:
        p = OOF_DIR / f"{arm}__r{r}f{f}.csv"
        if p.exists():
            oof_parts.append(pd.read_csv(p))
oof_all = pd.concat(oof_parts, ignore_index=True)
print(f"raw per-run OOF rows: {len(oof_all)}")

OOF_WIDE = {}
for iso in ISOFORMS:
    ep = PIC50_COL[iso]
    sub = oof_all[oof_all["isoform"] == iso]
    wide = sub.pivot_table(index=["repeat", "fold", "inchikey"], columns="arm",
                           values="y_pred", aggfunc="first").reset_index()
    wide.columns.name = None
    truth = curated[["inchikey", "Molecule_Name", ep, f"{ep}_conf_low", f"{ep}_conf_high"]].rename(
        columns={ep: "y_true", f"{ep}_conf_low": "conf_low", f"{ep}_conf_high": "conf_high"})
    wide = wide.merge(truth, on="inchikey", how="left")
    # Labelled rows only. np.corrcoef propagates NaN the instant either input has one, which
    # silently produced an all-NaN correlation table in notebook 31 -- dropped here, once, at
    # the source, rather than guarded at every call site.
    before = len(wide)
    wide = wide[wide["y_true"].notna()].reset_index(drop=True)
    arm_cols = [c for c in ARMS_PRESENT if c in wide.columns]
    ordered = ["repeat", "fold", "inchikey", "Molecule_Name"] + arm_cols + ["y_true", "conf_low", "conf_high"]
    wide = wide[ordered]
    OOF_WIDE[iso] = wide
    wide.to_csv(OOF_DIR / f"oof_long_{iso}.csv", index=False)
    print(f"{iso}: {before} rows -> {len(wide)} labelled  ({len(arm_cols)} arm columns)  "
          f"nulls per arm: {dict(wide[arm_cols].isna().sum())}")
print(f"\nwrote {len(ISOFORMS)} wide tables to {OOF_DIR}")

raw per-run OOF rows: 392400
CYP1A2: 9810 rows -> 2824 labelled  (10 arm columns)  nulls per arm: {'plain': 0, 'ecfp4_narrow__rf': 0, 'ecfp4_narrow__xgboost': 0, 'ecfp4_narrow__lightgbm': 0, 'mordred_pca__rf': 0, 'mordred_pca__xgboost': 0, 'mordred_pca__lightgbm': 0, 'single': 0, 'aid': 0, 'deadzone': 0}
CYP2C9: 9810 rows -> 2570 labelled  (10 arm columns)  nulls per arm: {'plain': 0, 'ecfp4_narrow__rf': 0, 'ecfp4_narrow__xgboost': 0, 'ecfp4_narrow__lightgbm': 0, 'mordred_pca__rf': 0, 'mordred_pca__xgboost': 0, 'mordred_pca__lightgbm': 0, 'single': 0, 'aid': 0, 'deadzone': 0}
CYP2D6: 9810 rows -> 2986 labelled  (10 arm columns)  nulls per arm: {'plain': 0, 'ecfp4_narrow__rf': 0, 'ecfp4_narrow__xgboost': 0, 'ecfp4_narrow__lightgbm': 0, 'mordred_pca__rf': 0, 'mordred_pca__xgboost': 0, 'mordred_pca__lightgbm': 0, 'single': 0, 'aid': 0, 'deadzone': 0}


CYP3A4: 9810 rows -> 4670 labelled  (10 arm columns)  nulls per arm: {'plain': 0, 'ecfp4_narrow__rf': 0, 'ecfp4_narrow__xgboost': 0, 'ecfp4_narrow__lightgbm': 0, 'mordred_pca__rf': 0, 'mordred_pca__xgboost': 0, 'mordred_pca__lightgbm': 0, 'single': 0, 'aid': 0, 'deadzone': 0}

wrote 4 wide tables to /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/34_butina_5x5/oof_long


### 4.1 Model comparison

Per isoform, per arm: out-of-fold ST-RAE, MAE, R2 and Spearman as mean and across-fold spread.
Paired tests against PLAIN follow this project's established protocol: **Levene's
homogeneity-of-variance branch is checked rather than assumed** (Brown–Forsythe, median-centred,
as in notebooks 05/24/29) — homogeneous variances give a paired *t*-test, a violation gives a
Wilcoxon signed-rank test — and Benjamini–Hochberg correction is applied **across arms within each
isoform** (the family of comparisons a reader makes when asking "which arm beats PLAIN on this
isoform"). The test actually used is reported per row, not just its p-value.

In [154]:
METRICS_4_1 = ["ST-RAE", "MAE", "R2", "Spearman_R"]
rs_ep = rs[rs["Endpoint"].isin(REGRESSION_ENDPOINTS)].copy()
rs_ep["isoform"] = rs_ep["Endpoint"].map(EP_TO_ISO)

summary_rows = []
for iso in ISOFORMS:
    for arm in ARMS_PRESENT:
        sub = rs_ep[(rs_ep["isoform"] == iso) & (rs_ep["arm"] == arm)]
        if not len(sub):
            continue
        row = {"isoform": iso, "arm": arm, "n_folds": len(sub)}
        for m in METRICS_4_1:
            row[f"{m}_mean"] = float(sub[m].mean())
            row[f"{m}_std"] = float(sub[m].std(ddof=1)) if len(sub) > 1 else np.nan
        summary_rows.append(row)
arm_summary = pd.DataFrame(summary_rows)
arm_summary.to_csv(OUT / "arm_summary.csv", index=False)

print("out-of-fold ST-RAE (lower is better), mean +/- across-fold std:")
piv = arm_summary.pivot(index="arm", columns="isoform", values="ST-RAE_mean")
pivs = arm_summary.pivot(index="arm", columns="isoform", values="ST-RAE_std")
order = piv.mean(axis=1).sort_values().index
for a in order:
    print(f"  {a:24s} " + "  ".join(f"{iso} {piv.loc[a,iso]:.4f}+/-{pivs.loc[a,iso]:.4f}"
                                    for iso in ISOFORMS if not pd.isna(piv.loc[a, iso])))

out-of-fold ST-RAE (lower is better), mean +/- across-fold std:
  deadzone                 CYP1A2 0.7717+/-0.0493  CYP2C9 0.5625+/-0.0479  CYP2D6 0.8850+/-0.0431  CYP3A4 0.4740+/-0.0372
  aid                      CYP1A2 0.8136+/-0.0556  CYP2C9 0.6442+/-0.0865  CYP2D6 0.9546+/-0.0740  CYP3A4 0.5607+/-0.0387
  plain                    CYP1A2 0.8544+/-0.0634  CYP2C9 0.6455+/-0.0566  CYP2D6 1.0179+/-0.0656  CYP3A4 0.5395+/-0.0437
  single                   CYP1A2 0.8841+/-0.0640  CYP2C9 0.7034+/-0.0831  CYP2D6 0.9960+/-0.0750  CYP3A4 0.5447+/-0.0337
  ecfp4_narrow__lightgbm   CYP1A2 0.8792+/-0.0447  CYP2C9 0.7583+/-0.0458  CYP2D6 0.9644+/-0.0474  CYP3A4 0.6022+/-0.0351
  ecfp4_narrow__rf         CYP1A2 0.8671+/-0.0488  CYP2C9 0.7458+/-0.0635  CYP2D6 1.0011+/-0.0562  CYP3A4 0.6201+/-0.0368
  ecfp4_narrow__xgboost    CYP1A2 0.9024+/-0.0545  CYP2C9 0.7861+/-0.0440  CYP2D6 1.0129+/-0.0596  CYP3A4 0.6388+/-0.0293
  mordred_pca__lightgbm    CYP1A2 0.9520+/-0.0494  CYP2C9 0.8675+/-0.0806  CYP2D6 

In [155]:
def paired_test(x, y, label):
    # x, y are the same (repeat, fold) units in the same order. Returns the branch actually
    # taken, not a pre-chosen test.
    d = np.asarray(y, float) - np.asarray(x, float)
    n = len(d)
    if n < 3:
        return {"test": "none (n<3)", "stat": np.nan, "p": np.nan, "mean_diff": float(d.mean()) if n else np.nan,
                "levene_p": np.nan, "n": n, "se": np.nan, "sign_resolved": False,
                "ci_low": np.nan, "ci_high": np.nan, "dz": np.nan,
                "n_improving": int((d < 0).sum())}
    lev_stat, lev_p = stats.levene(np.asarray(x, float), np.asarray(y, float), center="median")
    if lev_p > 0.05:
        test = "paired t"
        stat, p = stats.ttest_rel(np.asarray(y, float), np.asarray(x, float))
    else:
        test = "Wilcoxon signed-rank"
        try:
            stat, p = stats.wilcoxon(np.asarray(y, float), np.asarray(x, float))
        except ValueError:
            stat, p = np.nan, 1.0
    sd = d.std(ddof=1)
    se = sd / np.sqrt(n)
    tcrit = stats.t.ppf(0.975, n - 1)
    return {"test": test, "stat": float(stat), "p": float(p), "mean_diff": float(d.mean()),
            "levene_p": float(lev_p), "n": n, "se": float(se),
            "sign_resolved": bool(abs(d.mean()) > se),
            "ci_low": float(d.mean() - tcrit * se),
            "ci_high": float(d.mean() + tcrit * se),
            "dz": float(d.mean() / sd) if sd > 0 else np.nan,
            "n_improving": int((d < 0).sum())}


def bh(pvals):
    p = np.asarray(pvals, float)
    ok = ~np.isnan(p)
    out = np.full_like(p, np.nan)
    if ok.sum() == 0:
        return out
    q = p[ok]
    order = np.argsort(q)
    m = len(q)
    adj = np.empty(m)
    prev = 1.0
    for rank in range(m - 1, -1, -1):
        val = q[order[rank]] * m / (rank + 1)
        prev = min(prev, val)
        adj[order[rank]] = prev
    out[ok] = np.clip(adj, 0, 1)
    return out


test_rows = []
for iso in ISOFORMS:
    for metric in ["ST-RAE", "MAE"]:
        for arm in ARMS_PRESENT:
            if arm == "plain":
                continue
            shared = sorted(set(COMPLETED[arm]) & PLAIN_UNITS)
            if len(shared) < 3:
                continue
            key = lambda a: (rs_ep[(rs_ep["isoform"] == iso) & (rs_ep["arm"] == a)]
                             .set_index(["repeat", "fold"])[metric])
            sp, sa = key("plain"), key(arm)
            x = [float(sp.loc[u]) for u in shared]
            y = [float(sa.loc[u]) for u in shared]
            res = paired_test(x, y, f"{iso}/{arm}")
            res.update({"isoform": iso, "metric": metric, "arm": arm,
                        "mean_plain": float(np.mean(x)), "mean_arm": float(np.mean(y))})
            test_rows.append(res)
paired = pd.DataFrame(test_rows)
# BH across arms within each (isoform, metric) family.
paired["p_BH"] = np.nan
for (iso, metric), grp in paired.groupby(["isoform", "metric"]):
    paired.loc[grp.index, "p_BH"] = bh(grp["p"].to_numpy())
paired["verdict"] = np.where(paired["p_BH"] >= 0.05, "NOT DIFFERENT",
                    np.where(paired["mean_diff"] < 0, "SIGNIFICANTLY BETTER", "SIGNIFICANTLY WORSE"))
paired.to_csv(OUT / "paired_tests_vs_plain.csv", index=False)

for metric in ["ST-RAE", "MAE"]:
    print(f"\n=== paired tests vs PLAIN, {metric} (BH-corrected across arms within isoform) ===")
    sub = paired[paired["metric"] == metric]
    cols = ["isoform", "arm", "n", "mean_plain", "mean_arm", "mean_diff", "ci_low", "ci_high",
            "levene_p", "test", "p", "p_BH", "n_improving", "verdict"]
    print(sub[cols].round(4).to_string(index=False))


=== paired tests vs PLAIN, ST-RAE (BH-corrected across arms within isoform) ===
isoform                    arm  n  mean_plain  mean_arm  mean_diff  ci_low  ci_high  levene_p     test      p   p_BH  n_improving              verdict
 CYP1A2       ecfp4_narrow__rf 10      0.8544    0.8671     0.0127 -0.0352   0.0607    0.8696 paired t 0.5627 0.5627            4        NOT DIFFERENT
 CYP1A2  ecfp4_narrow__xgboost 10      0.8544    0.9024     0.0481 -0.0106   0.1068    0.9882 paired t 0.0969 0.1454            4        NOT DIFFERENT
 CYP1A2 ecfp4_narrow__lightgbm 10      0.8544    0.8792     0.0249 -0.0175   0.0673    0.6290 paired t 0.2173 0.2579            3        NOT DIFFERENT
 CYP1A2        mordred_pca__rf 10      0.8544    0.9411     0.0868  0.0318   0.1417    0.5087 paired t 0.0060 0.0135            2  SIGNIFICANTLY WORSE
 CYP1A2   mordred_pca__xgboost 10      0.8544    0.9878     0.1335  0.0968   0.1701    0.4528 paired t 0.0000 0.0002            0  SIGNIFICANTLY WORSE
 CYP1A2  mord

### 4.2 The split question

Every arm's Butina result is set beside its random-partition result wherever one exists, all read
**read-only**: `outputs/05_cv_comparison/` (PLAIN as `chemprop_chemeleoninit`, and all six tree
configs, 25 folds), `outputs/29_cv_confirmation/` (AID, 25 folds), `outputs/31_deadzone/`
(DEADZONE, one fold x three seeds). Each source is paired against **its own** PLAIN/baseline arm,
never against another notebook's, so each partition's difference is internally consistent.

**SINGLE has no random-partition reference** anywhere in this project — single-task Chemprop has
never been run on the frozen partition — so its row is reported as unavailable rather than
compared against something it is not comparable to.

In [156]:
# ---- Random-partition reference differences vs. that source's own PLAIN/baseline arm.
def nb05_strae(config):
    rows = []
    for r in range(5):
        for f in range(5):
            p = NB05_OUT / "scores" / f"{config}__repeat{r}_fold{f}.csv"
            if not p.exists():
                continue
            d = pd.read_csv(p)
            g = d.groupby("Endpoint")["ST-RAE"].mean()
            for ep, v in g.items():
                if ep in REGRESSION_ENDPOINTS:
                    rows.append({"repeat": r, "fold": f, "isoform": EP_TO_ISO[ep], "ST-RAE": float(v)})
    return pd.DataFrame(rows)


ref_rows = []
nb05_plain = nb05_strae("chemprop_chemeleoninit")
assert len(nb05_plain) == 100, f"expected 100 (fold, isoform) cells from notebook 05, got {len(nb05_plain)}"
for cfg in TREE_ARMS:
    t = nb05_strae(cfg)
    if not len(t):
        print(f"  no notebook 05 scores for {cfg} -- skipped")
        continue
    for iso in ISOFORMS:
        a = nb05_plain[nb05_plain["isoform"] == iso].set_index(["repeat", "fold"])["ST-RAE"]
        b = t[t["isoform"] == iso].set_index(["repeat", "fold"])["ST-RAE"]
        shared = sorted(set(a.index) & set(b.index))
        dd = (b.loc[shared] - a.loc[shared])
        se_ = float(dd.std(ddof=1) / np.sqrt(len(shared))) if len(shared) > 1 else np.nan
        ref_rows.append({"arm": cfg, "isoform": iso, "source": "nb05 (25 folds)",
                         "random_plain": float(a.loc[shared].mean()),
                         "random_arm": float(b.loc[shared].mean()),
                         "random_diff": float(dd.mean()), "random_se": se_,
                         "random_sign_resolved": bool(abs(dd.mean()) > se_) if se_ == se_ else False,
                         "random_n": len(shared)})

nb29 = pd.read_csv(NB29_OUT / "run_scores.csv")
nb29["isoform"] = nb29["Endpoint"].map(EP_TO_ISO)
for iso in ISOFORMS:
    a = nb29[(nb29["arm"] == "baseline") & (nb29["isoform"] == iso)].set_index(["repeat", "fold"])["ST-RAE"]
    b = nb29[(nb29["arm"] == "aid") & (nb29["isoform"] == iso)].set_index(["repeat", "fold"])["ST-RAE"]
    shared = sorted(set(a.index) & set(b.index))
    dd = (b.loc[shared] - a.loc[shared])
    se_ = float(dd.std(ddof=1) / np.sqrt(len(shared)))
    ref_rows.append({"arm": "aid", "isoform": iso, "source": "nb29 (25 folds)",
                     "random_plain": float(a.loc[shared].mean()), "random_arm": float(b.loc[shared].mean()),
                     "random_diff": float(dd.mean()), "random_se": se_,
                     "random_sign_resolved": bool(abs(dd.mean()) > se_), "random_n": len(shared)})

nb31 = pd.read_csv(NB31_OUT / "run_scores.csv")
nb31["isoform"] = nb31["Endpoint"].map(EP_TO_ISO)
nb31f = nb31[nb31["Endpoint"].isin(REGRESSION_ENDPOINTS)].groupby(
    ["arm", "seed", "isoform"], as_index=False)["ST-RAE"].mean()
for iso in ISOFORMS:
    a = nb31f[(nb31f["arm"] == "baseline") & (nb31f["isoform"] == iso)].set_index("seed")["ST-RAE"]
    b = nb31f[(nb31f["arm"] == "deadzone") & (nb31f["isoform"] == iso)].set_index("seed")["ST-RAE"]
    shared = sorted(set(a.index) & set(b.index))
    dd = (b.loc[shared] - a.loc[shared])
    se_ = float(dd.std(ddof=1) / np.sqrt(len(shared))) if len(shared) > 1 else np.nan
    ref_rows.append({"arm": "deadzone", "isoform": iso, "source": "nb31 (1 fold x 3 seeds)",
                     "random_plain": float(a.loc[shared].mean()), "random_arm": float(b.loc[shared].mean()),
                     "random_diff": float(dd.mean()), "random_se": se_,
                     "random_sign_resolved": bool(abs(dd.mean()) > se_) if se_ == se_ else False,
                     "random_n": len(shared)})

ref = pd.DataFrame(ref_rows)
print(f"random-partition reference differences assembled: {len(ref)} (arm, isoform) cells")
print(ref.groupby(["arm", "source"]).size().to_string())

random-partition reference differences assembled: 32 (arm, isoform) cells
arm                     source                 
aid                     nb29 (25 folds)            4
deadzone                nb31 (1 fold x 3 seeds)    4
ecfp4_narrow__lightgbm  nb05 (25 folds)            4
ecfp4_narrow__rf        nb05 (25 folds)            4
ecfp4_narrow__xgboost   nb05 (25 folds)            4
mordred_pca__lightgbm   nb05 (25 folds)            4
mordred_pca__rf         nb05 (25 folds)            4
mordred_pca__xgboost    nb05 (25 folds)            4


In [157]:
# ---- Sign agreement and ordering agreement between the two partitions.
but = paired[paired["metric"] == "ST-RAE"][["isoform", "arm", "mean_plain", "mean_arm",
                                            "mean_diff", "se", "sign_resolved", "p_BH",
                                            "verdict", "n"]].rename(
    columns={"mean_plain": "butina_plain", "mean_arm": "butina_arm",
             "mean_diff": "butina_diff", "se": "butina_se",
             "sign_resolved": "butina_sign_resolved", "p_BH": "butina_p_BH",
             "verdict": "butina_verdict", "n": "butina_n"})
agree = but.merge(ref, on=["arm", "isoform"], how="left")

# A sign-agreement test is only meaningful where BOTH partitions actually resolved a sign.
# This is not a cosmetic guard: on CYP1A2, notebook 05's own 25-fold difference between
# ecfp4_narrow__rf and chemprop_chemeleoninit is +0.000019 ST-RAE -- a genuine near-tie,
# verified directly against its stored per-fold scores. Comparing signs there manufactures a
# "flip" out of arithmetic noise. "Resolved" means |mean difference| exceeds its own standard
# error across the folds it was measured on -- a data-derived bar, not an invented epsilon.
# .eq(True) rather than .fillna(False).astype(bool): identical result on an object column
# holding True/False/NA, without pandas' deprecated silent downcast.
agree["both_resolved"] = (agree["random_diff"].notna()
                          & agree["butina_sign_resolved"].eq(True)
                          & agree["random_sign_resolved"].eq(True))
_same = np.sign(agree["butina_diff"]) == np.sign(agree["random_diff"])
agree["sign_agrees"] = np.where(agree["both_resolved"], _same, pd.NA)
agree["flip"] = ["" if pd.isna(v) else ("" if v else "FLIP") for v in agree["sign_agrees"]]
agree["status"] = np.where(agree["random_diff"].isna(), "no random reference",
                  np.where(~agree["both_resolved"], "sign not resolved",
                  np.where(agree["sign_agrees"] == True, "agrees", "FLIP")))
agree.to_csv(OUT / "partition_agreement.csv", index=False)

print("=== SIGN of each arm's ST-RAE difference against PLAIN, under both partitions ===")
print("(negative = better than PLAIN. A cell counts as a FLIP only when BOTH partitions")
print(" resolved a sign, i.e. |difference| > its own standard error on each side.)")
print(agree[["isoform", "arm", "source", "butina_diff", "butina_se", "random_diff", "random_se",
             "status", "butina_verdict"]].round(4).to_string(index=False))

res = agree[agree["both_resolved"]]
unres = agree[(agree["random_diff"].notna()) & (~agree["both_resolved"])]
print("")
print(f"sign agreement, over the {len(res)} (arm, isoform) cells where both partitions resolved a")
print(f"sign: {int((res['sign_agrees'] == True).sum())} agree, "
      f"{int((res['sign_agrees'] == False).sum())} flip.")
print(f"{len(unres)} further comparable cells had an UNRESOLVED sign on one or both sides and are")
print("excluded rather than counted either way:")
if len(unres):
    print(unres[["isoform", "arm", "butina_diff", "butina_se", "random_diff", "random_se"]]
          .round(5).to_string(index=False))
missing_ref = sorted(set(agree.loc[agree['random_diff'].isna(), 'arm']))
if missing_ref:
    print(f"no random-partition reference exists for: {missing_ref} -- reported as unavailable, "
          f"not compared.")

print("")
print("--- the AID question specifically ---")
nb33_tests = pd.read_csv(NB33_OUT / "part3_paired_tests.csv")
aid_rows = agree[agree["arm"] == "aid"]
if not len(aid_rows):
    print("The AID arm has fewer than 3 completed (repeat, fold) units paired with PLAIN, so no")
    print("paired difference exists for it yet and this comparison cannot be made. Reported as")
    print("unavailable rather than as a count of zero flips.")
else:
    # Notebook 33's "flips sign on three of four isoforms" is reconciled here against BOTH
    # possible bases rather than one being declared wrong. Against its OWN 5-fold random slice
    # the count is 4 of 4; against notebook 29's full 25-fold differences it is 3 of 4. The one
    # cell that separates the two bases is CYP2C9, whose 25-fold random difference is not
    # sign-resolved in the first place -- so "3 of 4" is right on the 25-fold basis, and the
    # apparent discrepancy is a property of that unresolved cell, not an error in either source.
    rows33 = []
    for iso in ISOFORMS:
        b33 = float(nb33_tests[(nb33_tests["partition"] == "BUTINA") &
                               (nb33_tests["isoform"] == iso)]["mean_diff"].iloc[0])
        r33 = float(nb33_tests[(nb33_tests["partition"] == "RANDOM") &
                               (nb33_tests["isoform"] == iso)]["mean_diff"].iloc[0])
        ar = aid_rows[aid_rows["isoform"] == iso]
        r29 = float(ar["random_diff"].iloc[0]) if len(ar) else np.nan
        se29 = float(ar["random_se"].iloc[0]) if len(ar) else np.nan
        bhere = float(ar["butina_diff"].iloc[0]) if len(ar) else np.nan
        rows33.append({"isoform": iso, "nb33_butina_5f": b33, "nb33_random_5f": r33,
                       "nb29_random_25f": r29, "nb29_random_se": se29,
                       "nb29_sign_resolved": bool(abs(r29) > se29) if se29 == se29 else False,
                       "flip_vs_nb33_own_random": bool(np.sign(b33) != np.sign(r33)),
                       "flip_vs_nb29_25fold": bool(np.sign(b33) != np.sign(r29)),
                       "this_notebook_butina": bhere,
                       "this_notebook_status": ar["status"].iloc[0] if len(ar) else "n/a"})
    aid_recon = pd.DataFrame(rows33)
    aid_recon.to_csv(OUT / "aid_flip_reconciliation.csv", index=False)
    print(aid_recon.round(4).to_string(index=False))
    n_own = int(aid_recon["flip_vs_nb33_own_random"].sum())
    n_29 = int(aid_recon["flip_vs_nb29_25fold"].sum())
    print("")
    print(f"notebook 33's OWN Butina differences, recomputed from its saved part3_paired_tests.csv:")
    print(f"  {n_own} of 4 sign changes against its own 5-fold random slice;")
    print(f"  {n_29} of 4 against notebook 29's full 25-fold random differences.")
    print(f"  Its prose says 'three of four'. That is exactly right on the 25-fold basis. The cell")
    print(f"  that separates the two counts is CYP2C9, whose 25-fold random difference is not")
    print(f"  sign-resolved (|diff| < its own SE), so it never had a direction to reverse.")
    print("")
    print("THIS notebook's repeated design, per isoform:")
    for _, row in aid_recon.iterrows():
        print(f"  {row['isoform']}: nb29 random(25f) {row['nb29_random_25f']:+.4f} | "
              f"nb33 butina(5f) {row['nb33_butina_5f']:+.4f} | "
              f"this notebook butina {row['this_notebook_butina']:+.4f} -> {row['this_notebook_status']}")
    aid_flips_here = int((aid_rows["sign_agrees"] == False).sum())
    aid_resolved_here = int(aid_rows["both_resolved"].sum())
    print("")
    print(f"AID sign flips in THIS notebook's repeated design: {aid_flips_here} of "
          f"{aid_resolved_here} sign-resolved isoforms "
          f"({len(aid_rows) - aid_resolved_here} unresolved, excluded).")
    if aid_resolved_here == 0:
        print("  -> No isoform resolved a sign on both sides, so this design can neither confirm")
        print("     nor overturn notebook 33's flip. Reported as inconclusive, not as agreement.")
    elif aid_flips_here >= 2:
        print("  -> the repeated design CONFIRMS notebook 33's flip on the isoforms it can resolve.")
    elif aid_flips_here == 0:
        print("  -> the repeated design OVERTURNS notebook 33's flip on every isoform it resolves.")
    else:
        print("  -> the repeated design PARTLY reproduces notebook 33's flip.")
    print("  A flip at screening tier reversing again here is a legitimate outcome, not an error:")
    print("  notebook 33's five-fold, one-partition result sits at the same evidentiary tier as")
    print("  notebook 28's AID screen, which notebook 29 then overturned.")

=== SIGN of each arm's ST-RAE difference against PLAIN, under both partitions ===
(negative = better than PLAIN. A cell counts as a FLIP only when BOTH partitions
 resolved a sign, i.e. |difference| > its own standard error on each side.)
isoform                    arm                  source  butina_diff  butina_se  random_diff  random_se              status       butina_verdict
 CYP1A2       ecfp4_narrow__rf         nb05 (25 folds)       0.0127     0.0212       0.0000     0.0095   sign not resolved        NOT DIFFERENT
 CYP1A2  ecfp4_narrow__xgboost         nb05 (25 folds)       0.0481     0.0259       0.0483     0.0097              agrees        NOT DIFFERENT
 CYP1A2 ecfp4_narrow__lightgbm         nb05 (25 folds)       0.0249     0.0187       0.0066     0.0104   sign not resolved        NOT DIFFERENT
 CYP1A2        mordred_pca__rf         nb05 (25 folds)       0.0868     0.0243       0.0723     0.0091              agrees  SIGNIFICANTLY WORSE
 CYP1A2   mordred_pca__xgboost         nb

In [158]:
# ---- Ordering agreement: do the two partitions rank the arms the same way?
order_rows = []
for iso in ISOFORMS:
    b = arm_summary[(arm_summary["isoform"] == iso)].set_index("arm")["ST-RAE_mean"]
    # Random-partition ORDER is only defined for arms with a reference; PLAIN is included via
    # each source's own baseline, and the tree arms + PLAIN all come from notebook 05, so they
    # are directly comparable on one scale. AID/DEADZONE come from other notebooks, so their
    # absolute level is not on nb05's scale -- they are placed by their own diff-vs-baseline
    # added to nb05's PLAIN level, stated explicitly as a construction rather than a measurement.
    r_level = {}
    nb05_plain_iso = float(nb05_plain[nb05_plain["isoform"] == iso]["ST-RAE"].mean())
    r_level["plain"] = nb05_plain_iso
    for _, row in ref[ref["isoform"] == iso].iterrows():
        if row["source"].startswith("nb05"):
            r_level[row["arm"]] = float(row["random_arm"])
        else:
            r_level[row["arm"]] = nb05_plain_iso + float(row["random_diff"])
    common = [a for a in b.index if a in r_level]
    if len(common) >= 3:
        rho, p = stats.spearmanr([b[a] for a in common], [r_level[a] for a in common])
    else:
        rho, p = np.nan, np.nan
    order_rows.append({"isoform": iso, "n_arms_compared": len(common), "spearman_rho": rho, "p": p,
                       "butina_best": b[common].idxmin() if common else None,
                       "random_best": min(common, key=lambda a: r_level[a]) if common else None})
ordering = pd.DataFrame(order_rows)
ordering.to_csv(OUT / "ordering_agreement.csv", index=False)
print("ORDERING agreement between partitions (Spearman rank correlation of arm mean ST-RAE):")
print(ordering.round(4).to_string(index=False))
print("\nWeak evidence by construction: the rank correlation is computed over a handful of arms,")
print("and the AID/DEADZONE random-partition levels are constructed from their own")
print("diff-vs-baseline rather than measured on notebook 05's scale. Read the SIGN table above")
print("as the primary evidence and this as a summary.")

ORDERING agreement between partitions (Spearman rank correlation of arm mean ST-RAE):
isoform  n_arms_compared  spearman_rho      p butina_best random_best
 CYP1A2                9        0.9833 0.0000    deadzone    deadzone
 CYP2C9                9        0.9833 0.0000    deadzone    deadzone
 CYP2D6                9        0.8500 0.0037    deadzone    deadzone
 CYP3A4                9        0.9833 0.0000    deadzone    deadzone

Weak evidence by construction: the rank correlation is computed over a handful of arms,
and the AID/DEADZONE random-partition levels are constructed from their own
diff-vs-baseline rather than measured on notebook 05's scale. Read the SIGN table above
as the primary evidence and this as a summary.


### 4.3 Decorrelation — why TREES and SINGLE are in this run at all

Error correlation means correlation of **residuals** (prediction minus truth), not of raw
predictions — two models can track the same true values closely and still make usefully different
errors (Dietterich, 2000; this project's own rule). Computed per isoform, pooled across all
completed folds, **restricted to labelled rows** (notebook 31's version came back silently all-NaN
from sparse per-isoform labelling; the restriction is applied once at the source in 4.0).

**The floor** is PLAIN-against-PLAIN across repeats: the same compound's residual under two
different cluster-to-fold assignments and two different training seeds. That is the correlation two
runs of the *same* recipe produce, so an arm at or above it is not decorrelated in any useful sense.

Reference bands: notebook 24 measured this project's existing 11-config pool at **0.792–0.908**;
notebook 29 measured AID against baseline at **0.850–0.933**. Both verified against their own saved
files below rather than quoted.

In [159]:
# ---- Verify the two quoted reference bands against their own saved files, not the prose.
# Notebook 24's documented band (0.792-0.908) is the range across isoforms of the MEAN
# off-diagonal pairwise error correlation -- not the range of individual pairs, which its own
# text separately reports as reaching 0.965. Recomputed here on that definition.
nb24_long = pd.read_csv(NB24_OUT / "error_correlation_long.csv")
off = nb24_long[nb24_long["config_1"] != nb24_long["config_2"]]
nb24_means = off.groupby("isoform")["error_correlation"].mean()
NB24_BAND_LO, NB24_BAND_HI = float(nb24_means.min()), float(nb24_means.max())
print("notebook 24, recomputed from its own error_correlation_long.csv:")
print(f"  mean off-diagonal error correlation per isoform: "
      + ", ".join(f"{i_}={v:.3f}" for i_, v in nb24_means.items()))
print(f"  band across isoforms: {NB24_BAND_LO:.3f}-{NB24_BAND_HI:.3f}   "
      f"(documented: 0.792-0.908)")
print(f"  individual-pair max (its own separately-reported figure): "
      f"{off['error_correlation'].max():.3f} (documented: 0.965)")
_ok24 = abs(NB24_BAND_LO - 0.792) < 0.005 and abs(NB24_BAND_HI - 0.908) < 0.005
print(f"  -> {'CONFIRMED' if _ok24 else 'DOES NOT MATCH -- the recomputed values are used, not the quoted ones'}")

nb29_corr = pd.read_csv(NB29_OUT / "error_correlation.csv")
NB29_LO = float(nb29_corr["aid_vs_baseline_mean"].min())
NB29_HI = float(nb29_corr["aid_vs_baseline_mean"].max())
NB29_FLOOR_LO = float(nb29_corr["baseline_fold_floor_mean"].min())
NB29_FLOOR_HI = float(nb29_corr["baseline_fold_floor_mean"].max())
print("\nnotebook 29, recomputed from its own error_correlation.csv:")
print(f"  AID vs baseline: {NB29_LO:.3f}-{NB29_HI:.3f}   (documented: 0.850-0.933)")
print(f"  its own baseline fold-to-fold floor: {NB29_FLOOR_LO:.3f}-{NB29_FLOOR_HI:.3f} "
      f"(documented: 0.867-0.920)")
_ok29 = abs(NB29_LO - 0.850) < 0.005 and abs(NB29_HI - 0.933) < 0.005
print(f"  -> {'CONFIRMED' if _ok29 else 'PARTIAL -- the recomputed values are used, not the quoted ones'}")
print(nb29_corr.round(4).to_string(index=False))

notebook 24, recomputed from its own error_correlation_long.csv:
  mean off-diagonal error correlation per isoform: CYP1A2=0.884, CYP2C9=0.842, CYP2D6=0.908, CYP3A4=0.792
  band across isoforms: 0.792-0.908   (documented: 0.792-0.908)
  individual-pair max (its own separately-reported figure): 0.965 (documented: 0.965)
  -> CONFIRMED

notebook 29, recomputed from its own error_correlation.csv:
  AID vs baseline: 0.850-0.933   (documented: 0.850-0.933)
  its own baseline fold-to-fold floor: 0.867-0.919 (documented: 0.867-0.920)
  -> CONFIRMED
isoform  n_folds  aid_vs_baseline_mean  aid_vs_baseline_std  baseline_fold_floor_mean  baseline_fold_floor_std  n_floor_pairs
 CYP1A2       25                0.9022               0.0228                    0.9044                   0.0095             10
 CYP2C9       25                0.8806               0.0238                    0.8913                   0.0081             10
 CYP2D6       25                0.9334               0.0244               

In [160]:
# ---- Pairwise arm error correlation, per isoform, pooled across folds.
DECORR_BAR = 0.85   # the brief's own stated bar: "if TREES come back below roughly 0.85"
corr_mats, floor_rows, decorr_rows = {}, [], []

for iso in ISOFORMS:
    w = OOF_WIDE[iso]
    arm_cols = [c for c in ARMS_PRESENT if c in w.columns]
    resid = w[["repeat", "fold", "inchikey"]].copy()
    for a in arm_cols:
        resid[a] = w[a] - w["y_true"]
    # Pairwise-complete Pearson. Every row here is labelled by construction; a NaN can still
    # appear if an arm has fewer completed folds than another, so pairs are intersected.
    M = pd.DataFrame(index=arm_cols, columns=arm_cols, dtype=float)
    for a in arm_cols:
        for b in arm_cols:
            sel = resid[a].notna() & resid[b].notna()
            M.loc[a, b] = (np.corrcoef(resid.loc[sel, a], resid.loc[sel, b])[0, 1]
                           if sel.sum() > 2 else np.nan)
    corr_mats[iso] = M
    M.to_csv(OUT / f"error_correlation_{iso}.csv")

    # FLOOR: PLAIN vs PLAIN across repeat pairs (same compound, different assignment + seed).
    pl = resid[["repeat", "inchikey", "plain"]].dropna()
    reps = sorted(pl["repeat"].unique())
    fl = []
    for r1, r2 in itertools.combinations(reps, 2):
        m = (pl[pl["repeat"] == r1][["inchikey", "plain"]]
             .merge(pl[pl["repeat"] == r2][["inchikey", "plain"]], on="inchikey",
                    suffixes=("_1", "_2")))
        if len(m) > 2:
            fl.append(float(np.corrcoef(m["plain_1"], m["plain_2"])[0, 1]))
    floor = float(np.mean(fl)) if fl else np.nan
    floor_rows.append({"isoform": iso, "n_repeat_pairs": len(fl), "plain_vs_plain_mean": floor,
                       "plain_vs_plain_min": float(np.min(fl)) if fl else np.nan,
                       "plain_vs_plain_max": float(np.max(fl)) if fl else np.nan})

    for a in arm_cols:
        if a == "plain":
            continue
        c = float(M.loc[a, "plain"])
        decorr_rows.append({"isoform": iso, "arm": a, "corr_vs_plain": c,
                            "plain_floor": floor,
                            "below_floor": bool(c < floor) if not np.isnan(floor) else None,
                            "below_bar_0.85": bool(c < DECORR_BAR),
                            "below_nb24_band": bool(c < NB24_BAND_LO),
                            "decorrelated": bool(c < DECORR_BAR and (np.isnan(floor) or c < floor))})

floors = pd.DataFrame(floor_rows)
decorr = pd.DataFrame(decorr_rows)
floors.to_csv(OUT / "plain_seed_floor.csv", index=False)
decorr.to_csv(OUT / "decorrelation.csv", index=False)

print("PLAIN-vs-PLAIN across-repeat residual correlation (the floor):")
print(floors.round(4).to_string(index=False))
print(f"\nerror correlation against PLAIN, per arm per isoform (bar {DECORR_BAR}, "
      f"notebook 24 band {NB24_BAND_LO:.3f}-{NB24_BAND_HI:.3f}):")
print(decorr.pivot(index="arm", columns="isoform", values="corr_vs_plain").round(3).to_string())

PLAIN-vs-PLAIN across-repeat residual correlation (the floor):
isoform  n_repeat_pairs  plain_vs_plain_mean  plain_vs_plain_min  plain_vs_plain_max
 CYP1A2               1               0.9295              0.9295              0.9295
 CYP2C9               1               0.8996              0.8996              0.8996
 CYP2D6               1               0.9098              0.9098              0.9098
 CYP3A4               1               0.8838              0.8838              0.8838

error correlation against PLAIN, per arm per isoform (bar 0.85, notebook 24 band 0.792-0.908):
isoform                 CYP1A2  CYP2C9  CYP2D6  CYP3A4
arm                                                   
aid                      0.897   0.874   0.924   0.842
deadzone                 0.950   0.932   0.935   0.912
ecfp4_narrow__lightgbm   0.872   0.784   0.885   0.769
ecfp4_narrow__rf         0.872   0.796   0.877   0.749
ecfp4_narrow__xgboost    0.849   0.780   0.865   0.746
mordred_pca__lightgbm    0.845 

In [161]:
# ---- The plain verdict, per arm, in words. Stated for every arm, not only the interesting ones.
print("=== DECORRELATION VERDICT, per arm ===")
verdict_lines = []
for arm in [a for a in ARMS_PRESENT if a != "plain"]:
    sub = decorr[decorr["arm"] == arm]
    if not len(sub):
        continue
    lo, hi = sub["corr_vs_plain"].min(), sub["corr_vs_plain"].max()
    n_below_bar = int(sub["below_bar_0.85"].sum())
    n_below_floor = int(sub["below_floor"].fillna(False).sum())
    n_below_band = int(sub["below_nb24_band"].sum())
    # Two independent references, reported separately rather than collapsed into one label:
    # the fixed 0.85 bar the brief names, and PLAIN's own across-repeat re-seed floor (the
    # correlation two runs of the SAME recipe produce). A floor of NaN means fewer than two
    # repeats completed, in which case only the bar can be judged and that is said explicitly.
    floor_known = bool(sub["plain_floor"].notna().all())
    detail = (f"below the {DECORR_BAR} bar on {n_below_bar}/{len(sub)} isoforms; below PLAIN's own "
              f"re-seed floor on {n_below_floor}/{len(sub)}"
              if floor_known else
              f"below the {DECORR_BAR} bar on {n_below_bar}/{len(sub)} isoforms; PLAIN's re-seed "
              f"floor is unavailable (fewer than two repeats), so only the bar is judged")
    if n_below_bar == len(sub) and (not floor_known or n_below_floor == len(sub)):
        v = f"DECORRELATED from PLAIN on every isoform -- {detail}"
    elif n_below_bar == 0:
        v = f"NOT decorrelated, pool-typical or above on every isoform -- {detail}"
    else:
        v = f"PARTIALLY decorrelated -- {detail}"
    line = (f"  {arm:24s} corr vs PLAIN {lo:.3f}-{hi:.3f}  "
            f"below {DECORR_BAR}: {n_below_bar}/{len(sub)}  below own floor: {n_below_floor}/{len(sub)}  "
            f"below nb24 band: {n_below_band}/{len(sub)}\n    -> {v}")
    print(line)
    verdict_lines.append({"arm": arm, "corr_min": float(lo), "corr_max": float(hi),
                          "n_below_bar": n_below_bar, "n_isoforms": len(sub),
                          "n_below_floor": n_below_floor, "n_below_nb24_band": n_below_band,
                          "verdict": v})
decorr_verdict = pd.DataFrame(verdict_lines)
decorr_verdict.to_csv(OUT / "decorrelation_verdict.csv", index=False)

MATERIALLY_DECORRELATED = sorted(decorr_verdict.loc[
    decorr_verdict["n_below_bar"] >= 1, "arm"].tolist()) if len(decorr_verdict) else []
TREE_DECORRELATED = [a for a in MATERIALLY_DECORRELATED if a in TREE_ARMS]
print(f"\narms below the {DECORR_BAR} bar on at least one isoform: "
      f"{MATERIALLY_DECORRELATED if MATERIALLY_DECORRELATED else 'NONE'}")
if TREE_DECORRELATED:
    print(f"\nTREES below the bar on at least one isoform ({TREE_DECORRELATED}) -- on the brief's")
    print("own stated reading, this project has its first genuine ensemble candidate. The blend in")
    print("4.4 therefore runs, with its nested honesty check.")
else:
    print("\nNo tree arm falls below the bar on any isoform. On the brief's own stated reading this")
    print("is equally informative: the shared ignorance is a property of the COMPOUNDS rather than")
    print("of the architecture, and the ensemble question closes on evidence rather than on cost.")

=== DECORRELATION VERDICT, per arm ===
  ecfp4_narrow__rf         corr vs PLAIN 0.749-0.877  below 0.85: 2/4  below own floor: 4/4  below nb24 band: 1/4
    -> PARTIALLY decorrelated -- below the 0.85 bar on 2/4 isoforms; below PLAIN's own re-seed floor on 4/4
  ecfp4_narrow__xgboost    corr vs PLAIN 0.746-0.865  below 0.85: 3/4  below own floor: 4/4  below nb24 band: 2/4
    -> PARTIALLY decorrelated -- below the 0.85 bar on 3/4 isoforms; below PLAIN's own re-seed floor on 4/4
  ecfp4_narrow__lightgbm   corr vs PLAIN 0.769-0.885  below 0.85: 2/4  below own floor: 4/4  below nb24 band: 2/4
    -> PARTIALLY decorrelated -- below the 0.85 bar on 2/4 isoforms; below PLAIN's own re-seed floor on 4/4
  mordred_pca__rf          corr vs PLAIN 0.718-0.881  below 0.85: 3/4  below own floor: 4/4  below nb24 band: 2/4
    -> PARTIALLY decorrelated -- below the 0.85 bar on 3/4 isoforms; below PLAIN's own re-seed floor on 4/4
  mordred_pca__xgboost     corr vs PLAIN 0.696-0.842  below 0.85: 4/4  be

### 4.4 Blend — run only if at least one arm is materially decorrelated from PLAIN

Per-isoform weights by **greedy selection with replacement in z-space** (Caruana et al. 2004's
selection mechanic, applied here to this partition's own out-of-fold predictions). The in-sample
figure is reported **together with** a nested honesty check: per-fold greedy selection scored on the
held-out fold. Without the nested number a per-isoform blend is just another selection that cannot
be defended, so if the nested check cannot be computed the blend is reported as **unusable** rather
than reported at its in-sample figure alone.

In [162]:
def strae(y_true, y_pred, lo, hi):
    return float(rae_soft_threshold_absolute_error(np.asarray(y_true, float), np.asarray(y_pred, float),
                                                  y_true_upper=np.asarray(hi, float),
                                                  y_true_lower=np.asarray(lo, float)))


# TWO PARAMETERISATIONS, and the reason both are reported rather than one.
#
# The brief specifies weights fitted "in z-space". Taken literally -- standardise each arm,
# average the selected z columns, then map back onto y_true's own mean and standard deviation --
# that map-back forces the blend's spread to equal the LABEL spread. This project has already
# measured, twice and independently, that matching label spread is not ST-RAE-optimal:
# notebook 26's spread sweep found the ST-RAE-optimal multiplier strictly between rho and 1.0 on
# every isoform, and notebook 27 Section 1 found board ST-RAE associated with spread ratio at
# Spearman -0.822. So the z-space parameterisation bakes in a variance expansion that ST-RAE
# penalises, and its in-sample score can come out WORSE than the best single arm -- which a
# greedy search initialised at the best single arm cannot do on its own objective. That is a
# property of the parameterisation, not a bug in the search, and it was caught by exactly that
# inconsistency during a dry run of this analysis.
#
# `space="raw"` is therefore reported as the primary blend: greedy selection with replacement on
# a weighted mean of the raw predictions, which is on the same scale as every single-arm score
# and is guaranteed not to be worse than the best single arm in sample. `space="z"` is reported
# beside it, as specified, so the cost of the parameterisation is visible rather than hidden.
def greedy_blend(df, arm_cols, space="raw", max_iter=50):
    y = df["y_true"].to_numpy(float)
    lo, hi = df["conf_low"].to_numpy(float), df["conf_high"].to_numpy(float)
    ymu, ysd = y.mean(), y.std(ddof=0)
    if space == "raw":
        P = {a: df[a].to_numpy(float) for a in arm_cols}
        back = lambda v: v
    elif space == "z":
        P = {}
        for a in arm_cols:
            v = df[a].to_numpy(float)
            sd = v.std(ddof=0)
            P[a] = (v - v.mean()) / sd if sd > 0 else np.zeros_like(v)
        back = lambda z: z * ysd + ymu
    else:
        raise ValueError(f"unknown space: {space}")

    def score_of(counts):
        tot = sum(counts.values())
        v = sum(P[a] * n for a, n in counts.items()) / tot
        return strae(y, back(v), lo, hi)

    best_single = min(arm_cols, key=lambda a: score_of({a: 1}))
    counts = {best_single: 1}
    best = score_of(counts)
    for _ in range(max_iter - 1):
        cand_best, cand_score = None, best
        for a in arm_cols:
            trial = dict(counts)
            trial[a] = trial.get(a, 0) + 1
            sc = score_of(trial)
            if sc < cand_score - 1e-12:
                cand_best, cand_score = a, sc
        if cand_best is None:
            break
        counts[cand_best] = counts.get(cand_best, 0) + 1
        best = cand_score
    tot = sum(counts.values())
    return {a: n / tot for a, n in counts.items()}, best, best_single


def apply_blend(fit_df, pred_df, weights, space="raw"):
    # Every statistic used to transform the held-out fold comes from the FITTING set only.
    if space == "raw":
        out = np.zeros(len(pred_df))
        for a, wt in weights.items():
            out = out + wt * pred_df[a].to_numpy(float)
        return out
    yf = fit_df["y_true"].to_numpy(float)
    ymu, ysd = yf.mean(), yf.std(ddof=0)
    z = np.zeros(len(pred_df))
    for a, wt in weights.items():
        vf = fit_df[a].to_numpy(float)
        sd = vf.std(ddof=0)
        zz = (pred_df[a].to_numpy(float) - vf.mean()) / sd if sd > 0 else np.zeros(len(pred_df))
        z = z + wt * zz
    return z * ysd + ymu

In [163]:
BLEND_RAN = False
blend_rows = []
if not MATERIALLY_DECORRELATED:
    print("BLEND NOT RUN -- no arm is materially decorrelated from PLAIN on any isoform, which is")
    print("this section's own stated precondition. Fitting blend weights over a pool whose members")
    print("all make the same errors would produce a number with no mechanism behind it.")
else:
    BLEND_RAN = True
    print(f"BLEND RUNS -- materially decorrelated arms present: {MATERIALLY_DECORRELATED}")
    print("")
    for iso in ISOFORMS:
        w = OOF_WIDE[iso]
        arm_cols = [c for c in ARMS_PRESENT if c in w.columns and w[c].notna().all()]
        if len(arm_cols) < 2:
            print(f"{iso}: fewer than 2 arms with complete coverage -- skipped")
            continue
        blend_units = sorted(set(map(tuple, w[["repeat", "fold"]].values.tolist())))
        row = {"isoform": iso, "n_arms": len(arm_cols), "n_units": len(blend_units)}
        for space in ["raw", "z"]:
            weights, in_score, init_arm = greedy_blend(w, arm_cols, space=space)
            nested_pred, nested_true, nested_lo, nested_hi = [], [], [], []
            for (r, f) in blend_units:
                held = (w["repeat"] == r) & (w["fold"] == f)
                fit_df, pred_df = w[~held], w[held]
                if len(fit_df) < 50 or not len(pred_df):
                    continue
                wts, _, _ = greedy_blend(fit_df, arm_cols, space=space)
                nested_pred.append(apply_blend(fit_df, pred_df, wts, space=space))
                nested_true.append(pred_df["y_true"].to_numpy(float))
                nested_lo.append(pred_df["conf_low"].to_numpy(float))
                nested_hi.append(pred_df["conf_high"].to_numpy(float))
            if nested_pred:
                nested = strae(np.concatenate(nested_true), np.concatenate(nested_pred),
                               np.concatenate(nested_lo), np.concatenate(nested_hi))
                usable = True
            else:
                nested, usable = np.nan, False
            row[f"{space}_in_sample"] = in_score
            row[f"{space}_nested"] = nested
            row[f"{space}_usable"] = usable
            row[f"{space}_n_nested_folds"] = len(nested_pred)
            row[f"{space}_init_arm"] = init_arm
            row[f"{space}_weights"] = json.dumps({k: round(v, 4) for k, v in
                                                  sorted(weights.items(), key=lambda kv: -kv[1])})
        best_arm = min(arm_cols, key=lambda a: strae(w["y_true"], w[a], w["conf_low"], w["conf_high"]))
        row["best_single_arm"] = best_arm
        row["best_single_strae"] = strae(w["y_true"], w[best_arm], w["conf_low"], w["conf_high"])
        row["plain_strae"] = (strae(w["y_true"], w["plain"], w["conf_low"], w["conf_high"])
                              if "plain" in arm_cols else np.nan)
        blend_rows.append(row)
        print(f"{iso}  ({len(blend_units)} folds, {len(arm_cols)} arms)")
        print(f"   PLAIN {row['plain_strae']:.4f} | best single ({best_arm}) {row['best_single_strae']:.4f}")
        print(f"   raw-space: in-sample {row['raw_in_sample']:.4f} -> NESTED {row['raw_nested']:.4f}")
        print(f"     weights {row['raw_weights']}")
        print(f"   z-space:   in-sample {row['z_in_sample']:.4f} -> NESTED {row['z_nested']:.4f}")
        print(f"     weights {row['z_weights']}")

if blend_rows:
    blend = pd.DataFrame(blend_rows)
    blend.to_csv(OUT / "blend_results.csv", index=False)
    print("")
    print("The NESTED figure is the only defensible one. The in-sample figure sits beside it so the")
    print("gap between them -- the optimism a per-isoform selection buys itself -- stays visible.")
    print("")
    for _, r_ in blend.iterrows():
        # Sanity property of the search, asserted rather than assumed: greedy selection
        # initialised at the best single arm cannot be worse than it in sample, on its own
        # objective. Raw space shares ST-RAE's scale so this must hold there.
        if r_["raw_in_sample"] > r_["best_single_strae"] + 1e-9:
            print(f"  {r_['isoform']}: WARNING -- raw-space in-sample ({r_['raw_in_sample']:.4f}) "
                  f"exceeds the best single arm ({r_['best_single_strae']:.4f}); the search or the "
                  f"objective is inconsistent. Investigate before quoting this row.")
        for space in ["raw", "z"]:
            if not r_[f"{space}_usable"]:
                print(f"  {r_['isoform']} ({space}): nested check could not be computed -- UNUSABLE.")
                continue
            beats = r_[f"{space}_nested"] < r_["plain_strae"]
            print(f"  {r_['isoform']} ({space}): nested blend {'BEATS' if beats else 'does NOT beat'} "
                  f"PLAIN ({r_[f'{space}_nested']:.4f} vs {r_['plain_strae']:.4f}); optimism "
                  f"{r_[f'{space}_nested'] - r_[f'{space}_in_sample']:+.4f}")
    print("")
    z_worse = int((blend["z_in_sample"] > blend["best_single_strae"] + 1e-9).sum())
    print(f"z-space in-sample worse than the best single arm on {z_worse} of {len(blend)} isoforms.")
    if z_worse:
        print("This is the parameterisation cost described above, not a search failure: mapping the")
        print("blended z back onto the LABEL spread expands variance, and ST-RAE penalises that --")
        print("consistent with notebook 26's spread sweep and notebook 27 Section 1's own findings.")
        print("The raw-space blend is the one to read.")

BLEND RUNS -- materially decorrelated arms present: ['aid', 'ecfp4_narrow__lightgbm', 'ecfp4_narrow__rf', 'ecfp4_narrow__xgboost', 'mordred_pca__lightgbm', 'mordred_pca__rf', 'mordred_pca__xgboost']



CYP1A2  (10 folds, 10 arms)
   PLAIN 0.8501 | best single (deadzone) 0.7680
   raw-space: in-sample 0.7374 -> NESTED 0.7473
     weights {"deadzone": 0.5, "aid": 0.3333, "ecfp4_narrow__xgboost": 0.1667}
   z-space:   in-sample 0.8879 -> NESTED 0.9003
     weights {"deadzone": 0.3333, "aid": 0.3333, "ecfp4_narrow__xgboost": 0.1667, "mordred_pca__xgboost": 0.1667}
CYP2C9  (10 folds, 10 arms)
   PLAIN 0.6401 | best single (deadzone) 0.5596
   raw-space: in-sample 0.5456 -> NESTED 0.5484
     weights {"deadzone": 0.75, "aid": 0.25}
   z-space:   in-sample 0.6261 -> NESTED 0.6329
     weights {"deadzone": 0.4, "aid": 0.3, "ecfp4_narrow__rf": 0.1, "mordred_pca__xgboost": 0.1, "ecfp4_narrow__xgboost": 0.1}


CYP2D6  (10 folds, 10 arms)
   PLAIN 1.0093 | best single (deadzone) 0.8779
   raw-space: in-sample 0.8570 -> NESTED 0.8648
     weights {"deadzone": 0.5, "aid": 0.25, "ecfp4_narrow__lightgbm": 0.125, "mordred_pca__lightgbm": 0.125}
   z-space:   in-sample 1.0868 -> NESTED 1.0944
     weights {"deadzone": 0.2857, "ecfp4_narrow__xgboost": 0.1429, "mordred_pca__lightgbm": 0.1429, "aid": 0.1429, "mordred_pca__xgboost": 0.1429, "ecfp4_narrow__lightgbm": 0.1429}


CYP3A4  (10 folds, 10 arms)
   PLAIN 0.5367 | best single (deadzone) 0.4725
   raw-space: in-sample 0.4532 -> NESTED 0.4620
     weights {"deadzone": 0.6, "aid": 0.2, "ecfp4_narrow__lightgbm": 0.2}
   z-space:   in-sample 0.5008 -> NESTED 0.5017
     weights {"deadzone": 0.5, "aid": 0.25, "ecfp4_narrow__rf": 0.25}

The NESTED figure is the only defensible one. The in-sample figure sits beside it so the
gap between them -- the optimism a per-isoform selection buys itself -- stays visible.

  CYP1A2 (raw): nested blend BEATS PLAIN (0.7473 vs 0.8501); optimism +0.0099
  CYP1A2 (z): nested blend does NOT beat PLAIN (0.9003 vs 0.8501); optimism +0.0124
  CYP2C9 (raw): nested blend BEATS PLAIN (0.5484 vs 0.6401); optimism +0.0027
  CYP2C9 (z): nested blend BEATS PLAIN (0.6329 vs 0.6401); optimism +0.0068
  CYP2D6 (raw): nested blend BEATS PLAIN (0.8648 vs 1.0093); optimism +0.0078
  CYP2D6 (z): nested blend does NOT beat PLAIN (1.0944 vs 1.0093); optimism +0.0077
  CYP3A4 (raw): nested blend

## Part 5 — Figures

House style, matching notebooks 21/24/25/26/27/29/31/33: `figsize=(10,6)`, `dpi=150`,
`bbox_inches="tight"`, base font size 10, top and right spines removed, legends without frames,
titles 11pt bold and left-aligned **stating the finding**, axis labels carrying units, `#8C8C8C`
grey for the reference series and `#B0413E` for the series carrying the argument. Each figure's
underlying table is written to CSV alongside it.

In [164]:
plt.rcParams.update({"font.size": 10, "figure.dpi": 110, "savefig.dpi": 150,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
                     "legend.frameon": False})
GREY, ACCENT = "#8C8C8C", "#B0413E"
MID = "#4C4C4C"


def finish(fig, path):
    fig.savefig(path, bbox_inches="tight", dpi=150)
    plt.close(fig)
    print(f"wrote {path}")

In [165]:
# ---- FIGURE 1: out-of-fold ST-RAE by arm and isoform, with across-fold spread.
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=False)
arm_plot_order = (arm_summary.groupby("arm")["ST-RAE_mean"].mean().sort_values(ascending=False).index.tolist())
for ax, iso in zip(axes.ravel(), ISOFORMS):
    sub = arm_summary[arm_summary["isoform"] == iso].set_index("arm").reindex(arm_plot_order).dropna(subset=["ST-RAE_mean"])
    y = np.arange(len(sub))
    colors = [ACCENT if a == "plain" else GREY for a in sub.index]
    ax.barh(y, sub["ST-RAE_mean"], xerr=sub["ST-RAE_std"].fillna(0), color=colors,
            error_kw={"ecolor": MID, "elinewidth": 1, "capsize": 2}, height=0.72)
    ax.set_yticks(y)
    ax.set_yticklabels(sub.index, fontsize=7.5)
    ax.axvline(1.0, color=MID, lw=0.8, ls=":")
    plain_v = arm_summary[(arm_summary["isoform"] == iso) & (arm_summary["arm"] == "plain")]["ST-RAE_mean"]
    if len(plain_v):
        ax.axvline(float(plain_v.iloc[0]), color=ACCENT, lw=0.9, ls="--", alpha=0.7)
    n = int(sub["n_folds"].max())
    ax.set_title(f"{iso} (n={n} folds)", fontsize=10)
    ax.set_xlabel("ST-RAE, lower is better")
fig.suptitle("Cluster-disjoint out-of-fold ST-RAE: no arm separates from PLAIN by more than its own fold spread"
             if (paired[(paired.metric == 'ST-RAE') & (paired.verdict != 'NOT DIFFERENT')].empty)
             else "Cluster-disjoint out-of-fold ST-RAE by arm (dashed red = PLAIN; dotted = naive-mean parity)",
             fontsize=11, fontweight="bold", x=0.01, ha="left")
fig.tight_layout(rect=[0, 0, 1, 0.94])
finish(fig, FIG_OUT / "strae_by_arm_isoform.png")
arm_summary.to_csv(OUT / "figure1_strae_by_arm_isoform.csv", index=False)

wrote /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/34_butina_5x5/figures/strae_by_arm_isoform.png


**Figure 1 — `strae_by_arm_isoform.png`.** Out-of-fold ST-RAE for every arm on the repeated
cluster-disjoint partition, one panel per isoform, bars ordered worst-to-best by cross-isoform mean
and error bars showing the across-fold standard deviation. PLAIN, the reference every other arm is
compared against, is in red with its level marked as a dashed line; the dotted line at 1.0 is parity
with a constant mean predictor. Panel titles carry the number of completed folds behind each
estimate, so the reader can see directly how much the spread is worth. Read against Figure 2: this
panel says where each arm *sits*, while Figure 2 says whether that position survives a change of
split scheme.

In [166]:
# ---- FIGURE 2: each arm's difference against PLAIN under BOTH partitions, paired.
# The figure that answers 4.2 -- a sign flip is visible as a pair straddling zero.
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True)
plot_arms = [a for a in ALL_ARMS if a != "plain" and a in ARMS_PRESENT]
for ax, iso in zip(axes.ravel(), ISOFORMS):
    sub = agree[agree["isoform"] == iso].set_index("arm").reindex(plot_arms)
    y = np.arange(len(sub))
    for yi, (arm, row) in zip(y, sub.iterrows()):
        bd, rd, st = row["butina_diff"], row["random_diff"], row.get("status")
        if pd.notna(bd) and pd.notna(rd):
            flip = st == "FLIP"
            unres = st == "sign not resolved"
            col = ACCENT if flip else GREY
            ax.plot([rd, bd], [yi, yi], color=col, lw=1.6 if flip else 1.0,
                    alpha=0.9 if flip else (0.3 if unres else 0.6),
                    ls=":" if unres else "-", zorder=1)
            ax.scatter([rd], [yi], marker="o", s=26, facecolor="white", edgecolor=GREY,
                       linewidth=1.2, alpha=0.4 if unres else 1.0, zorder=3)
            ax.scatter([bd], [yi], marker="s", s=28, color=(ACCENT if flip else MID),
                       alpha=0.4 if unres else 1.0, zorder=3)
        elif pd.notna(bd):
            ax.scatter([bd], [yi], marker="s", s=28, facecolor="white", edgecolor=MID,
                       linewidth=1.0, zorder=3)
    ax.axvline(0.0, color="black", lw=0.9)
    ax.set_yticks(y)
    ax.set_yticklabels(sub.index, fontsize=7.5)
    ax.invert_yaxis()
    n_flip = int((sub["status"] == "FLIP").sum())
    n_res = int((sub["status"].isin(["FLIP", "agrees"])).sum())
    n_unres = int((sub["status"] == "sign not resolved").sum())
    ax.set_title(f"{iso} -- {n_flip} of {n_res} sign-resolved arms flip ({n_unres} unresolved)",
                 fontsize=9.5)
for ax in axes[1]:
    ax.set_xlabel("ST-RAE difference against PLAIN (negative = better than PLAIN)")
h = [plt.Line2D([], [], marker="o", ls="", markerfacecolor="white", markeredgecolor=GREY, label="random partition"),
     plt.Line2D([], [], marker="s", ls="", color=MID, label="cluster-disjoint (this notebook)"),
     plt.Line2D([], [], color=ACCENT, lw=1.6, label="sign flips between partitions"),
     plt.Line2D([], [], color=GREY, lw=1.0, ls=":", alpha=0.4, label="sign not resolved (|diff| < its SE)"),
     plt.Line2D([], [], marker="s", ls="", markerfacecolor="white", markeredgecolor=MID,
                label="no random-partition reference")]
fig.suptitle("Does the split scheme change the conclusion? Paired arm-vs-PLAIN differences under both partitions",
             fontsize=11, fontweight="bold", x=0.01, ha="left")
fig.tight_layout(rect=[0, 0.07, 1, 0.94])
# Legend below the panels -- inside any panel it overlapped a real data marker.
fig.legend(handles=h, loc="lower center", ncol=3, fontsize=7.5, bbox_to_anchor=(0.5, 0.0))
finish(fig, FIG_OUT / "partition_agreement.png")
agree.to_csv(OUT / "figure2_partition_agreement.csv", index=False)

wrote /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/34_butina_5x5/figures/partition_agreement.png


**Figure 2 — `partition_agreement.png`.** For each isoform, every arm's out-of-fold ST-RAE
difference against PLAIN measured twice: once on the frozen random partition (open grey circle,
read from notebooks 05/29/31) and once on this notebook's repeated cluster-disjoint partition
(filled square), the two joined so the reader sees the pair rather than two separate numbers. The
vertical line at zero is PLAIN; anything to its left beat PLAIN on that partition. **A pair that
straddles the zero line is a sign flip** — the two partitions disagree about whether the arm helps
at all — and those pairs are drawn in red with a heavier connector. SINGLE has no random-partition
counterpart anywhere in this project and is drawn as a single square with that stated on the figure
rather than silently compared against something else. This is the figure that answers Part 4.2.

In [167]:
# ---- FIGURE 3: pairwise arm error-correlation matrix per isoform, with notebook 24's band
# marked on the colourbar and the PLAIN across-repeat floor annotated in each panel.
# Colour range starts at 0.6, not 0: every value in this matrix lies between ~0.65 and 1.0, and
# a 0-1 scale renders the whole thing one shade of red, hiding exactly the differences the
# figure exists to show. Stated on the colourbar so the truncation is not silent.
fig, axes = plt.subplots(2, 2, figsize=(10, 6.6))
VMIN, VMAX = 0.6, 1.0
im = None
for k, (ax, iso) in enumerate(zip(axes.ravel(), ISOFORMS)):
    M = corr_mats[iso]
    cols = [a for a in ALL_ARMS if a in M.columns]
    Mo = M.loc[cols, cols].astype(float)
    im = ax.imshow(Mo.to_numpy(), vmin=VMIN, vmax=VMAX, cmap="RdYlBu_r", aspect="auto")
    ax.set_xticks(range(len(cols)))
    # Only the bottom row carries x tick labels -- on the top row they collided with the
    # bottom row's panel titles.
    if k >= 2:
        ax.set_xticklabels(cols, rotation=90, fontsize=5.5)
    else:
        ax.set_xticklabels([])
    ax.set_yticks(range(len(cols)))
    ax.set_yticklabels(cols, fontsize=5.5)
    fl = floors[floors["isoform"] == iso]
    flv = float(fl["plain_vs_plain_mean"].iloc[0]) if len(fl) else np.nan
    ax.set_title(f"{iso} -- PLAIN across-repeat floor {flv:.3f}" if not np.isnan(flv)
                 else f"{iso} -- floor unavailable (one repeat)", fontsize=9.5)
    if "plain" in cols:
        j = cols.index("plain")
        ax.axvline(j, color=ACCENT, lw=1.4, alpha=0.95)
        ax.axhline(j, color=ACCENT, lw=1.4, alpha=0.95)
fig.tight_layout(rect=[0, 0, 0.88, 0.93])
cax = fig.add_axes([0.90, 0.12, 0.022, 0.72])
cb = fig.colorbar(im, cax=cax)
# Band edges become real ticks rather than free-floating text over the colourbar label.
ticks = sorted({0.6, 0.7, 0.8, 0.9, 1.0, round(NB24_BAND_LO, 3), round(NB24_BAND_HI, 3)})
cb.set_ticks(ticks)
cb.set_ticklabels([(f"{t:.2f}  nb24" if abs(t - NB24_BAND_LO) < 1e-6 or abs(t - NB24_BAND_HI) < 1e-6
                    else f"{t:.2f}") for t in ticks])
cb.ax.tick_params(labelsize=7)
for v in [NB24_BAND_LO, NB24_BAND_HI]:
    cb.ax.axhline(v, color="black", lw=1.4)
cb.set_label("error correlation (residuals, labelled rows only); scale truncated at 0.6",
             fontsize=7.5, labelpad=6)
fig.suptitle("Error correlation between arms: the shared-ignorance question, with notebook 24's pool band marked",
             fontsize=11, fontweight="bold", x=0.01, ha="left")
finish(fig, FIG_OUT / "error_correlation_matrix.png")
pd.concat([corr_mats[iso].assign(isoform=iso) for iso in ISOFORMS]).to_csv(
    OUT / "figure3_error_correlation_matrix.csv")

wrote /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/34_butina_5x5/figures/error_correlation_matrix.png


**Figure 3 — `error_correlation_matrix.png`.** Pairwise correlation of **residuals** (prediction
minus truth, labelled rows only) between every pair of arms, one panel per isoform, pooled across
all completed folds. The two black lines on the shared colourbar mark notebook 24's measured band
for this project's existing 11-config pool (0.79–0.91), so a pair can be read immediately as
"pool-typical" or "unusually decorrelated". The red cross-hairs pick out PLAIN's row and column —
the comparison that decides whether an arm is an ensemble candidate — and each panel title carries
that isoform's PLAIN-against-PLAIN across-repeat floor, the correlation two runs of the *same*
recipe produce under different cluster-to-fold assignments and seeds. An arm at or above its own
panel's floor is not decorrelated in any useful sense, whatever its absolute value. This is the
figure that answers Part 4.3.

### Cost actually incurred, and what it says about the timing gate

The gate is applied exactly as specified — mechanically, after repeat 0's PLAIN and AID *first*
folds complete. This cell reports what the arms eventually cost across every completed fold, and
checks the gate's decision against those realised means. It is a self-criticism of the rule, not a
re-decision: nothing below changes `N_REPEATS_ACTIVE`, which stayed at whatever the gate set
before any of this was knowable.

In [168]:
tm = pd.read_csv(TIMINGS_PATH)
real = tm[tm["wall_min"] > 0.5]
per_kind_real = real.groupby("arm_kind")["wall_min"].agg(["mean", "min", "max", "size"])
print("REALISED cost per (repeat, fold) unit, across every completed fold:")
print(per_kind_real.round(2).to_string())
print(f"\ntotal real training wall time: {real['wall_min'].sum()/60:.2f} h "
      f"over {len(real)} units")

print("\nper-epoch time, min to max within each arm (notebook 28 saw 10.6x degradation on an")
print("unchanged workload; the flag in this notebook's harness is 3x):")
mpe = real[real["min_per_epoch"].notna()].groupby("arm_kind")["min_per_epoch"].agg(["min", "max"])
mpe["ratio"] = mpe["max"] / mpe["min"]
print(mpe.round(3).to_string())
print(f"worst within-arm per-epoch ratio: {mpe['ratio'].max():.2f}x -- "
      f"{'no degradation' if mpe['ratio'].max() < 3 else 'DEGRADATION FLAGGED'}")

# Did the gate's fold-0 projection match what the run actually cost?
if DESIGN:
    gate_est = DESIGN["measured_min_per_fold_by_arm"]
    print("\ngate-time estimate vs. realised mean, per arm (min per fold):")
    rows = []
    for k in ["plain", "trees", "single", "aid", "deadzone"]:
        est = float(gate_est.get(k, np.nan))
        got = float(per_kind_real.loc[k, "mean"]) if k in per_kind_real.index else np.nan
        rows.append({"arm": k, "gate_estimate": est, "realised_mean": got,
                     "ratio": est / got if got else np.nan})
    gate_cmp = pd.DataFrame(rows)
    print(gate_cmp.round(2).to_string(index=False))
    gate_cmp.to_csv(OUT / "gate_estimate_vs_realised.csv", index=False)

    realised_per_fold = float(per_kind_real["mean"].sum())
    realised_5 = 5 * N_FOLDS * realised_per_fold / 60.0
    band_now = ("<= 20 h -> 5 repeats" if realised_5 <= 20 else
                "20-30 h -> 3 repeats" if realised_5 <= 30 else
                "> 30 h -> 2 repeats")
    print(f"\ngate's projected five-repeat total: {DESIGN['projected_5repeat_h']:.2f} h "
          f"-> {DESIGN['n_repeats_active']} repeat(s) ({DESIGN['band']})")
    print(f"same projection from REALISED per-fold means: {realised_5:.2f} h -> {band_now}")
    if band_now.endswith(f"{DESIGN['n_repeats_active']} repeats"):
        print("The gate's decision matches what the realised costs would have chosen.")
    else:
        aid_gate = float(gate_est.get("aid", np.nan))
        aid_real = float(per_kind_real.loc["aid", "mean"])
        print("")
        print("*** THE GATE'S DECISION DOES NOT MATCH WHAT THE REALISED COSTS WOULD HAVE CHOSEN.")
        print(f"    Reported plainly rather than buried: the reduction rested on AID's FOLD-0 run")
        print(f"    ({aid_gate:.2f} min), which turned out to be that arm's slowest fold -- its")
        print(f"    eventual 10-fold mean is {aid_real:.2f} min ({aid_gate/aid_real:.2f}x faster than")
        print(f"    fold 0 suggested), and essentially notebook 29's own 34.02 min/fold figure.")
        print(f"    The projection landed at {DESIGN['projected_5repeat_h']:.2f} h against a 30 h")
        print(f"    threshold -- {100*(DESIGN['projected_5repeat_h']/30 - 1):.1f}% over -- so one")
        print(f"    unrepresentative fold cost this design a repeat.")
        print("    The rule was applied correctly and is NOT being re-decided after the fact; what")
        print("    this shows is that a gate projecting from a single fold of the most expensive arm")
        print("    is fragile, and a future run should project from at least two or three folds of")
        print("    that arm before committing. The consequence for THIS notebook is a 2x5 design")
        print("    where a 3x5 was affordable -- stated as a real limitation of the run.")
        record_decision(
            decision=("Left the timing gate's 2-repeat reduction in place although the realised "
                      "per-fold costs would have supported 3 repeats"),
            reason=(f"The gate fired mechanically after repeat 0's AID fold 0, per the brief, and that "
                    f"fold ({aid_gate:.2f} min) proved to be the AID arm's slowest of ten; its eventual "
                    f"mean was {aid_real:.2f} min. The five-repeat projection came to "
                    f"{DESIGN['projected_5repeat_h']:.2f} h against the brief's 30 h threshold, so the "
                    f"rule selected 2 repeats where the realised means give {realised_5:.2f} h and would "
                    f"have selected 3."),
            alternatives=("Re-run the gate on the realised means and extend to a third repeat (would be "
                          "re-deciding a pre-specified rule after seeing its outcome, which is the "
                          "selection behaviour this project's CLAUDE.md explicitly warns against); "
                          "restart the whole design under a revised gate (cost already sunk)."),
            authority="autonomy rule: record, do not improvise around, a consequence of a pre-specified rule")

REALISED cost per (repeat, fold) unit, across every completed fold:
           mean    min    max  size
arm_kind                           
aid       34.82  28.61  47.06    10
deadzone   9.70   7.54  12.63    10
plain      7.23   5.59  15.56    10
single    11.35   9.48  13.49    10
trees      1.15   1.11   1.19    10

total real training wall time: 10.71 h over 50 units

per-epoch time, min to max within each arm (notebook 28 saw 10.6x degradation on an
unchanged workload; the flag in this notebook's harness is 3x):
            min    max  ratio
arm_kind                     
aid       3.888  4.144  1.066
deadzone  0.636  0.754  1.186
plain     0.635  0.801  1.261
single    0.474  0.813  1.715
worst within-arm per-epoch ratio: 1.72x -- no degradation

gate-time estimate vs. realised mean, per arm (min per fold):
     arm  gate_estimate  realised_mean  ratio
   plain           8.54           7.23   1.18
   trees           1.15           1.15   1.00
  single          11.16          11.35

## Part 6 — Verdict

In [169]:
print("=" * 78)
print("NOTEBOOK 34 -- VERDICT")
print("=" * 78)
print(f"\nDESIGN ACTUALLY RUN: {N_REPEATS_ACTIVE} repeat(s) x {N_FOLDS} cluster-disjoint folds")
if DESIGN:
    print(f"  timing gate: {DESIGN.get('band')}  (projected 5-repeat total "
          f"{DESIGN.get('projected_5repeat_h', float('nan')):.1f} h)")
_units_done = pd.read_csv(RUN_SCORES_PATH)[["repeat", "fold", "arm"]].drop_duplicates()
print(f"  completed (repeat, fold, arm) units: {len(_units_done)} "
      f"({_units_done['arm'].nunique()} arms x {len(_units_done[['repeat','fold']].drop_duplicates())} folds)")
print(f"  arms with data: {ARMS_PRESENT}")
if ABORT["aborted"]:
    print(f"  *** RUN ABORTED at {ABORT['where']}: {ABORT['reason']}")
if DEADLINE_SKIPS:
    print(f"  *** {len(DEADLINE_SKIPS)} units skipped by the {WALL_DEADLINE_H}h wall guard")
print(f"  elapsed: {(time.time() - RUN_STARTED_AT)/3600:.2f} h since campaign start")

print("\n--- 4.1 MODEL COMPARISON (ST-RAE, BH-corrected within isoform) ---")
st = paired[paired["metric"] == "ST-RAE"]
for iso in ISOFORMS:
    sub = st[st["isoform"] == iso]
    better = sub[sub["verdict"] == "SIGNIFICANTLY BETTER"]["arm"].tolist()
    worse = sub[sub["verdict"] == "SIGNIFICANTLY WORSE"]["arm"].tolist()
    print(f"  {iso}: better than PLAIN {better if better else 'none'}; "
          f"worse {worse if worse else 'none'}; "
          f"not different {len(sub) - len(better) - len(worse)} of {len(sub)} arms")

print("\n--- 4.2 THE SPLIT QUESTION ---")
n_res_all = int((agree["status"] == "agrees").sum() + (agree["status"] == "FLIP").sum())
print(f"  {int((agree['status'] == 'FLIP').sum())} of {n_res_all} sign-RESOLVED (arm, isoform) cells "
      f"flip between partitions;")
print(f"  {int((agree['status'] == 'sign not resolved').sum())} comparable cells had no resolved "
      f"sign on one or both sides (excluded, not counted either way);")
print(f"  {int((agree['status'] == 'no random reference').sum())} have no random-partition reference "
      f"at all (SINGLE).")
aid_rows2 = agree[agree["arm"] == "aid"]
if not len(aid_rows2):
    print("  AID specifically: not enough completed units to form a paired difference -- the")
    print("    comparison notebook 33's flip claim needs is unavailable in this run.")
else:
    aid_flips = int((aid_rows2["status"] == "FLIP").sum())
    aid_res = int(aid_rows2["status"].isin(["FLIP", "agrees"]).sum())
    print(f"  AID specifically: {aid_flips} of {aid_res} sign-resolved isoforms flip under the")
    print(f"    REPEATED design ({len(aid_rows2) - aid_res} unresolved). Notebook 33 reported")
    print(f"    3 of 4 at five folds on one partition -- exact on the 25-fold basis; see the")
    print(f"    reconciliation table in 4.2 for why its own 5-fold slice gives 4 of 4.")
    if aid_res == 0:
        print("    -> INCONCLUSIVE: this design resolved no AID sign, so it neither confirms nor")
        print("       overturns notebook 33.")
    elif aid_flips >= 2:
        print("    -> the repeated design CONFIRMS notebook 33's flip where it can resolve a sign.")
    elif aid_flips == 0:
        print("    -> the repeated design OVERTURNS notebook 33's flip on every resolved isoform.")
    else:
        print("    -> the repeated design PARTLY reproduces notebook 33's flip.")
print("    A flip at screening tier reversing again here is a legitimate outcome, not an error:")
print("    notebook 33's five-fold, one-partition result sits at the same evidentiary tier as")
print("    notebook 28's AID screen, which notebook 29 then overturned.")
print("  Ordering agreement (Spearman, weak by construction):")
for _, r_ in ordering.iterrows():
    print(f"    {r_['isoform']}: rho={r_['spearman_rho']:.3f} (p={r_['p']:.3f}) over "
          f"{int(r_['n_arms_compared'])} arms; butina best={r_['butina_best']}, "
          f"random best={r_['random_best']}")

print("\n--- 4.3 DECORRELATION ---")
for _, r_ in decorr_verdict.iterrows():
    print(f"  {r_['arm']:24s} {r_['corr_min']:.3f}-{r_['corr_max']:.3f}  -> {r_['verdict']}")
if TREE_DECORRELATED:
    print(f"\n  TREES on raw features ARE decorrelated on at least one isoform "
          f"({TREE_DECORRELATED}) -- this project's first genuine ensemble candidate.")
else:
    print("\n  No tree arm on raw fingerprint/descriptor features falls below the 0.85 bar on any")
    print("  isoform. Tree models on raw features are the strongest available test of whether the")
    print("  error correlation this project keeps measuring is a property of the MODELS or of the")
    print("  COMPOUNDS, and they answer it: it is the compounds. The ensemble question closes on")
    print("  evidence rather than on cost.")

print("\n--- 4.4 BLEND ---")
if not BLEND_RAN:
    print("  Not run -- its own stated precondition (a materially decorrelated arm) was not met.")
elif blend_rows:
    print("  Nested (honest) figures, raw-prediction space as primary; z-space as specified beside it.")
    for r_ in blend_rows:
        print(f"  {r_['isoform']}: PLAIN {r_['plain_strae']:.4f} | best single "
              f"({r_['best_single_arm']}) {r_['best_single_strae']:.4f} | "
              f"raw nested {r_['raw_nested']:.4f} | z nested {r_['z_nested']:.4f}")
    beats_raw = [r_["isoform"] for r_ in blend_rows
                 if r_["raw_usable"] and r_["raw_nested"] < r_["plain_strae"]]
    print(f"  Nested raw-space blend beats PLAIN on: {beats_raw if beats_raw else 'no isoform'}.")
    print("  Reported on the nested figure alone -- the in-sample figure is in blend_results.csv")
    print("  beside it so the selection optimism is visible, never quoted on its own.")

print("\n--- SCOPE ---")
print("  No submission built, validated or sent. No full-data retrain. No adoption recommended.")
print("  This notebook produces evidence, not a deployment.")
print("=" * 78)

NOTEBOOK 34 -- VERDICT

DESIGN ACTUALLY RUN: 2 repeat(s) x 5 cluster-disjoint folds
  timing gate: projected over 30 h  (projected 5-repeat total 31.7 h)
  completed (repeat, fold, arm) units: 100 (10 arms x 10 folds)
  arms with data: ['plain', 'ecfp4_narrow__rf', 'ecfp4_narrow__xgboost', 'ecfp4_narrow__lightgbm', 'mordred_pca__rf', 'mordred_pca__xgboost', 'mordred_pca__lightgbm', 'single', 'aid', 'deadzone']
  elapsed: 11.42 h since campaign start

--- 4.1 MODEL COMPARISON (ST-RAE, BH-corrected within isoform) ---
  CYP1A2: better than PLAIN ['deadzone']; worse ['mordred_pca__rf', 'mordred_pca__xgboost', 'mordred_pca__lightgbm']; not different 5 of 9 arms
  CYP2C9: better than PLAIN ['deadzone']; worse ['ecfp4_narrow__rf', 'ecfp4_narrow__xgboost', 'ecfp4_narrow__lightgbm', 'mordred_pca__rf', 'mordred_pca__xgboost', 'mordred_pca__lightgbm', 'single']; not different 1 of 9 arms
  CYP2D6: better than PLAIN ['ecfp4_narrow__lightgbm', 'aid', 'deadzone']; worse ['mordred_pca__xgboost']; no

## Decisions taken without approval

This section exists whether or not it has content, per the autonomy rules this notebook was run
under. Every entry is also written to `outputs/34_butina_5x5/autonomous_decisions.json`.

In [170]:
# Decisions taken during the build itself, recorded here rather than only in prose.
record_decision(
    decision="Removed OMP_NUM_THREADS=1 / KMP_DUPLICATE_LIB_OK=TRUE from the setup cell after the sanity check failed on the first execution, and asserted them unset",
    reason=("The first execution copied notebook 33's setup cell verbatim, including "
            "OMP_NUM_THREADS=1, and the Part 3 sanity check failed at 0.0378 max abs ST-RAE "
            "difference against notebook 05's stored repeat0_fold0 scores (threshold 0.03), "
            "aborting cleanly before any design run started. The training input CSV was "
            "byte-identical to notebook 05's own and the chemprop argv identical to notebook 31's, "
            "and val_loss was bit-identical for epochs 0-1 before diverging at epoch 2, so the "
            "fault was arithmetic, not data. Re-running with those two variables unset and nothing "
            "else changed reproduced notebook 05 exactly (0.00000000, bit-identical val_loss "
            "trajectory); notebooks 29 and 31, the two that reproduce notebook 05 to 0.0000, do "
            "not set them. Restricting BLAS/OMP to one thread changes floating-point reduction "
            "order and so the training trajectory."),
    alternatives=("Raise the sanity threshold to admit the 0.0378 difference (would have silently "
                  "accepted a pipeline that does not reproduce this project's own stored numbers, "
                  "the exact failure the check exists to catch); keep the variable and abandon the "
                  "sanity comparison."),
    authority="the brief's own HARD FAILURE rule for a sanity mismatch, plus the requirement to diagnose rather than improvise around a failure")

record_decision(
    decision="Treated TREES as six separate arms (ecfp4_narrow and mordred_pca x rf/xgboost/lightgbm) rather than one averaged arm",
    reason=("The brief names three algorithms on 'descriptor and fingerprint features' without saying "
            "whether to report them individually or pooled. Reporting them individually is the option "
            "that changes least from notebook 33, which reported each tabular config separately, and it "
            "is strictly more informative for the decorrelation question -- a pooled average could hide "
            "one genuinely decorrelated member behind five correlated ones."),
    alternatives="Average the six into a single TREES arm (fewer rows in every table, but loses per-config decorrelation).",
    authority="autonomy rule: take the most conservative option and record it")

record_decision(
    decision=f"Added a {WALL_DEADLINE_H}h wall-clock guard that skips further runs (recording each skip) once exceeded",
    reason=("The brief caps the run at 24 hours with no human available. The timing gate reduces the "
            "DESIGN up front but cannot react to notebook 28's documented 10.6x mid-run per-epoch "
            "degradation, so without a deadline an unlucky slowdown could leave Parts 4/5 never "
            "reached and the run with no analysis at all. Because the order is by repeat, stopping at "
            "the guard still leaves a complete design at whatever repeat count finished."),
    alternatives="No guard (risks producing training output with no analysis); a shorter guard (wastes available budget).",
    authority="autonomy rule: take the most conservative option and record it")

record_decision(
    decision="Mapped notebook 05's manifest (repeat, fold) seeds onto the Butina (repeat, fold) cells by index",
    reason=("The two partitions' fold indices carry no shared meaning, so this is a convention, not a "
            "correspondence, and it is stated as such in Part 2. It reuses this project's own 25-seed "
            "table rather than inventing a new one, and because the seed is held constant across arms "
            "within a fold it cancels out of every arm-vs-arm comparison."),
    alternatives="Generate fresh seeds (a new, unjustified seed table); reuse one seed everywhere (loses the per-fold seed variation this project logs).",
    authority="autonomy rule: take the most conservative option and record it")

record_decision(
    decision="Applied Benjamini-Hochberg correction across ARMS within each isoform (not across isoforms within an arm)",
    reason=("The brief says to apply BH without naming the family. Within-isoform across-arms is the "
            "family a reader actually forms when asking which arm beats PLAIN on a given isoform, and it "
            "is the more conservative choice here because there are more arms (9) than isoforms (4)."),
    alternatives="BH across isoforms within each arm, as notebook 29 used for its single two-arm comparison (a smaller family, so weaker correction).",
    authority="autonomy rule: take the most conservative option and record it")

with open(OUT / "autonomous_decisions.json", "w") as fh:
    json.dump(AUTONOMOUS_DECISIONS, fh, indent=2)

print(f"{len(AUTONOMOUS_DECISIONS)} decision(s) recorded:\n")
for i, d in enumerate(AUTONOMOUS_DECISIONS, 1):
    print(f"{i}. {d['decision']}")
    print(f"   WHY: {d['reason']}")
    print(f"   ALTERNATIVES: {d['alternatives']}")
    print(f"   AUTHORITY: {d['authority']}\n")

7 decision(s) recorded:

1. Carried on with five seeded cluster-to-fold assignments although repeat 0 does not reproduce notebook 33's partition label-for-label
   WHY: Repeat 0 does not reproduce notebook 33's own partition (best-case agreement 22.4%). Notebook 33 broke both ties deterministically by lowest index (stable argsort, then np.argmin over fold counts); the brief specifies the tie be broken by that repeat's seed, which makes nb33's draw one particular member of the family rather than seed 0's member. Carrying on with the five seeded assignments, per the brief's own instruction -- notebook 33's results therefore sit on a partition that is NOT in this notebook's set of five.
   ALTERNATIVES: Force repeat 0 to be notebook 33's own deterministic assignment and seed only repeats 1-4 (would make the five repeats non-exchangeable, so the across-repeat spread would no longer estimate one thing); or abandon the repeated design.
   AUTHORITY: the brief's own instruction for this exact

## Scope check — fails loudly

Two independent tests, following notebook 30's fixed version rather than notebook 29's, whose
substring path matching without reading `git status` codes raised a false violation on its own first
run: **TEST 1** looks only at *tracked* content changes under a protected path (a `??` untracked
entry is a pre-existing uncommitted directory, not a modification by this notebook, and is covered
by TEST 2 instead); **TEST 2** checks that nothing inside a protected tree was *written* during this
campaign, using the first-execution timestamp persisted in `campaign.json` so the check spans every
resume rather than only the last `nbconvert` invocation.

In [171]:
st_out = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                        cwd=REPO_ROOT).stdout
print("git status --porcelain:")
print(st_out if st_out.strip() else "(clean)")

PROTECTED = [
    "data/folds/cv_folds.csv", "data/folds/cv_folds_butina.csv",
    "data/processed/", "data/raw/", "src/",
    "outputs/05_cv_comparison/", "outputs/11_caruana_prep/", "outputs/11b_caruana_selection/",
    "outputs/24_nonnegative_stacking/", "outputs/27_calibration/", "outputs/27b_aid_cyp2d6_corrected/",
    "outputs/28_external_data/", "outputs/29_cv_confirmation/", "outputs/30_aid_full_retrain/",
    "outputs/31_deadzone/", "outputs/32_deadzone_aid_retrain/", "outputs/33_butina_split/",
    "outputs/board_solved_calibration/", "outputs/19_cyp2c9_revert/", "outputs/10c_control_submission/",
    "outputs/12_caruana_retrain_predict/",
]

tracked_violations = []
for line in st_out.splitlines():
    code_, path = line[:2], line[3:].strip()
    if any(x in path for x in PROTECTED) and code_.strip() != "??":
        tracked_violations.append(line)

written_during_run = []
for prot in PROTECTED:
    q = REPO_ROOT / prot
    if not q.exists():
        continue
    candidates = [q] if q.is_file() else [x for x in q.rglob("*") if x.is_file()]
    for fp in candidates:
        try:
            if fp.stat().st_mtime > RUN_STARTED_AT:
                written_during_run.append(str(fp.relative_to(REPO_ROOT)))
        except OSError:
            pass

print(f"\nTEST 1 -- tracked modifications under a protected path: {len(tracked_violations)}")
for t in tracked_violations:
    print(f"    VIOLATION: {t}")
print(f"TEST 2 -- files written inside a protected tree during this campaign: {len(written_during_run)}")
for t in written_during_run[:25]:
    print(f"    VIOLATION: {t}")

if tracked_violations or written_during_run:
    raise ValueError("a protected path was modified -- investigate before trusting anything above")
print("\nconfirmed: no protected path modified, by either test.")
print(f"\nnew partition file written by this notebook: {BUTINA5X5_PATH.relative_to(REPO_ROOT)}")
print(f"outputs written by this notebook: {sorted(q.name for q in OUT.iterdir())}")
print(f"\nfigures: {sorted(q.name for q in FIG_OUT.iterdir())}")

git status --porcelain:
 M CLAUDE.md
 M notebooks/27b_aid_cyp2d6_corrected.ipynb
?? data/folds/cv_folds_butina.csv
?? data/folds/cv_folds_butina_5x5.csv
?? notebooks/33_butina_cluster_split.ipynb
?? notebooks/34_butina_5x5_repeated_cv.ipynb
?? outputs/33_butina_split/
?? outputs/34_butina_5x5/


TEST 1 -- tracked modifications under a protected path: 0
TEST 2 -- files written inside a protected tree during this campaign: 0

confirmed: no protected path modified, by either test.

new partition file written by this notebook: data/folds/cv_folds_butina_5x5.csv
outputs written by this notebook: ['aid_flip_reconciliation.csv', 'arm_summary.csv', 'autonomous_decisions.json', 'blend_results.csv', 'campaign.json', 'chemprop_runs', 'curated_deadzone_target_r0.csv', 'curated_deadzone_target_r1.csv', 'cv_folds_butina_5x5_sha256.txt', 'deadzone_target_checks.csv', 'decorrelation.csv', 'decorrelation_verdict.csv', 'design.json', 'error_correlation_CYP1A2.csv', 'error_correlation_CYP2C9.csv', 'error